# CSE498R — Long-Tail Logit Calibration for HSI

## Notebook 04 — Long-Tail Training Baselines

### Purpose

This notebook trains and validates the long-tail comparison baselines using the
frozen experimental protocol established in Notebooks 01–03.

Methods:

1. Focal Loss
2. LDAM-DRW
3. Balanced Softmax
4. Logit-Adjusted Loss

All methods use the same:

- frozen train/validation/test center indices;
- frozen PCA-normalized cubes;
- HybridSN backbone;
- dataset-specific patch sizes;
- seeds: 42, 123, 3407;
- 100 training epochs;
- batch size 256.

Checkpoint selection in this notebook uses validation data only.

The official test set remains sealed until all long-tail checkpoints are frozen.

## Notebook 04 Protocol Lock

The following rules are mandatory:

- Notebook 01 splits must not be regenerated.
- Notebook 02 PCA preprocessing must not be recomputed.
- Notebook 03 CE checkpoints/logits must not be modified.
- Class priors/counts must come only from the actual 70% training split.
- Validation data may be used for hyperparameter and checkpoint selection.
- Official-test labels/results must not be used during training or tuning.
- Training seeds are fixed at:
  - 42
  - 123
  - 3407
- Long-tail checkpoint selection uses validation Average Accuracy (AA).
- All trained baseline artifacts must be saved with hashes before test access.

In [ ]:
# ============================================================
# CELL 03 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive"
)

print(
    "Google Drive mounted successfully."
)

Mounted at /content/drive
Google Drive mounted successfully.


In [ ]:
# ============================================================
# CELL 04 — IMPORT LIBRARIES
# ============================================================

import os
import sys
import gc
import json
import time
import math
import random
import hashlib
import platform
import shutil

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    cohen_kappa_score,
    f1_score,
)

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
)


print("=" * 90)
print("NOTEBOOK 04 ENVIRONMENT")
print("=" * 90)

print(
    "Python       :",
    sys.version.split()[0]
)

print(
    "NumPy        :",
    np.__version__
)

print(
    "Pandas       :",
    pd.__version__
)

print(
    "Scikit-learn :",
    sklearn.__version__
)

print(
    "PyTorch      :",
    torch.__version__
)

print("=" * 90)

NOTEBOOK 04 ENVIRONMENT
Python       : 3.12.13
NumPy        : 2.0.2
Pandas       : 2.2.2
Scikit-learn : 1.6.1
PyTorch      : 2.11.0+cu128


In [ ]:
# ============================================================
# CELL 05 — EXACT PROJECT ROOT + DIRECTORY LOCK
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/CSE498/LTLC"
)

AUDIT_DIR = (
    PROJECT_ROOT /
    "audit"
)

SPLIT_DIR = (
    PROJECT_ROOT /
    "split_indices"
)

PCA_CACHE_DIR = (
    PROJECT_ROOT /
    "data_cache" /
    "pca_normalized_cubes"
)

RUNS_DIR = (
    PROJECT_ROOT /
    "runs"
)

NOTEBOOK02_AUDIT_DIR = (
    AUDIT_DIR /
    "notebook02_preprocessing"
)

NOTEBOOK03_AUDIT_DIR = (
    AUDIT_DIR /
    "notebook03_hybridsn"
)

NOTEBOOK04_AUDIT_DIR = (
    AUDIT_DIR /
    "notebook04_longtail"
)

NOTEBOOK04_RUNS_DIR = (
    RUNS_DIR /
    "longtail_training_baselines"
)


# ------------------------------------------------------------
# Create only Notebook 04 output folders.
# Never recreate/overwrite frozen earlier artifacts.
# ------------------------------------------------------------

NOTEBOOK04_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

NOTEBOOK04_RUNS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


required_existing_dirs = {
    "PROJECT_ROOT":
        PROJECT_ROOT,

    "AUDIT_DIR":
        AUDIT_DIR,

    "SPLIT_DIR":
        SPLIT_DIR,

    "PCA_CACHE_DIR":
        PCA_CACHE_DIR,

    "NOTEBOOK02_AUDIT_DIR":
        NOTEBOOK02_AUDIT_DIR,

    "NOTEBOOK03_AUDIT_DIR":
        NOTEBOOK03_AUDIT_DIR,
}


print("=" * 105)
print("LTLC PROJECT PATH VERIFICATION")
print("=" * 105)


ALL_PROJECT_PATHS_PASS = True


for name, path in (
    required_existing_dirs.items()
):

    exists = path.exists()

    ALL_PROJECT_PATHS_PASS &= exists

    print(
        f"{name:24s}: "
        f"{'PASS' if exists else 'FAIL'} | "
        f"{path}"
    )


print("-" * 105)

print(
    "PROJECT ROOT:",
    PROJECT_ROOT
)

print()

print(
    "PROJECT PATH GATE:",
    "PASS"
    if ALL_PROJECT_PATHS_PASS
    else "FAIL"
)


if not ALL_PROJECT_PATHS_PASS:

    raise RuntimeError(
        "Required LTLC project directories are missing."
    )


print("=" * 105)

LTLC PROJECT PATH VERIFICATION
PROJECT_ROOT            : PASS | /content/drive/MyDrive/CSE498/LTLC
AUDIT_DIR               : PASS | /content/drive/MyDrive/CSE498/LTLC/audit
SPLIT_DIR               : PASS | /content/drive/MyDrive/CSE498/LTLC/split_indices
PCA_CACHE_DIR           : PASS | /content/drive/MyDrive/CSE498/LTLC/data_cache/pca_normalized_cubes
NOTEBOOK02_AUDIT_DIR    : PASS | /content/drive/MyDrive/CSE498/LTLC/audit/notebook02_preprocessing
NOTEBOOK03_AUDIT_DIR    : PASS | /content/drive/MyDrive/CSE498/LTLC/audit/notebook03_hybridsn
---------------------------------------------------------------------------------------------------------
PROJECT ROOT: /content/drive/MyDrive/CSE498/LTLC

PROJECT PATH GATE: PASS


In [ ]:
# ============================================================
# CELL 06 — LOAD FROZEN NOTEBOOK 01–03 ARTIFACTS
# ============================================================

NOTEBOOK01_METADATA_PATH = (
    AUDIT_DIR /
    "notebook01_protocol_metadata.json"
)

NOTEBOOK02_MASTER_PATH = (
    NOTEBOOK02_AUDIT_DIR /
    "notebook02_master_preprocessing_audit.json"
)

PATCH_MANIFEST_PATH = (
    NOTEBOOK02_AUDIT_DIR /
    "canonical_patch_manifest.json"
)

NOTEBOOK03_MASTER_PATH = (
    NOTEBOOK03_AUDIT_DIR /
    "notebook03_master_audit.json"
)

CE_CHECKPOINT_MANIFEST_PATH = (
    NOTEBOOK03_AUDIT_DIR /
    "ce_checkpoint_manifest.json"
)

INFERENCE_MANIFEST_PATH = (
    NOTEBOOK03_AUDIT_DIR /
    "frozen_inference" /
    "hybridsn_ce_inference_output_manifest.json"
)


FROZEN_REQUIRED_FILES = {
    "Notebook 01 metadata":
        NOTEBOOK01_METADATA_PATH,

    "Notebook 02 master audit":
        NOTEBOOK02_MASTER_PATH,

    "Notebook 02 patch manifest":
        PATCH_MANIFEST_PATH,

    "Notebook 03 master audit":
        NOTEBOOK03_MASTER_PATH,

    "Notebook 03 checkpoint manifest":
        CE_CHECKPOINT_MANIFEST_PATH,

    "Notebook 03 inference manifest":
        INFERENCE_MANIFEST_PATH,
}


print("=" * 110)
print("FROZEN NOTEBOOK 01–03 ARTIFACT CHECK")
print("=" * 110)


for name, path in (
    FROZEN_REQUIRED_FILES.items()
):

    exists = path.exists()

    print(
        f"{name:34s}: "
        f"{'PASS' if exists else 'FAIL'}"
    )

    if not exists:

        raise FileNotFoundError(
            path
        )


with open(
    NOTEBOOK01_METADATA_PATH,
    "r",
    encoding="utf-8"
) as f:

    NB01 = json.load(f)


with open(
    NOTEBOOK02_MASTER_PATH,
    "r",
    encoding="utf-8"
) as f:

    NB02 = json.load(f)


with open(
    PATCH_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as f:

    PATCH_MANIFEST = json.load(f)


with open(
    NOTEBOOK03_MASTER_PATH,
    "r",
    encoding="utf-8"
) as f:

    NB03 = json.load(f)


with open(
    CE_CHECKPOINT_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as f:

    CE_CHECKPOINT_MANIFEST = json.load(f)


with open(
    INFERENCE_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as f:

    CE_INFERENCE_MANIFEST = json.load(f)


NB03_GATE_PASS = all(
    bool(value)
    for value
    in NB03[
        "gates"
    ].values()
)


print()
print(
    "Notebook 03 frozen gates:",
    "PASS"
    if NB03_GATE_PASS
    else "FAIL"
)

print(
    "Frozen CE checkpoints   :",
    CE_CHECKPOINT_MANIFEST[
        "total_checkpoints"
    ]
)

print(
    "Frozen inference outputs:",
    CE_INFERENCE_MANIFEST[
        "total_output_files"
    ]
)

print("=" * 110)


if not NB03_GATE_PASS:

    raise RuntimeError(
        "Notebook 03 master audit is not fully PASS."
    )

FROZEN NOTEBOOK 01–03 ARTIFACT CHECK
Notebook 01 metadata              : PASS
Notebook 02 master audit          : PASS
Notebook 02 patch manifest        : PASS
Notebook 03 master audit          : PASS
Notebook 03 checkpoint manifest   : PASS
Notebook 03 inference manifest    : PASS

Notebook 03 frozen gates: PASS
Frozen CE checkpoints   : 9
Frozen inference outputs: 18


In [ ]:
# ============================================================
# CELL 07 — T4 GPU GATE
# ============================================================

CUDA_AVAILABLE = (
    torch.cuda.is_available()
)


print("=" * 90)
print("NOTEBOOK 04 GPU CHECK")
print("=" * 90)

print(
    "CUDA available :",
    "PASS"
    if CUDA_AVAILABLE
    else "FAIL"
)


if not CUDA_AVAILABLE:

    raise RuntimeError(
        "GPU unavailable. Change runtime to T4 GPU."
    )


DEVICE = torch.device(
    "cuda"
)

GPU_NAME = torch.cuda.get_device_name(
    0
)

GPU_MEMORY_GB = (
    torch.cuda.get_device_properties(
        0
    ).total_memory
    /
    1024**3
)


print(
    "GPU            :",
    GPU_NAME
)

print(
    f"GPU memory     : "
    f"{GPU_MEMORY_GB:.2f} GB"
)

print(
    "CUDA version   :",
    torch.version.cuda
)

print(
    "cuDNN version  :",
    torch.backends.cudnn.version()
)

print(
    "Device         :",
    DEVICE
)

print()

print(
    "GPU GATE: PASS"
)

print("=" * 90)

NOTEBOOK 04 GPU CHECK
CUDA available : PASS
GPU            : Tesla T4
GPU memory     : 14.56 GB
CUDA version   : 12.8
cuDNN version  : 91900
Device         : cuda

GPU GATE: PASS


In [ ]:
# ============================================================
# CELL 08 — FROZEN DATASET REGISTRY
# ============================================================

DATASETS = {}


for dataset_name in [
    "Pingan",
    "Qingyun",
    "Tangdaowan",
]:

    nb1 = NB01[
        "datasets"
    ][dataset_name]

    nb2 = NB02[
        "datasets"
    ][dataset_name]

    patch = PATCH_MANIFEST[
        "datasets"
    ][dataset_name]


    DATASETS[
        dataset_name
    ] = {

        "cube_shape":
            tuple(
                nb1[
                    "cube_shape"
                ]
            ),

        "num_classes":
            int(
                nb1[
                    "evaluation_class_count"
                ]
            ),

        "evaluation_class_ids":
            [
                int(x)
                for x in
                nb1[
                    "evaluation_class_ids"
                ]
            ],

        "pca_components":
            int(
                nb2[
                    "pca_components"
                ]
            ),

        "patch_size":
            int(
                patch[
                    "patch_size"
                ]
            ),

        "patch_shape":
            tuple(
                patch[
                    "patch_shape"
                ]
            ),

        "normalized_cube_file":
            nb2[
                "normalized_cube_file"
            ],

        "normalized_cube_sha256":
            nb2[
                "normalized_cube_sha256"
            ],

        "split_sha256":
            nb2[
                "split_sha256"
            ],
    }


print("=" * 100)
print("NOTEBOOK 04 — FROZEN DATASET REGISTRY")
print("=" * 100)


for dataset_name, cfg in (
    DATASETS.items()
):

    print(
        f"{dataset_name:12s} | "
        f"Classes={cfg['num_classes']:2d} | "
        f"PCA={cfg['pca_components']:2d} | "
        f"Patch={cfg['patch_shape']}"
    )


print("=" * 100)

NOTEBOOK 04 — FROZEN DATASET REGISTRY
Pingan       | Classes=10 | PCA=15 | Patch=(13, 13, 15)
Qingyun      | Classes= 6 | PCA=15 | Patch=(11, 11, 15)
Tangdaowan   | Classes=16 | PCA=20 | Patch=(9, 9, 20)


In [ ]:
# ============================================================
# CELL 09 — LOAD + HASH-VERIFY FROZEN SPLITS
# ============================================================

def sha256_array(
    array
):

    array = np.ascontiguousarray(
        array
    )

    hasher = hashlib.sha256()

    hasher.update(
        str(
            array.shape
        ).encode(
            "utf-8"
        )
    )

    hasher.update(
        str(
            array.dtype
        ).encode(
            "utf-8"
        )
    )

    hasher.update(
        array.tobytes()
    )

    return hasher.hexdigest()


def combined_split_hash(
    split
):

    train_hash = sha256_array(
        split[
            "train_indices"
        ]
    )

    val_hash = sha256_array(
        split[
            "val_indices"
        ]
    )

    test_hash = sha256_array(
        split[
            "test_indices"
        ]
    )


    hasher = hashlib.sha256()

    hasher.update(
        train_hash.encode(
            "utf-8"
        )
    )

    hasher.update(
        val_hash.encode(
            "utf-8"
        )
    )

    hasher.update(
        test_hash.encode(
            "utf-8"
        )
    )

    return hasher.hexdigest()


SPLIT_FILES = {

    "Pingan":
        SPLIT_DIR /
        "pingan_fixed_split_seed2026.npz",

    "Qingyun":
        SPLIT_DIR /
        "qingyun_fixed_split_seed2026.npz",

    "Tangdaowan":
        SPLIT_DIR /
        "tangdaowan_fixed_split_seed2026.npz",
}


FROZEN_SPLITS = {}

SPLIT_AUDIT_RESULTS = {}


print("=" * 105)
print("FROZEN SPLIT RELOAD + SHA-256 AUDIT")
print("=" * 105)


for dataset_name in DATASETS:

    path = SPLIT_FILES[
        dataset_name
    ]


    if not path.exists():

        raise FileNotFoundError(
            path
        )


    with np.load(
        path,
        allow_pickle=False
    ) as saved:

        split = {
            key:
                saved[key].copy()

            for key in
                saved.files
        }


    observed_hash = (
        combined_split_hash(
            split
        )
    )


    hash_pass = (
        observed_hash
        ==
        DATASETS[
            dataset_name
        ][
            "split_sha256"
        ]
    )


    K = DATASETS[
        dataset_name
    ][
        "num_classes"
    ]


    model_label_range_pass = all([

        split[
            "train_labels_model"
        ].min() >= 0,

        split[
            "train_labels_model"
        ].max() < K,

        split[
            "val_labels_model"
        ].min() >= 0,

        split[
            "val_labels_model"
        ].max() < K,

        split[
            "test_labels_model"
        ].min() >= 0,

        split[
            "test_labels_model"
        ].max() < K,
    ])


    overall_pass = all([
        hash_pass,
        model_label_range_pass,
    ])


    FROZEN_SPLITS[
        dataset_name
    ] = split


    SPLIT_AUDIT_RESULTS[
        dataset_name
    ] = overall_pass


    print(
        f"\n{dataset_name.upper()}"
    )

    print("-" * 70)

    print(
        "Split SHA-256 :",
        "PASS"
        if hash_pass
        else "FAIL"
    )

    print(
        "Model labels  :",
        "PASS"
        if model_label_range_pass
        else "FAIL"
    )

    print(
        "Train centers :",
        len(
            split[
                "train_indices"
            ]
        )
    )

    print(
        "Val centers   :",
        len(
            split[
                "val_indices"
            ]
        )
    )

    print(
        "Test centers  :",
        len(
            split[
                "test_indices"
            ]
        )
    )

    print(
        "STATUS        :",
        "PASS"
        if overall_pass
        else "FAIL"
    )


ALL_SPLITS_PASS = all(
    SPLIT_AUDIT_RESULTS.values()
)


print(
    "\n" + "=" * 105
)

print(
    "FROZEN SPLIT GATE:",
    "PASS"
    if ALL_SPLITS_PASS
    else "FAIL"
)

print("=" * 105)

FROZEN SPLIT RELOAD + SHA-256 AUDIT

PINGAN
----------------------------------------------------------------------
Split SHA-256 : PASS
Model labels  : PASS
Train centers : 79869
Val centers   : 34230
Test centers  : 1026838
STATUS        : PASS

QINGYUN
----------------------------------------------------------------------
Split SHA-256 : PASS
Model labels  : PASS
Train centers : 66844
Val centers   : 28648
Test centers  : 859401
STATUS        : PASS

TANGDAOWAN
----------------------------------------------------------------------
Split SHA-256 : PASS
Model labels  : PASS
Train centers : 24760
Val centers   : 10612
Test centers  : 469917
STATUS        : PASS

FROZEN SPLIT GATE: PASS


In [ ]:
# ============================================================
# CELL 10 — FROZEN CLASS-STATISTICS INTEGRITY AUDIT
# ============================================================

CLASS_STATS_FILES = {

    "Pingan":
        AUDIT_DIR /
        "pingan_ltlc_class_statistics.csv",

    "Qingyun":
        AUDIT_DIR /
        "qingyun_ltlc_class_statistics.csv",

    "Tangdaowan":
        AUDIT_DIR /
        "tangdaowan_ltlc_class_statistics.csv",
}


FROZEN_CLASS_STATS = {}

CLASS_STATS_AUDIT = {}


print("=" * 110)
print("NOTEBOOK 04 — FROZEN CLASS-STATISTICS AUDIT")
print("=" * 110)


for dataset_name in DATASETS:

    path = CLASS_STATS_FILES[
        dataset_name
    ]


    if not path.exists():

        raise FileNotFoundError(
            path
        )


    df = pd.read_csv(
        path
    )


    K = DATASETS[
        dataset_name
    ][
        "num_classes"
    ]


    # --------------------------------------------------------
    # Expected model-label ordered training counts
    # --------------------------------------------------------

    train_labels = (
        FROZEN_SPLITS[
            dataset_name
        ][
            "train_labels_model"
        ]
    )


    observed_counts = np.bincount(

        train_labels,

        minlength=K

    ).astype(
        np.int64
    )


    saved_counts = (
        df[
            "train_count"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    count_match_pass = (
        np.array_equal(
            observed_counts,
            saved_counts
        )
    )


    # --------------------------------------------------------
    # Class IDs
    # --------------------------------------------------------

    expected_class_ids = np.arange(
        1,
        K + 1,
        dtype=np.int64
    )


    class_id_pass = (
        np.array_equal(

            df[
                "class_id"
            ].to_numpy(
                dtype=np.int64
            ),

            expected_class_ids
        )
    )


    # --------------------------------------------------------
    # Priors must be training-only priors
    # --------------------------------------------------------

    reconstructed_priors = (

        saved_counts.astype(
            np.float64
        )

        /

        saved_counts.sum()
    )


    saved_priors = (
        df[
            "prior"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    prior_pass = np.allclose(

        reconstructed_priors,

        saved_priors,

        atol=1e-6
    )


    prior_sum_pass = np.isclose(
        saved_priors.sum(),
        1.0,
        atol=1e-6
    )


    # --------------------------------------------------------
    # Rarity
    # --------------------------------------------------------

    rarity_pass = bool(

        df[
            "rarity"
        ].between(
            0.0,
            1.0
        ).all()
    )


    imbalance_ratio = (

        saved_counts.max()

        /

        saved_counts.min()
    )


    overall_pass = all([

        count_match_pass,
        class_id_pass,
        prior_pass,
        prior_sum_pass,
        rarity_pass,
    ])


    CLASS_STATS_AUDIT[
        dataset_name
    ] = overall_pass


    FROZEN_CLASS_STATS[
        dataset_name
    ] = df.copy()


    print(
        f"\n{dataset_name.upper()}"
    )

    print("-" * 75)

    print(
        "Class-stat file     : PASS"
    )

    print(
        "Train counts exact  :",
        "PASS"
        if count_match_pass
        else "FAIL"
    )

    print(
        "Class IDs exact     :",
        "PASS"
        if class_id_pass
        else "FAIL"
    )

    print(
        "Training priors     :",
        "PASS"
        if prior_pass
        else "FAIL"
    )

    print(
        f"Prior sum           : "
        f"{saved_priors.sum():.8f}"
    )

    print(
        "Rarity range        :",
        "PASS"
        if rarity_pass
        else "FAIL"
    )

    print(
        f"Imbalance ratio     : "
        f"{imbalance_ratio:.6f}:1"
    )

    print(
        "STATUS              :",
        "PASS"
        if overall_pass
        else "FAIL"
    )


ALL_CLASS_STATS_PASS = all(
    CLASS_STATS_AUDIT.values()
)


print(
    "\n" + "=" * 110
)

print(
    "FROZEN CLASS-STATISTICS GATE:",
    "PASS"
    if ALL_CLASS_STATS_PASS
    else "FAIL"
)

print()

print(
    "No validation/test statistics were used "
    "to construct class priors."
)

print(
    "Official-test inference outputs remain untouched."
)

print("=" * 110)

NOTEBOOK 04 — FROZEN CLASS-STATISTICS AUDIT

PINGAN
---------------------------------------------------------------------------
Class-stat file     : PASS
Train counts exact  : PASS
Class IDs exact     : PASS
Training priors     : PASS
Prior sum           : 1.00000000
Rarity range        : PASS
Imbalance ratio     : 70.996491:1
STATUS              : PASS

QINGYUN
---------------------------------------------------------------------------
Class-stat file     : PASS
Train counts exact  : PASS
Class IDs exact     : PASS
Training priors     : PASS
Prior sum           : 1.00000000
Rarity range        : PASS
Imbalance ratio     : 28.467836:1
STATUS              : PASS

TANGDAOWAN
---------------------------------------------------------------------------
Class-stat file     : PASS
Train counts exact  : PASS
Class IDs exact     : PASS
Training priors     : PASS
Prior sum           : 1.00000000
Rarity range        : PASS
Imbalance ratio     : 191.777778:1
STATUS              : PASS

FROZEN CLA

## Long-Tail Training Baseline Protocol Lock

Notebook 04 trains four long-tail baselines:

1. Focal Loss
2. LDAM-DRW
3. Balanced Softmax
4. Logit-Adjusted Loss (LA-loss)

### Common training controls

All baselines use the same:

- frozen Notebook 01 train/validation splits;
- frozen Notebook 02 PCA-normalized cubes;
- HybridSN architecture used in Notebook 03;
- dataset-specific patch sizes;
- batch size = 256;
- epochs = 100;
- optimizer = Adam;
- learning rate = 1e-3;
- Adam betas = (0.9, 0.999);
- Adam epsilon = 1e-7;
- legacy Keras-style decay = 1e-6;
- no augmentation;
- no test-set tuning;
- seeds = 42, 123, 3407.

### Long-tail checkpoint selection

For every training run, the checkpoint is selected by:

1. highest validation AA;
2. then highest validation Macro-F1;
3. then highest validation OA;
4. then lowest validation loss;
5. otherwise the earlier epoch is retained.

### Validation-only baseline tuning

For baselines with two candidate settings:

1. Train both candidates using seed 42.
2. Compare their best validation-AA checkpoints.
3. Select using validation AA.
4. Tie → validation Macro-F1.
5. Tie → validation OA.
6. Tie → choose the smaller/simpler hyperparameter.
7. Freeze the selected setting.
8. Train seeds 123 and 3407 using only that selected setting.
9. Never revise the setting after official-test evaluation.

The official test set is not used for hyperparameter or checkpoint selection.

In [ ]:
# ============================================================
# CELL 12 — BASELINE HYPERPARAMETER BUDGET LOCK
# ============================================================

TRAINING_SEEDS = [
    42,
    123,
    3407,
]

TUNING_SEED = 42

NUM_EPOCHS = 100
BATCH_SIZE = 256


COMMON_TRAINING_CONFIG = {

    "epochs":
        NUM_EPOCHS,

    "batch_size":
        BATCH_SIZE,

    "optimizer":
        "Adam",

    "learning_rate":
        1e-3,

    "adam_betas":
        [0.9, 0.999],

    "adam_epsilon":
        1e-7,

    "legacy_decay":
        1e-6,

    "augmentation":
        False,

    "checkpoint_primary":
        "validation_AA",
}


BASELINE_SEARCH_SPACE = {

    "Focal": {

        "tunable":
            True,

        "parameter":
            "gamma",

        "candidates":
            [1.0, 2.0],

        "class_frequency_alpha":
            False,
    },


    "LDAM_DRW": {

        "tunable":
            True,

        "parameter":
            "max_margin_C",

        "candidates":
            [0.25, 0.50],

        "scale":
            30.0,

        "drw_start_epoch":
            80,

        "effective_number_beta":
            0.9999,
    },


    "BalancedSoftmax": {

        "tunable":
            False,

        "parameter":
            None,

        "candidates":
            ["training_counts"],
    },


    "LA_Loss": {

        "tunable":
            True,

        "parameter":
            "tau",

        "candidates":
            [0.5, 1.0],
    },
}


# ------------------------------------------------------------
# Training-budget accounting
# ------------------------------------------------------------

#
# Tuned method per dataset:
#   seed42 candidate A
#   seed42 candidate B
#   seed123 selected setting
#   seed3407 selected setting
# = 4 runs
#
# 3 tuned methods × 3 datasets × 4 = 36
#
# Balanced Softmax:
# 3 datasets × 3 seeds = 9
#
# Notebook 04 total = 45 new runs
#

EXPECTED_NOTEBOOK04_RUNS = (
    3 * 3 * 4
    +
    3 * 3
)

EXPECTED_CORE_PROJECT_TRAININGS = (
    9 + EXPECTED_NOTEBOOK04_RUNS
)


print("=" * 110)
print("NOTEBOOK 04 — LOCKED BASELINE SEARCH SPACE")
print("=" * 110)

print(
    "Training seeds       :",
    TRAINING_SEEDS
)

print(
    "Tuning seed          :",
    TUNING_SEED
)

print(
    "Epochs               :",
    NUM_EPOCHS
)

print(
    "Batch size           :",
    BATCH_SIZE
)

print()


for method_name, cfg in (
    BASELINE_SEARCH_SPACE.items()
):

    print(
        f"{method_name:17s} | "
        f"Candidates = "
        f"{cfg['candidates']}"
    )


print("-" * 110)

print(
    "New Notebook 04 runs :",
    EXPECTED_NOTEBOOK04_RUNS
)

print(
    "CE + LT baseline runs:",
    EXPECTED_CORE_PROJECT_TRAININGS
)

print("=" * 110)


assert EXPECTED_NOTEBOOK04_RUNS == 45

assert EXPECTED_CORE_PROJECT_TRAININGS == 54

NOTEBOOK 04 — LOCKED BASELINE SEARCH SPACE
Training seeds       : [42, 123, 3407]
Tuning seed          : 42
Epochs               : 100
Batch size           : 256

Focal             | Candidates = [1.0, 2.0]
LDAM_DRW          | Candidates = [0.25, 0.5]
BalancedSoftmax   | Candidates = ['training_counts']
LA_Loss           | Candidates = [0.5, 1.0]
--------------------------------------------------------------------------------------------------------------
New Notebook 04 runs : 45
CE + LT baseline runs: 54


In [ ]:
# ============================================================
# CELL 13 — MODEL-LABEL ORDERED CLASS ARRAYS
# ============================================================

BASELINE_CLASS_ARRAYS = {}


print("=" * 110)
print("MODEL-LABEL ORDERED TRAINING CLASS ARRAYS")
print("=" * 110)


for dataset_name in DATASETS:

    df = (
        FROZEN_CLASS_STATS[
            dataset_name
        ]
        .sort_values(
            "class_id"
        )
        .reset_index(
            drop=True
        )
    )


    counts = (
        df[
            "train_count"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    priors = (
        df[
            "prior"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    rarity = (
        df[
            "rarity"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    groups = (
        df[
            "frequency_group"
        ]
        .astype(str)
        .to_numpy()
    )


    reconstructed_priors = (

        counts.astype(
            np.float64
        )

        /

        counts.sum()
    )


    prior_reconstruction_pass = (
        np.allclose(
            priors,
            reconstructed_priors,
            atol=1e-6
        )
    )


    if not prior_reconstruction_pass:

        raise RuntimeError(
            f"{dataset_name}: "
            "training prior reconstruction failed."
        )


    BASELINE_CLASS_ARRAYS[
        dataset_name
    ] = {

        "counts":
            counts,

        "priors":
            priors,

        "log_priors":
            np.log(
                priors
            ),

        "rarity":
            rarity,

        "frequency_groups":
            groups,
    }


    print(
        f"\n{dataset_name.upper()}"
    )

    print("-" * 75)

    print(
        "Model labels       :",
        list(
            range(
                len(counts)
            )
        )
    )

    print(
        "Train counts       :",
        counts.tolist()
    )

    print(
        "Prior reconstruction:",
        "PASS"
    )

    print(
        f"Prior sum          : "
        f"{priors.sum():.8f}"
    )

    print(
        f"Imbalance ratio    : "
        f"{counts.max() / counts.min():.6f}:1"
    )


print(
    "\n" + "=" * 110
)

print(
    "MODEL-LABEL CLASS ARRAY GATE: PASS"
)

print("=" * 110)

MODEL-LABEL ORDERED TRAINING CLASS ARRAYS

PINGAN
---------------------------------------------------------------------------
Model labels       : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Train counts       : [3425, 40468, 585, 6229, 1453, 986, 979, 5818, 570, 19356]
Prior reconstruction: PASS
Prior sum          : 1.00000000
Imbalance ratio    : 70.996491:1

QINGYUN
---------------------------------------------------------------------------
Model labels       : [0, 1, 2, 3, 4, 5]
Train counts       : [19472, 12565, 966, 684, 15242, 17915]
Prior reconstruction: PASS
Prior sum          : 1.00000000
Imbalance ratio    : 28.467836:1

TANGDAOWAN
---------------------------------------------------------------------------
Model labels       : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
Train counts       : [1271, 2723, 1668, 2974, 91, 1819, 692, 3140, 1504, 87, 1040, 36, 83, 43, 685, 6904]
Prior reconstruction: PASS
Prior sum          : 1.00000000
Imbalance ratio    : 191.777778:1

MODEL-LAB

In [ ]:
# ============================================================
# CELL 14 — REPRODUCIBILITY + CHECKPOINT POLICY
# ============================================================

def set_global_seed(
    seed
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    torch.cuda.manual_seed_all(
        seed
    )

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False

    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )


CHECKPOINT_TOL = 1e-12


def is_better_longtail_checkpoint(
    candidate,
    best
):

    """
    Notebook 04 checkpoint rule:

    1. higher validation AA
    2. higher validation Macro-F1
    3. higher validation OA
    4. lower validation loss
    5. otherwise keep earlier checkpoint
    """

    if best is None:

        return True


    # 1. AA
    if (
        candidate["aa"]
        >
        best["aa"]
        +
        CHECKPOINT_TOL
    ):

        return True


    if not math.isclose(
        candidate["aa"],
        best["aa"],
        abs_tol=CHECKPOINT_TOL
    ):

        return False


    # 2. Macro-F1
    if (
        candidate["macro_f1"]
        >
        best["macro_f1"]
        +
        CHECKPOINT_TOL
    ):

        return True


    if not math.isclose(
        candidate["macro_f1"],
        best["macro_f1"],
        abs_tol=CHECKPOINT_TOL
    ):

        return False


    # 3. OA
    if (
        candidate["oa"]
        >
        best["oa"]
        +
        CHECKPOINT_TOL
    ):

        return True


    if not math.isclose(
        candidate["oa"],
        best["oa"],
        abs_tol=CHECKPOINT_TOL
    ):

        return False


    # 4. Lower validation loss
    if (
        candidate["loss"]
        <
        best["loss"]
        -
        CHECKPOINT_TOL
    ):

        return True


    # 5. Earlier checkpoint retained
    return False


set_global_seed(
    42
)


print("=" * 100)
print("NOTEBOOK 04 REPRODUCIBILITY / CHECKPOINT POLICY")
print("=" * 100)

print(
    "Model seeds           :",
    TRAINING_SEEDS
)

print(
    "Deterministic cuDNN    :",
    torch.backends.cudnn.deterministic
)

print(
    "cuDNN benchmark        :",
    torch.backends.cudnn.benchmark
)

print(
    "Checkpoint primary     : Validation AA"
)

print(
    "Tie 1                  : Validation Macro-F1"
)

print(
    "Tie 2                  : Validation OA"
)

print(
    "Tie 3                  : Lower validation loss"
)

print(
    "Final tie              : Earlier epoch"
)

print("=" * 100)

NOTEBOOK 04 REPRODUCIBILITY / CHECKPOINT POLICY
Model seeds           : [42, 123, 3407]
Deterministic cuDNN    : True
cuDNN benchmark        : False
Checkpoint primary     : Validation AA
Tie 1                  : Validation Macro-F1
Tie 2                  : Validation OA
Tie 3                  : Lower validation loss
Final tie              : Earlier epoch


In [ ]:
# ============================================================
# CELL 15 — FOCAL / BALANCED SOFTMAX / LA-LOSS
# ============================================================

class FocalLoss(nn.Module):

    def __init__(
        self,
        gamma
    ):

        super().__init__()

        self.gamma = float(
            gamma
        )


    def forward(
        self,
        logits,
        targets
    ):

        ce = F.cross_entropy(
            logits,
            targets,
            reduction="none"
        )

        pt = torch.exp(
            -ce
        )

        loss = (

            (
                1.0 - pt
            )
            **
            self.gamma

        ) * ce


        return loss.mean()


# ============================================================

class BalancedSoftmaxLoss(nn.Module):

    def __init__(
        self,
        class_counts
    ):

        super().__init__()


        counts = torch.as_tensor(
            class_counts,
            dtype=torch.float32
        )


        if torch.any(
            counts <= 0
        ):

            raise ValueError(
                "Balanced Softmax requires "
                "positive training counts."
            )


        self.register_buffer(

            "log_counts",

            torch.log(
                counts
            )
        )


    def forward(
        self,
        logits,
        targets
    ):

        adjusted_logits = (

            logits
            +
            self.log_counts
        )


        return F.cross_entropy(
            adjusted_logits,
            targets
        )


# ============================================================

class LogitAdjustedLoss(nn.Module):

    def __init__(
        self,
        class_priors,
        tau
    ):

        super().__init__()


        priors = torch.as_tensor(
            class_priors,
            dtype=torch.float32
        )


        if torch.any(
            priors <= 0
        ):

            raise ValueError(
                "LA-loss requires positive priors."
            )


        self.tau = float(
            tau
        )


        self.register_buffer(

            "log_priors",

            torch.log(
                priors
            )
        )


    def forward(
        self,
        logits,
        targets
    ):

        adjusted_logits = (

            logits
            +
            self.tau
            *
            self.log_priors
        )


        return F.cross_entropy(
            adjusted_logits,
            targets
        )


print("=" * 100)
print("TRAINING LOSS DEFINITIONS")
print("=" * 100)

print(
    "Focal Loss       : READY"
)

print(
    "Balanced Softmax : READY"
)

print(
    "LA-loss          : READY"
)

print("=" * 100)

TRAINING LOSS DEFINITIONS
Focal Loss       : READY
Balanced Softmax : READY
LA-loss          : READY


In [ ]:
# ============================================================
# CELL 16 — LDAM + DEFERRED REWEIGHTING
# ============================================================

class LDAMLoss(nn.Module):

    def __init__(
        self,
        class_counts,
        max_margin,
        scale=30.0
    ):

        super().__init__()


        counts = np.asarray(
            class_counts,
            dtype=np.float64
        )


        if np.any(
            counts <= 0
        ):

            raise ValueError(
                "LDAM requires positive class counts."
            )


        margins = (

            1.0

            /

            np.power(
                counts,
                0.25
            )
        )


        margins = (

            margins

            *

            (
                float(
                    max_margin
                )

                /

                margins.max()
            )
        )


        self.register_buffer(

            "margins",

            torch.as_tensor(
                margins,
                dtype=torch.float32
            )
        )


        self.scale = float(
            scale
        )


    def forward(
        self,
        logits,
        targets,
        class_weights=None
    ):

        adjusted_logits = (
            logits.clone()
        )


        row_idx = torch.arange(
            logits.shape[0],
            device=logits.device
        )


        adjusted_logits[
            row_idx,
            targets
        ] -= self.margins[
            targets
        ]


        return F.cross_entropy(

            self.scale
            *
            adjusted_logits,

            targets,

            weight=
                class_weights
        )


# ------------------------------------------------------------
# Effective-number DRW weights
# ------------------------------------------------------------

def effective_number_weights(
    class_counts,
    beta=0.9999,
    device=None
):

    counts = np.asarray(
        class_counts,
        dtype=np.float64
    )


    effective_num = (

        1.0

        -

        np.power(
            beta,
            counts
        )
    )


    weights = (

        (1.0 - beta)

        /

        effective_num
    )


    # Normalize so average class weight = 1.
    weights = (

        weights

        /

        weights.sum()

        *

        len(weights)
    )


    tensor = torch.as_tensor(
        weights,
        dtype=torch.float32
    )


    if device is not None:

        tensor = tensor.to(
            device
        )


    return tensor


# ------------------------------------------------------------
# Frozen DRW schedule
#
# Epochs are numbered 1 ... 100.
# Reweighting begins AT epoch 80.
# ------------------------------------------------------------

def get_ldam_drw_weights(
    epoch,
    class_counts,
    device
):

    if epoch < 80:

        return None


    return effective_number_weights(

        class_counts,

        beta=0.9999,

        device=device
    )


print("=" * 100)
print("LDAM-DRW DEFINITION")
print("=" * 100)

print(
    "Margin candidates :",
    [0.25, 0.50]
)

print(
    "Scale             :",
    30.0
)

print(
    "DRW start epoch   :",
    80
)

print(
    "Effective beta    :",
    0.9999
)

print("=" * 100)

LDAM-DRW DEFINITION
Margin candidates : [0.25, 0.5]
Scale             : 30.0
DRW start epoch   : 80
Effective beta    : 0.9999


In [ ]:
# ============================================================
# CELL 17 — MATHEMATICAL LOSS SANITY AUDIT
# ============================================================

set_global_seed(
    42
)


LOSS_SANITY_RESULTS = {}


print("=" * 115)
print("LONG-TAIL TRAINING LOSS — MATHEMATICAL SANITY AUDIT")
print("=" * 115)


for dataset_name in DATASETS:

    arrays = BASELINE_CLASS_ARRAYS[
        dataset_name
    ]

    counts = arrays[
        "counts"
    ]

    priors = arrays[
        "priors"
    ]

    K = len(
        counts
    )


    synthetic_logits = torch.randn(
        64,
        K,
        dtype=torch.float32
    )


    synthetic_targets = (

        torch.arange(
            64,
            dtype=torch.long
        )

        %

        K
    )


    # --------------------------------------------------------
    # 1. Focal gamma=0 must reproduce ordinary CE
    # --------------------------------------------------------

    ce_value = F.cross_entropy(
        synthetic_logits,
        synthetic_targets
    )


    focal_zero_value = FocalLoss(
        gamma=0.0
    )(
        synthetic_logits,
        synthetic_targets
    )


    focal_ce_pass = torch.allclose(

        ce_value,
        focal_zero_value,

        atol=1e-7,
        rtol=1e-6
    )


    # --------------------------------------------------------
    # 2. Balanced Softmax and LA-loss tau=1
    #
    # log(n_k) and log(pi_k) differ only by log(N),
    # which is the same constant for every class.
    # --------------------------------------------------------

    bs_value = BalancedSoftmaxLoss(
        counts
    )(
        synthetic_logits,
        synthetic_targets
    )


    la_one_value = LogitAdjustedLoss(
        priors,
        tau=1.0
    )(
        synthetic_logits,
        synthetic_targets
    )


    bs_la_pass = torch.allclose(

        bs_value,
        la_one_value,

        atol=1e-6,
        rtol=1e-6
    )


    # --------------------------------------------------------
    # 3. Normal Focal candidate
    # --------------------------------------------------------

    focal_two_value = FocalLoss(
        gamma=2.0
    )(
        synthetic_logits,
        synthetic_targets
    )


    # --------------------------------------------------------
    # 4. LDAM
    # --------------------------------------------------------

    ldam_module = LDAMLoss(

        counts,

        max_margin=0.50,

        scale=30.0
    )


    ldam_value = ldam_module(
        synthetic_logits,
        synthetic_targets
    )


    margins = (
        ldam_module
        .margins
        .detach()
        .cpu()
        .numpy()
    )


    rarest_idx = int(
        np.argmin(
            counts
        )
    )

    largest_idx = int(
        np.argmax(
            counts
        )
    )


    ldam_margin_direction_pass = (

        margins[
            rarest_idx
        ]

        >

        margins[
            largest_idx
        ]
    )


    ldam_max_margin_pass = np.isclose(

        margins.max(),

        0.50,

        atol=1e-6
    )


    # --------------------------------------------------------
    # 5. DRW schedule
    # --------------------------------------------------------

    weights_epoch79 = (
        get_ldam_drw_weights(

            epoch=79,

            class_counts=counts,

            device=torch.device(
                "cpu"
            )
        )
    )


    weights_epoch80 = (
        get_ldam_drw_weights(

            epoch=80,

            class_counts=counts,

            device=torch.device(
                "cpu"
            )
        )
    )


    drw_schedule_pass = all([

        weights_epoch79 is None,

        weights_epoch80
        is not None,
    ])


    drw_weights_np = (
        weights_epoch80
        .cpu()
        .numpy()
    )


    drw_positive_pass = bool(
        np.all(
            drw_weights_np > 0
        )
    )


    drw_finite_pass = bool(
        np.all(
            np.isfinite(
                drw_weights_np
            )
        )
    )


    drw_mean_pass = np.isclose(

        drw_weights_np.mean(),

        1.0,

        atol=1e-6
    )


    drw_tail_direction_pass = (

        drw_weights_np[
            rarest_idx
        ]

        >

        drw_weights_np[
            largest_idx
        ]
    )


    # --------------------------------------------------------
    # 6. All loss values finite
    # --------------------------------------------------------

    finite_losses_pass = all([

        bool(
            torch.isfinite(
                ce_value
            ).item()
        ),

        bool(
            torch.isfinite(
                focal_two_value
            ).item()
        ),

        bool(
            torch.isfinite(
                bs_value
            ).item()
        ),

        bool(
            torch.isfinite(
                la_one_value
            ).item()
        ),

        bool(
            torch.isfinite(
                ldam_value
            ).item()
        ),
    ])


    overall_pass = all([

        focal_ce_pass,
        bs_la_pass,
        ldam_margin_direction_pass,
        ldam_max_margin_pass,
        drw_schedule_pass,
        drw_positive_pass,
        drw_finite_pass,
        drw_mean_pass,
        drw_tail_direction_pass,
        finite_losses_pass,
    ])


    LOSS_SANITY_RESULTS[
        dataset_name
    ] = bool(
        overall_pass
    )


    print(
        f"\n{dataset_name.upper()}"
    )

    print("-" * 82)

    print(
        "Focal gamma=0 == CE       :",
        "PASS"
        if focal_ce_pass
        else "FAIL"
    )

    print(
        "BalancedSoftmax == LA τ=1 :",
        "PASS"
        if bs_la_pass
        else "FAIL"
    )

    print(
        "LDAM rare margin > head   :",
        "PASS"
        if ldam_margin_direction_pass
        else "FAIL"
    )

    print(
        "LDAM max margin exact     :",
        "PASS"
        if ldam_max_margin_pass
        else "FAIL"
    )

    print(
        "DRW off@79 / on@80        :",
        "PASS"
        if drw_schedule_pass
        else "FAIL"
    )

    print(
        "DRW weights positive      :",
        "PASS"
        if drw_positive_pass
        else "FAIL"
    )

    print(
        "DRW weights finite        :",
        "PASS"
        if drw_finite_pass
        else "FAIL"
    )

    print(
        "DRW mean weight = 1       :",
        "PASS"
        if drw_mean_pass
        else "FAIL"
    )

    print(
        "DRW rare > head weight    :",
        "PASS"
        if drw_tail_direction_pass
        else "FAIL"
    )

    print(
        "All losses finite         :",
        "PASS"
        if finite_losses_pass
        else "FAIL"
    )

    print(
        "STATUS                    :",
        "PASS"
        if overall_pass
        else "FAIL"
    )


ALL_LOSS_SANITY_PASS = all(
    LOSS_SANITY_RESULTS.values()
)


print(
    "\n" + "=" * 115
)

print(
    "LONG-TAIL LOSS SANITY GATE:",
    "PASS"
    if ALL_LOSS_SANITY_PASS
    else "FAIL"
)

print("=" * 115)

LONG-TAIL TRAINING LOSS — MATHEMATICAL SANITY AUDIT

PINGAN
----------------------------------------------------------------------------------
Focal gamma=0 == CE       : PASS
BalancedSoftmax == LA τ=1 : PASS
LDAM rare margin > head   : PASS
LDAM max margin exact     : PASS
DRW off@79 / on@80        : PASS
DRW weights positive      : PASS
DRW weights finite        : PASS
DRW mean weight = 1       : PASS
DRW rare > head weight    : PASS
All losses finite         : PASS
STATUS                    : PASS

QINGYUN
----------------------------------------------------------------------------------
Focal gamma=0 == CE       : PASS
BalancedSoftmax == LA τ=1 : PASS
LDAM rare margin > head   : PASS
LDAM max margin exact     : PASS
DRW off@79 / on@80        : PASS
DRW weights positive      : PASS
DRW weights finite        : PASS
DRW mean weight = 1       : PASS
DRW rare > head weight    : PASS
All losses finite         : PASS
STATUS                    : PASS

TANGDAOWAN
---------------------------

In [ ]:
# ============================================================
# CELL 18 — FREEZE NOTEBOOK 04 TRAINING PROTOCOL
# ============================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:

                break

            h.update(
                chunk
            )


    return h.hexdigest()


NOTEBOOK04_PROTOCOL = {

    "project":
        "CSE498R LTLC",

    "notebook":
        "04_LongTail_Training_Baselines",

    "protocol_version":
        "v3.1_IMPLEMENTATION_LOCKED",

    "datasets": {
        name: {
            "classes":
                int(
                    DATASETS[
                        name
                    ][
                        "num_classes"
                    ]
                ),

            "pca_components":
                int(
                    DATASETS[
                        name
                    ][
                        "pca_components"
                    ]
                ),

            "patch_size":
                int(
                    DATASETS[
                        name
                    ][
                        "patch_size"
                    ]
                ),
        }

        for name in DATASETS
    },

    "training_seeds":
        TRAINING_SEEDS,

    "tuning_seed":
        TUNING_SEED,

    "common_training":
        COMMON_TRAINING_CONFIG,

    "baseline_search_space":
        BASELINE_SEARCH_SPACE,

    "candidate_selection": {

        "primary":
            "validation_AA",

        "tie_1":
            "validation_Macro_F1",

        "tie_2":
            "validation_OA",

        "final_tie":
            "smaller_simpler_candidate",
    },

    "checkpoint_selection": {

        "primary":
            "validation_AA",

        "tie_1":
            "validation_Macro_F1",

        "tie_2":
            "validation_OA",

        "tie_3":
            "lower_validation_loss",

        "final_tie":
            "earlier_epoch",
    },

    "expected_new_training_runs":
        EXPECTED_NOTEBOOK04_RUNS,

    "expected_core_training_runs_including_CE":
        EXPECTED_CORE_PROJECT_TRAININGS,

    "test_used_for_hyperparameter_selection":
        False,

    "test_used_for_checkpoint_selection":
        False,
}


NOTEBOOK04_PROTOCOL_PATH = (

    NOTEBOOK04_AUDIT_DIR /

    "baseline_training_protocol_v3_1_locked.json"
)


serialized_protocol = json.dumps(

    NOTEBOOK04_PROTOCOL,

    indent=2,

    sort_keys=True
)


if NOTEBOOK04_PROTOCOL_PATH.exists():

    with open(
        NOTEBOOK04_PROTOCOL_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        existing_protocol = (
            json.load(f)
        )


    protocol_match_pass = (

        existing_protocol

        ==

        NOTEBOOK04_PROTOCOL
    )


    if not protocol_match_pass:

        raise RuntimeError(
            "An existing Notebook 04 protocol file "
            "does not match the current locked protocol."
        )

else:

    with open(
        NOTEBOOK04_PROTOCOL_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            serialized_protocol
        )


protocol_sha256 = sha256_file(
    NOTEBOOK04_PROTOCOL_PATH
)


NOTEBOOK04_PRETRAIN_GATE = {

    "project_paths":
        bool(
            ALL_PROJECT_PATHS_PASS
        ),

    "notebook03_frozen":
        bool(
            NB03_GATE_PASS
        ),

    "gpu":
        bool(
            CUDA_AVAILABLE
        ),

    "frozen_splits":
        bool(
            ALL_SPLITS_PASS
        ),

    "training_class_statistics":
        bool(
            ALL_CLASS_STATS_PASS
        ),

    "loss_equations":
        bool(
            ALL_LOSS_SANITY_PASS
        ),

    "search_budget_45_runs":
        (
            EXPECTED_NOTEBOOK04_RUNS
            ==
            45
        ),

    "official_test_not_used_for_tuning":
        True,

    "protocol_file_saved":
        NOTEBOOK04_PROTOCOL_PATH.exists(),
}


NOTEBOOK04_PRETRAIN_GATE_PASS = all(
    NOTEBOOK04_PRETRAIN_GATE.values()
)


print("=" * 115)
print("LTLC NOTEBOOK 04 — PRE-TRAINING PROTOCOL FREEZE")
print("=" * 115)


for name, passed in (
    NOTEBOOK04_PRETRAIN_GATE.items()
):

    print(
        f"{'PASS' if passed else 'FAIL':5s} | "
        f"{name}"
    )


print("-" * 115)

print(
    "Protocol file :",
    NOTEBOOK04_PROTOCOL_PATH
)

print(
    "Protocol SHA  :",
    protocol_sha256
)

print(
    "New runs      :",
    EXPECTED_NOTEBOOK04_RUNS
)

print(
    "Core total    :",
    EXPECTED_CORE_PROJECT_TRAININGS
)

print("=" * 115)

print(
    "NOTEBOOK 04 PRE-TRAINING GATE:",
    "PASS"
    if NOTEBOOK04_PRETRAIN_GATE_PASS
    else "FAIL"
)

print("=" * 115)


if not NOTEBOOK04_PRETRAIN_GATE_PASS:

    raise RuntimeError(
        "Notebook 04 pre-training gate failed. "
        "Do not begin baseline training."
    )

LTLC NOTEBOOK 04 — PRE-TRAINING PROTOCOL FREEZE
PASS  | project_paths
PASS  | notebook03_frozen
PASS  | gpu
PASS  | frozen_splits
PASS  | training_class_statistics
PASS  | loss_equations
PASS  | search_budget_45_runs
PASS  | official_test_not_used_for_tuning
PASS  | protocol_file_saved
-------------------------------------------------------------------------------------------------------------------
Protocol file : /content/drive/MyDrive/CSE498/LTLC/audit/notebook04_longtail/baseline_training_protocol_v3_1_locked.json
Protocol SHA  : 053179d263b118bea1da4a7c02c128c2c160cb13398ebc4adaa2027577c0d577
New runs      : 45
Core total    : 54
NOTEBOOK 04 PRE-TRAINING GATE: PASS


In [ ]:
# ============================================================
# CELL 19 — LOCAL PCA CACHE + SHA-256 VERIFICATION
# ============================================================

LOCAL_DATA_DIR = Path(
    "/content/ltlc_notebook04_data"
)

LOCAL_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def resolve_frozen_cube_path(
    dataset_name
):

    recorded = Path(
        DATASETS[
            dataset_name
        ][
            "normalized_cube_file"
        ]
    )


    candidates = []


    # Exact recorded path
    candidates.append(
        recorded
    )


    # File name under canonical cache folder
    candidates.append(
        PCA_CACHE_DIR /
        recorded.name
    )


    # Possible project-relative path
    candidates.append(
        PROJECT_ROOT /
        recorded
    )


    for candidate in candidates:

        if candidate.exists():

            return candidate


    raise FileNotFoundError(
        f"Could not resolve frozen PCA cube for "
        f"{dataset_name}. Recorded value: {recorded}"
    )


LOCAL_CUBE_PATHS = {}
LOCAL_CUBE_MMAPS = {}

PCA_CACHE_AUDIT = {}


print("=" * 115)
print("NOTEBOOK 04 — FROZEN PCA CACHE LOCALIZATION")
print("=" * 115)


for dataset_name in DATASETS:

    source_path = (
        resolve_frozen_cube_path(
            dataset_name
        )
    )


    expected_sha = (
        DATASETS[
            dataset_name
        ][
            "normalized_cube_sha256"
        ]
    )


    # --------------------------------------------------------
    # Verify immutable Drive source first
    # --------------------------------------------------------

    source_sha = sha256_file(
        source_path
    )


    source_hash_pass = (
        source_sha
        ==
        expected_sha
    )


    if not source_hash_pass:

        raise RuntimeError(
            f"{dataset_name}: frozen PCA source SHA mismatch."
        )


    local_path = (
        LOCAL_DATA_DIR /
        source_path.name
    )


    # --------------------------------------------------------
    # Reuse valid local copy if already present.
    # Otherwise copy from frozen Drive source.
    # --------------------------------------------------------

    needs_copy = True


    if local_path.exists():

        existing_local_sha = (
            sha256_file(
                local_path
            )
        )

        if (
            existing_local_sha
            ==
            expected_sha
        ):

            needs_copy = False


    if needs_copy:

        if local_path.exists():

            local_path.unlink()


        shutil.copy2(
            source_path,
            local_path
        )


    local_sha = sha256_file(
        local_path
    )


    local_hash_pass = (
        local_sha
        ==
        expected_sha
    )


    if not local_hash_pass:

        raise RuntimeError(
            f"{dataset_name}: local PCA cache SHA mismatch."
        )


    cube = np.load(
        local_path,
        mmap_mode="r"
    )


    H = int(
        DATASETS[
            dataset_name
        ][
            "cube_shape"
        ][0]
    )

    W = int(
        DATASETS[
            dataset_name
        ][
            "cube_shape"
        ][1]
    )

    C = int(
        DATASETS[
            dataset_name
        ][
            "pca_components"
        ]
    )


    expected_shape = (
        H,
        W,
        C
    )


    shape_pass = (
        tuple(
            cube.shape
        )
        ==
        expected_shape
    )


    dtype_pass = (
        cube.dtype
        ==
        np.float32
    )


    overall_pass = all([
        source_hash_pass,
        local_hash_pass,
        shape_pass,
        dtype_pass,
    ])


    if not overall_pass:

        raise RuntimeError(
            f"{dataset_name}: PCA cache audit failed."
        )


    LOCAL_CUBE_PATHS[
        dataset_name
    ] = local_path


    LOCAL_CUBE_MMAPS[
        dataset_name
    ] = cube


    PCA_CACHE_AUDIT[
        dataset_name
    ] = bool(
        overall_pass
    )


    print(
        f"\n{dataset_name.upper()}"
    )

    print("-" * 78)

    print(
        "Drive SHA-256   :",
        "PASS"
    )

    print(
        "Local SHA-256   :",
        "PASS"
    )

    print(
        "Shape           :",
        cube.shape,
        "PASS"
        if shape_pass
        else "FAIL"
    )

    print(
        "Dtype           :",
        cube.dtype,
        "PASS"
        if dtype_pass
        else "FAIL"
    )

    print(
        "Local path      :",
        local_path
    )

    print(
        "STATUS          :",
        "PASS"
    )


ALL_PCA_CACHE_LOCAL_PASS = all(
    PCA_CACHE_AUDIT.values()
)


print(
    "\n" + "=" * 115
)

print(
    "LOCAL FROZEN PCA CACHE GATE:",
    "PASS"
    if ALL_PCA_CACHE_LOCAL_PASS
    else "FAIL"
)

print("=" * 115)

NOTEBOOK 04 — FROZEN PCA CACHE LOCALIZATION

PINGAN
------------------------------------------------------------------------------
Drive SHA-256   : PASS
Local SHA-256   : PASS
Shape           : (1230, 1000, 15) PASS
Dtype           : float32 PASS
Local path      : /content/ltlc_notebook04_data/pingan_pca15_rowminmax_float32.npy
STATUS          : PASS

QINGYUN
------------------------------------------------------------------------------
Drive SHA-256   : PASS
Local SHA-256   : PASS
Shape           : (880, 1360, 15) PASS
Dtype           : float32 PASS
Local path      : /content/ltlc_notebook04_data/qingyun_pca15_rowminmax_float32.npy
STATUS          : PASS

TANGDAOWAN
------------------------------------------------------------------------------
Drive SHA-256   : PASS
Local SHA-256   : PASS
Shape           : (1740, 860, 20) PASS
Dtype           : float32 PASS
Local path      : /content/ltlc_notebook04_data/tangdaowan_pca20_rowminmax_float32.npy
STATUS          : PASS

LOCAL FROZEN PCA 

In [ ]:
# ============================================================
# CELL 20 — LAZY ZERO-PADDED PATCH DATASET
# ============================================================

class FrozenHSIPatchDataset(
    Dataset
):

    def __init__(
        self,
        cube,
        flat_indices,
        labels,
        patch_size
    ):

        self.cube = cube

        self.flat_indices = np.asarray(
            flat_indices,
            dtype=np.int64
        )

        self.labels = np.asarray(
            labels,
            dtype=np.int64
        )

        self.patch_size = int(
            patch_size
        )

        self.margin = (
            self.patch_size
            //
            2
        )


        if (
            self.patch_size
            %
            2
            !=
            1
        ):

            raise ValueError(
                "Patch size must be odd."
            )


        if (
            len(
                self.flat_indices
            )
            !=
            len(
                self.labels
            )
        ):

            raise ValueError(
                "Index/label length mismatch."
            )


        self.H = int(
            cube.shape[0]
        )

        self.W = int(
            cube.shape[1]
        )

        self.C = int(
            cube.shape[2]
        )


    def __len__(
        self
    ):

        return len(
            self.flat_indices
        )


    def _extract_patch(
        self,
        flat_index
    ):

        row = int(
            flat_index
            //
            self.W
        )

        col = int(
            flat_index
            %
            self.W
        )


        m = self.margin
        p = self.patch_size


        # ----------------------------------------------------
        # Fast path: center is fully inside cube
        # ----------------------------------------------------

        if (
            row - m >= 0
            and
            row + m < self.H
            and
            col - m >= 0
            and
            col + m < self.W
        ):

            patch = np.asarray(

                self.cube[
                    row - m:
                    row + m + 1,

                    col - m:
                    col + m + 1,

                    :
                ],

                dtype=np.float32

            ).copy()


        # ----------------------------------------------------
        # Edge path: exact zero padding
        # ----------------------------------------------------

        else:

            patch = np.zeros(
                (
                    p,
                    p,
                    self.C
                ),
                dtype=np.float32
            )


            src_r0 = max(
                row - m,
                0
            )

            src_r1 = min(
                row + m + 1,
                self.H
            )

            src_c0 = max(
                col - m,
                0
            )

            src_c1 = min(
                col + m + 1,
                self.W
            )


            dst_r0 = (
                src_r0
                -
                (
                    row - m
                )
            )

            dst_r1 = (
                dst_r0
                +
                (
                    src_r1
                    -
                    src_r0
                )
            )

            dst_c0 = (
                src_c0
                -
                (
                    col - m
                )
            )

            dst_c1 = (
                dst_c0
                +
                (
                    src_c1
                    -
                    src_c0
                )
            )


            patch[
                dst_r0:dst_r1,
                dst_c0:dst_c1,
                :
            ] = self.cube[
                src_r0:src_r1,
                src_c0:src_c1,
                :
            ]


        return patch


    def __getitem__(
        self,
        item
    ):

        flat_index = int(
            self.flat_indices[
                item
            ]
        )


        patch = self._extract_patch(
            flat_index
        )


        # H × W × C
        # →
        # 1 × C × H × W

        patch_tensor = torch.from_numpy(

            np.transpose(
                patch,
                (
                    2,
                    0,
                    1
                )
            ).copy()

        ).unsqueeze(
            0
        )


        label_tensor = torch.tensor(
            int(
                self.labels[
                    item
                ]
            ),
            dtype=torch.long
        )


        index_tensor = torch.tensor(
            flat_index,
            dtype=torch.long
        )


        return (
            patch_tensor,
            label_tensor,
            index_tensor
        )


print(
    "FrozenHSIPatchDataset: READY"
)

FrozenHSIPatchDataset: READY


In [ ]:
# ============================================================
# CELL 21 — TRAIN/VALIDATION DATASETS + PATCH AUDIT
# ============================================================

TRAIN_DATASETS = {}
VAL_DATASETS = {}

PATCH_PIPELINE_AUDIT = {}


print("=" * 115)
print("NOTEBOOK 04 — TRAIN/VALIDATION LAZY PATCH AUDIT")
print("=" * 115)


for dataset_name in DATASETS:

    split = FROZEN_SPLITS[
        dataset_name
    ]


    cube = LOCAL_CUBE_MMAPS[
        dataset_name
    ]


    patch_size = int(
        DATASETS[
            dataset_name
        ][
            "patch_size"
        ]
    )


    train_ds = FrozenHSIPatchDataset(

        cube=cube,

        flat_indices=
            split[
                "train_indices"
            ],

        labels=
            split[
                "train_labels_model"
            ],

        patch_size=
            patch_size
    )


    val_ds = FrozenHSIPatchDataset(

        cube=cube,

        flat_indices=
            split[
                "val_indices"
            ],

        labels=
            split[
                "val_labels_model"
            ],

        patch_size=
            patch_size
    )


    TRAIN_DATASETS[
        dataset_name
    ] = train_ds


    VAL_DATASETS[
        dataset_name
    ] = val_ds


    # --------------------------------------------------------
    # Inspect deterministic positions
    # --------------------------------------------------------

    audit_positions = [
        0,
        len(
            train_ds
        ) // 2,
        len(
            train_ds
        ) - 1,
    ]


    sample_passes = []


    for position in audit_positions:

        x, y, flat_idx = (
            train_ds[
                position
            ]
        )


        expected_index = int(
            split[
                "train_indices"
            ][
                position
            ]
        )


        expected_label = int(
            split[
                "train_labels_model"
            ][
                position
            ]
        )


        expected_shape = (
            1,
            int(
                DATASETS[
                    dataset_name
                ][
                    "pca_components"
                ]
            ),
            patch_size,
            patch_size,
        )


        shape_pass = (
            tuple(
                x.shape
            )
            ==
            expected_shape
        )


        dtype_pass = (
            x.dtype
            ==
            torch.float32
        )


        index_pass = (
            int(
                flat_idx.item()
            )
            ==
            expected_index
        )


        label_pass = (
            int(
                y.item()
            )
            ==
            expected_label
        )


        H = cube.shape[0]
        W = cube.shape[1]

        row = (
            expected_index
            //
            W
        )

        col = (
            expected_index
            %
            W
        )


        m = (
            patch_size
            //
            2
        )


        observed_center = (
            x[
                0,
                :,
                m,
                m
            ]
            .numpy()
        )


        expected_center = np.asarray(
            cube[
                row,
                col,
                :
            ],
            dtype=np.float32
        )


        center_pass = np.array_equal(
            observed_center,
            expected_center
        )


        finite_pass = bool(
            torch.isfinite(
                x
            ).all()
        )


        sample_passes.append(
            all([
                shape_pass,
                dtype_pass,
                index_pass,
                label_pass,
                center_pass,
                finite_pass,
            ])
        )


    overall_pass = all(
        sample_passes
    )


    PATCH_PIPELINE_AUDIT[
        dataset_name
    ] = bool(
        overall_pass
    )


    print(
        f"\n{dataset_name.upper()}"
    )

    print("-" * 80)

    print(
        "Train samples :",
        len(
            train_ds
        )
    )

    print(
        "Val samples   :",
        len(
            val_ds
        )
    )

    print(
        "Tensor shape  :",
        tuple(
            train_ds[0][0].shape
        )
    )

    print(
        "Index audit   :",
        "PASS"
        if overall_pass
        else "FAIL"
    )

    print(
        "Label audit   :",
        "PASS"
        if overall_pass
        else "FAIL"
    )

    print(
        "Center audit  :",
        "PASS"
        if overall_pass
        else "FAIL"
    )

    print(
        "Finite audit  :",
        "PASS"
        if overall_pass
        else "FAIL"
    )

    print(
        "STATUS        :",
        "PASS"
        if overall_pass
        else "FAIL"
    )


ALL_PATCH_PIPELINE_PASS = all(
    PATCH_PIPELINE_AUDIT.values()
)


print(
    "\n" + "=" * 115
)

print(
    "TRAIN/VAL PATCH PIPELINE GATE:",
    "PASS"
    if ALL_PATCH_PIPELINE_PASS
    else "FAIL"
)

print("=" * 115)

NOTEBOOK 04 — TRAIN/VALIDATION LAZY PATCH AUDIT

PINGAN
--------------------------------------------------------------------------------
Train samples : 79869
Val samples   : 34230
Tensor shape  : (1, 15, 13, 13)
Index audit   : PASS
Label audit   : PASS
Center audit  : PASS
Finite audit  : PASS
STATUS        : PASS

QINGYUN
--------------------------------------------------------------------------------
Train samples : 66844
Val samples   : 28648
Tensor shape  : (1, 15, 11, 11)
Index audit   : PASS
Label audit   : PASS
Center audit  : PASS
Finite audit  : PASS
STATUS        : PASS

TANGDAOWAN
--------------------------------------------------------------------------------
Train samples : 24760
Val samples   : 10612
Tensor shape  : (1, 20, 9, 9)
Index audit   : PASS
Label audit   : PASS
Center audit  : PASS
Finite audit  : PASS
STATUS        : PASS

TRAIN/VAL PATCH PIPELINE GATE: PASS


In [ ]:
# ============================================================
# CELL 22 — DETERMINISTIC TRAIN/VALIDATION DATALOADERS
# ============================================================

NUM_WORKERS = 0
PIN_MEMORY = True


def make_train_val_loaders(
    dataset_name,
    seed
):

    generator = torch.Generator()

    generator.manual_seed(
        int(
            seed
        )
    )


    train_loader = DataLoader(

        TRAIN_DATASETS[
            dataset_name
        ],

        batch_size=
            BATCH_SIZE,

        shuffle=
            True,

        num_workers=
            NUM_WORKERS,

        pin_memory=
            PIN_MEMORY,

        drop_last=
            False,

        generator=
            generator,
    )


    val_loader = DataLoader(

        VAL_DATASETS[
            dataset_name
        ],

        batch_size=
            BATCH_SIZE,

        shuffle=
            False,

        num_workers=
            NUM_WORKERS,

        pin_memory=
            PIN_MEMORY,

        drop_last=
            False,
    )


    return (
        train_loader,
        val_loader
    )


DATALOADER_AUDIT = {}


print("=" * 115)
print("NOTEBOOK 04 — TRAIN/VALIDATION DATALOADER AUDIT")
print("=" * 115)


for dataset_name in DATASETS:

    train_loader, val_loader = (
        make_train_val_loaders(
            dataset_name,
            seed=42
        )
    )


    train_x, train_y, train_idx = (
        next(
            iter(
                train_loader
            )
        )
    )


    val_x, val_y, val_idx = (
        next(
            iter(
                val_loader
            )
        )
    )


    C = int(
        DATASETS[
            dataset_name
        ][
            "pca_components"
        ]
    )

    P = int(
        DATASETS[
            dataset_name
        ][
            "patch_size"
        ]
    )

    K = int(
        DATASETS[
            dataset_name
        ][
            "num_classes"
        ]
    )


    expected_sample_shape = (
        1,
        C,
        P,
        P
    )


    train_shape_pass = (
        tuple(
            train_x.shape[1:]
        )
        ==
        expected_sample_shape
    )


    val_shape_pass = (
        tuple(
            val_x.shape[1:]
        )
        ==
        expected_sample_shape
    )


    dtype_pass = all([

        train_x.dtype
        ==
        torch.float32,

        val_x.dtype
        ==
        torch.float32,

        train_y.dtype
        ==
        torch.int64,

        val_y.dtype
        ==
        torch.int64,

        train_idx.dtype
        ==
        torch.int64,

        val_idx.dtype
        ==
        torch.int64,
    ])


    label_range_pass = all([

        int(
            train_y.min()
        )
        >=
        0,

        int(
            train_y.max()
        )
        <
        K,

        int(
            val_y.min()
        )
        >=
        0,

        int(
            val_y.max()
        )
        <
        K,
    ])


    # Validation must remain ordered.
    expected_first_val_indices = (
        FROZEN_SPLITS[
            dataset_name
        ][
            "val_indices"
        ][
            :len(
                val_idx
            )
        ]
    )


    val_order_pass = np.array_equal(

        val_idx.numpy(),

        expected_first_val_indices
    )


    finite_pass = all([

        bool(
            torch.isfinite(
                train_x
            ).all()
        ),

        bool(
            torch.isfinite(
                val_x
            ).all()
        ),
    ])


    overall_pass = all([

        train_shape_pass,
        val_shape_pass,
        dtype_pass,
        label_range_pass,
        val_order_pass,
        finite_pass,
    ])


    DATALOADER_AUDIT[
        dataset_name
    ] = bool(
        overall_pass
    )


    print(
        f"\n{dataset_name.upper()}"
    )

    print("-" * 82)

    print(
        "Train batch:",
        tuple(
            train_x.shape
        )
    )

    print(
        "Val batch  :",
        tuple(
            val_x.shape
        )
    )

    print(
        "Dtypes     :",
        "PASS"
        if dtype_pass
        else "FAIL"
    )

    print(
        "Label range:",
        "PASS"
        if label_range_pass
        else "FAIL"
    )

    print(
        "Val order  :",
        "PASS"
        if val_order_pass
        else "FAIL"
    )

    print(
        "Finite     :",
        "PASS"
        if finite_pass
        else "FAIL"
    )

    print(
        "STATUS     :",
        "PASS"
        if overall_pass
        else "FAIL"
    )


ALL_DATALOADER_PASS = all(
    DATALOADER_AUDIT.values()
)


print(
    "\n" + "=" * 115
)

print(
    "TRAIN/VAL DATALOADER GATE:",
    "PASS"
    if ALL_DATALOADER_PASS
    else "FAIL"
)

print("=" * 115)

NOTEBOOK 04 — TRAIN/VALIDATION DATALOADER AUDIT

PINGAN
----------------------------------------------------------------------------------
Train batch: (256, 1, 15, 13, 13)
Val batch  : (256, 1, 15, 13, 13)
Dtypes     : PASS
Label range: PASS
Val order  : PASS
Finite     : PASS
STATUS     : PASS

QINGYUN
----------------------------------------------------------------------------------
Train batch: (256, 1, 15, 11, 11)
Val batch  : (256, 1, 15, 11, 11)
Dtypes     : PASS
Label range: PASS
Val order  : PASS
Finite     : PASS
STATUS     : PASS

TANGDAOWAN
----------------------------------------------------------------------------------
Train batch: (256, 1, 20, 9, 9)
Val batch  : (256, 1, 20, 9, 9)
Dtypes     : PASS
Label range: PASS
Val order  : PASS
Finite     : PASS
STATUS     : PASS

TRAIN/VAL DATALOADER GATE: PASS


In [ ]:
# ============================================================
# CELL 23 — EXACT HYBRIDSN BACKBONE
# ============================================================

class HybridSN(
    nn.Module
):

    def __init__(
        self,
        spectral_channels,
        patch_size,
        num_classes,
        dropout=0.4
    ):

        super().__init__()


        self.spectral_channels = int(
            spectral_channels
        )

        self.patch_size = int(
            patch_size
        )

        self.num_classes = int(
            num_classes
        )


        self.conv3d_1 = nn.Conv3d(
            in_channels=1,
            out_channels=8,
            kernel_size=(
                7,
                3,
                3
            )
        )


        self.conv3d_2 = nn.Conv3d(
            in_channels=8,
            out_channels=16,
            kernel_size=(
                5,
                3,
                3
            )
        )


        self.conv3d_3 = nn.Conv3d(
            in_channels=16,
            out_channels=32,
            kernel_size=(
                3,
                3,
                3
            )
        )


        spectral_after_3d = (
            self.spectral_channels
            -
            12
        )


        if spectral_after_3d <= 0:

            raise ValueError(
                "Too few spectral channels for HybridSN."
            )


        conv2d_in_channels = (
            32
            *
            spectral_after_3d
        )


        self.conv2d = nn.Conv2d(
            in_channels=
                conv2d_in_channels,

            out_channels=
                64,

            kernel_size=
                3
        )


        spatial_after_2d = (
            self.patch_size
            -
            8
        )


        if spatial_after_2d <= 0:

            raise ValueError(
                "Patch too small for HybridSN."
            )


        self.flat_dim = (
            64
            *
            spatial_after_2d
            *
            spatial_after_2d
        )


        self.fc1 = nn.Linear(
            self.flat_dim,
            256
        )


        self.dropout1 = nn.Dropout(
            dropout
        )


        self.fc2 = nn.Linear(
            256,
            128
        )


        self.dropout2 = nn.Dropout(
            dropout
        )


        self.classifier = nn.Linear(
            128,
            self.num_classes
        )


    def forward_features(
        self,
        x
    ):

        x = F.relu(
            self.conv3d_1(
                x
            )
        )


        x = F.relu(
            self.conv3d_2(
                x
            )
        )


        x = F.relu(
            self.conv3d_3(
                x
            )
        )


        batch_size = x.shape[0]


        x = x.reshape(
            batch_size,
            (
                x.shape[1]
                *
                x.shape[2]
            ),
            x.shape[3],
            x.shape[4]
        )


        x = F.relu(
            self.conv2d(
                x
            )
        )


        return x


    def forward(
        self,
        x
    ):

        x = self.forward_features(
            x
        )


        x = torch.flatten(
            x,
            start_dim=1
        )


        x = F.relu(
            self.fc1(
                x
            )
        )


        x = self.dropout1(
            x
        )


        x = F.relu(
            self.fc2(
                x
            )
        )


        x = self.dropout2(
            x
        )


        logits = self.classifier(
            x
        )


        return logits


print(
    "HybridSN backbone: READY"
)

HybridSN backbone: READY


In [ ]:
# ============================================================
# CELL 24 — HYBRIDSN ARCHITECTURE EQUIVALENCE AUDIT
# ============================================================

EXPECTED_HYBRIDSN = {

    "Pingan": {
        "after_3d":
            (
                32,
                3,
                7,
                7
            ),

        "after_2d":
            (
                64,
                5,
                5
            ),

        "flat_dim":
            1600,

        "params":
            519546,
    },


    "Qingyun": {
        "after_3d":
            (
                32,
                3,
                5,
                5
            ),

        "after_2d":
            (
                64,
                3,
                3
            ),

        "flat_dim":
            576,

        "params":
            256886,
    },


    "Tangdaowan": {
        "after_3d":
            (
                32,
                8,
                3,
                3
            ),

        "after_2d":
            (
                64,
                1,
                1
            ),

        "flat_dim":
            64,

        "params":
            219264,
    },
}


ARCHITECTURE_AUDIT = {}


print("=" * 115)
print("NOTEBOOK 03 → NOTEBOOK 04 HYBRIDSN EQUIVALENCE")
print("=" * 115)


for dataset_name in DATASETS:

    cfg = DATASETS[
        dataset_name
    ]


    model = HybridSN(

        spectral_channels=
            cfg[
                "pca_components"
            ],

        patch_size=
            cfg[
                "patch_size"
            ],

        num_classes=
            cfg[
                "num_classes"
            ],

        dropout=
            0.4
    )


    model.eval()


    C = cfg[
        "pca_components"
    ]

    P = cfg[
        "patch_size"
    ]

    K = cfg[
        "num_classes"
    ]


    dummy = torch.zeros(
        2,
        1,
        C,
        P,
        P,
        dtype=torch.float32
    )


    with torch.no_grad():

        x1 = F.relu(
            model.conv3d_1(
                dummy
            )
        )

        x2 = F.relu(
            model.conv3d_2(
                x1
            )
        )

        x3 = F.relu(
            model.conv3d_3(
                x2
            )
        )


        after_3d = tuple(
            x3.shape[1:]
        )


        reshaped = x3.reshape(
            x3.shape[0],
            x3.shape[1]
            *
            x3.shape[2],
            x3.shape[3],
            x3.shape[4]
        )


        x4 = F.relu(
            model.conv2d(
                reshaped
            )
        )


        after_2d = tuple(
            x4.shape[1:]
        )


        logits = model(
            dummy
        )


    params = sum(
        p.numel()
        for p in model.parameters()
    )


    expected = (
        EXPECTED_HYBRIDSN[
            dataset_name
        ]
    )


    after3d_pass = (
        after_3d
        ==
        expected[
            "after_3d"
        ]
    )


    after2d_pass = (
        after_2d
        ==
        expected[
            "after_2d"
        ]
    )


    flat_pass = (
        model.flat_dim
        ==
        expected[
            "flat_dim"
        ]
    )


    params_pass = (
        params
        ==
        expected[
            "params"
        ]
    )


    logits_pass = (
        tuple(
            logits.shape
        )
        ==
        (
            2,
            K
        )
    )


    overall_pass = all([

        after3d_pass,
        after2d_pass,
        flat_pass,
        params_pass,
        logits_pass,
    ])


    ARCHITECTURE_AUDIT[
        dataset_name
    ] = bool(
        overall_pass
    )


    print(
        f"\n{dataset_name.upper()}"
    )

    print("-" * 82)

    print(
        "After 3D convs :",
        after_3d,
        "PASS"
        if after3d_pass
        else "FAIL"
    )

    print(
        "After 2D conv  :",
        after_2d,
        "PASS"
        if after2d_pass
        else "FAIL"
    )

    print(
        "Flat dim       :",
        model.flat_dim,
        "PASS"
        if flat_pass
        else "FAIL"
    )

    print(
        "Parameters     :",
        params,
        "PASS"
        if params_pass
        else "FAIL"
    )

    print(
        "Output logits  :",
        tuple(
            logits.shape
        ),
        "PASS"
        if logits_pass
        else "FAIL"
    )

    print(
        "STATUS         :",
        "PASS"
        if overall_pass
        else "FAIL"
    )


ALL_ARCHITECTURE_PASS = all(
    ARCHITECTURE_AUDIT.values()
)


print(
    "\n" + "=" * 115
)

print(
    "HYBRIDSN ARCHITECTURE EQUIVALENCE GATE:",
    "PASS"
    if ALL_ARCHITECTURE_PASS
    else "FAIL"
)

print("=" * 115)

NOTEBOOK 03 → NOTEBOOK 04 HYBRIDSN EQUIVALENCE

PINGAN
----------------------------------------------------------------------------------
After 3D convs : (32, 3, 7, 7) PASS
After 2D conv  : (64, 5, 5) PASS
Flat dim       : 1600 PASS
Parameters     : 519546 PASS
Output logits  : (2, 10) PASS
STATUS         : PASS

QINGYUN
----------------------------------------------------------------------------------
After 3D convs : (32, 3, 5, 5) PASS
After 2D conv  : (64, 3, 3) PASS
Flat dim       : 576 PASS
Parameters     : 256886 PASS
Output logits  : (2, 6) PASS
STATUS         : PASS

TANGDAOWAN
----------------------------------------------------------------------------------
After 3D convs : (32, 8, 3, 3) PASS
After 2D conv  : (64, 1, 1) PASS
Flat dim       : 64 PASS
Parameters     : 219264 PASS
Output logits  : (2, 16) PASS
STATUS         : PASS

HYBRIDSN ARCHITECTURE EQUIVALENCE GATE: PASS


In [ ]:
# ============================================================
# CELL 25 — PRE-TRAINING DATA/MODEL PIPELINE GATE
# ============================================================

OFFICIAL_TEST_DATASET_CREATED = False

OFFICIAL_TEST_DATALOADER_CREATED = False


NOTEBOOK04_PIPELINE_GATE = {

    "frozen_pca_cache":
        bool(
            ALL_PCA_CACHE_LOCAL_PASS
        ),

    "lazy_train_val_patches":
        bool(
            ALL_PATCH_PIPELINE_PASS
        ),

    "train_val_dataloaders":
        bool(
            ALL_DATALOADER_PASS
        ),

    "hybridsn_notebook03_equivalent":
        bool(
            ALL_ARCHITECTURE_PASS
        ),

    "official_test_dataset_not_created":
        (
            not
            OFFICIAL_TEST_DATASET_CREATED
        ),

    "official_test_dataloader_not_created":
        (
            not
            OFFICIAL_TEST_DATALOADER_CREATED
        ),

    "protocol_already_frozen":
        bool(
            NOTEBOOK04_PRETRAIN_GATE_PASS
        ),
}


NOTEBOOK04_PIPELINE_GATE_PASS = all(
    NOTEBOOK04_PIPELINE_GATE.values()
)


print("=" * 115)
print("LTLC NOTEBOOK 04 — TRAIN/VALIDATION PIPELINE GATE")
print("=" * 115)


for name, passed in (
    NOTEBOOK04_PIPELINE_GATE.items()
):

    print(
        f"{'PASS' if passed else 'FAIL':5s} | "
        f"{name}"
    )


print("-" * 115)

print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print("=" * 115)

print(
    "NOTEBOOK 04 PIPELINE STATUS:",
    "PASS"
    if NOTEBOOK04_PIPELINE_GATE_PASS
    else "FAIL"
)

print("=" * 115)


if not NOTEBOOK04_PIPELINE_GATE_PASS:

    raise RuntimeError(
        "Train/validation pipeline gate failed."
    )

LTLC NOTEBOOK 04 — TRAIN/VALIDATION PIPELINE GATE
PASS  | frozen_pca_cache
PASS  | lazy_train_val_patches
PASS  | train_val_dataloaders
PASS  | hybridsn_notebook03_equivalent
PASS  | official_test_dataset_not_created
PASS  | official_test_dataloader_not_created
PASS  | protocol_already_frozen
-------------------------------------------------------------------------------------------------------------------
Official-test dataset created    : False
Official-test DataLoader created : False
NOTEBOOK 04 PIPELINE STATUS: PASS


In [ ]:
# ============================================================
# CELL 26 — CLASSIFICATION METRIC ENGINE + SANITY AUDIT
# ============================================================

def compute_classification_metrics(
    y_true,
    y_pred,
    num_classes,
    require_all_classes=True
):

    y_true = np.asarray(
        y_true,
        dtype=np.int64
    )

    y_pred = np.asarray(
        y_pred,
        dtype=np.int64
    )

    K = int(
        num_classes
    )


    if (
        y_true.ndim != 1
        or
        y_pred.ndim != 1
    ):

        raise ValueError(
            "y_true and y_pred must be 1D."
        )


    if (
        len(y_true)
        !=
        len(y_pred)
    ):

        raise ValueError(
            "Prediction/label length mismatch."
        )


    if len(y_true) == 0:

        raise ValueError(
            "Metric input cannot be empty."
        )


    labels = np.arange(
        K,
        dtype=np.int64
    )


    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=labels
    )


    support = (
        cm.sum(
            axis=1
        )
    )


    if (
        require_all_classes
        and
        np.any(
            support == 0
        )
    ):

        missing = np.where(
            support == 0
        )[0].tolist()

        raise RuntimeError(
            f"Validation split missing model classes: "
            f"{missing}"
        )


    class_recall = np.divide(

        np.diag(
            cm
        ).astype(
            np.float64
        ),

        support,

        out=np.zeros(
            K,
            dtype=np.float64
        ),

        where=(
            support > 0
        )
    )


    oa = float(
        accuracy_score(
            y_true,
            y_pred
        )
    )


    aa = float(
        class_recall.mean()
    )


    kappa = float(
        cohen_kappa_score(
            y_true,
            y_pred,
            labels=labels
        )
    )


    macro_f1 = float(
        f1_score(
            y_true,
            y_pred,
            labels=labels,
            average="macro",
            zero_division=0
        )
    )


    return {

        "oa":
            oa,

        "aa":
            aa,

        "kappa":
            kappa,

        "macro_f1":
            macro_f1,

        "class_recall":
            class_recall,

        "confusion_matrix":
            cm,

        "support":
            support,
    }


# ------------------------------------------------------------
# Independent deterministic toy test
# ------------------------------------------------------------

toy_true = np.array([
    0, 0,
    1, 1,
    2, 2
])

toy_pred = np.array([
    0, 1,
    1, 1,
    2, 0
])


toy_metrics = (
    compute_classification_metrics(
        toy_true,
        toy_pred,
        num_classes=3
    )
)


expected_oa = (
    4.0 /
    6.0
)

expected_aa = (
    (
        0.5
        +
        1.0
        +
        0.5
    )
    /
    3.0
)


oa_pass = np.isclose(
    toy_metrics[
        "oa"
    ],
    expected_oa
)


aa_pass = np.isclose(
    toy_metrics[
        "aa"
    ],
    expected_aa
)


recall_pass = np.allclose(

    toy_metrics[
        "class_recall"
    ],

    np.array([
        0.5,
        1.0,
        0.5
    ])
)


finite_pass = all(
    np.isfinite([
        toy_metrics[
            "oa"
        ],
        toy_metrics[
            "aa"
        ],
        toy_metrics[
            "kappa"
        ],
        toy_metrics[
            "macro_f1"
        ],
    ])
)


METRIC_ENGINE_SANITY_PASS = all([
    oa_pass,
    aa_pass,
    recall_pass,
    finite_pass,
])


print("=" * 105)
print("NOTEBOOK 04 — CLASSIFICATION METRIC ENGINE")
print("=" * 105)

print(
    f"Toy OA             : "
    f"{toy_metrics['oa']:.6f}"
)

print(
    f"Expected OA        : "
    f"{expected_oa:.6f}"
)

print(
    "OA formula         :",
    "PASS"
    if oa_pass
    else "FAIL"
)

print()

print(
    f"Toy AA             : "
    f"{toy_metrics['aa']:.6f}"
)

print(
    f"Expected AA        : "
    f"{expected_aa:.6f}"
)

print(
    "AA formula         :",
    "PASS"
    if aa_pass
    else "FAIL"
)

print(
    "Per-class recalls  :",
    toy_metrics[
        "class_recall"
    ].tolist()
)

print(
    "Recall audit       :",
    "PASS"
    if recall_pass
    else "FAIL"
)

print(
    "All metrics finite :",
    "PASS"
    if finite_pass
    else "FAIL"
)

print("-" * 105)

print(
    "METRIC ENGINE SANITY GATE:",
    "PASS"
    if METRIC_ENGINE_SANITY_PASS
    else "FAIL"
)

print("=" * 105)

NOTEBOOK 04 — CLASSIFICATION METRIC ENGINE
Toy OA             : 0.666667
Expected OA        : 0.666667
OA formula         : PASS

Toy AA             : 0.666667
Expected AA        : 0.666667
AA formula         : PASS
Per-class recalls  : [0.5, 1.0, 0.5]
Recall audit       : PASS
All metrics finite : PASS
---------------------------------------------------------------------------------------------------------
METRIC ENGINE SANITY GATE: PASS


In [ ]:
# ============================================================
# CELL 27 — ADAM + LEGACY KERAS LR DECAY
# ============================================================

BASE_LEARNING_RATE = 1e-3
LEGACY_LR_DECAY = 1e-6

ADAM_BETAS = (
    0.9,
    0.999
)

ADAM_EPSILON = 1e-7


def legacy_keras_learning_rate(
    global_step,
    base_lr=BASE_LEARNING_RATE,
    decay=LEGACY_LR_DECAY
):

    return float(

        base_lr

        /

        (
            1.0
            +
            decay
            *
            int(
                global_step
            )
        )
    )


def make_notebook04_optimizer(
    model
):

    return torch.optim.Adam(

        model.parameters(),

        lr=
            BASE_LEARNING_RATE,

        betas=
            ADAM_BETAS,

        eps=
            ADAM_EPSILON,

        weight_decay=
            0.0
    )


def set_optimizer_legacy_lr(
    optimizer,
    global_step
):

    lr = legacy_keras_learning_rate(
        global_step
    )


    for param_group in (
        optimizer.param_groups
    ):

        param_group[
            "lr"
        ] = lr


    return lr


# ------------------------------------------------------------
# Independent optimizer sanity test
# ------------------------------------------------------------

toy_model = nn.Linear(
    4,
    2
)

toy_optimizer = (
    make_notebook04_optimizer(
        toy_model
    )
)


lr0 = legacy_keras_learning_rate(
    0
)

lr1 = legacy_keras_learning_rate(
    1
)

lr1000 = legacy_keras_learning_rate(
    1000
)


lr_formula_pass = all([

    np.isclose(
        lr0,
        1e-3
    ),

    lr1 < lr0,

    lr1000 < lr1,
])


weight_decay_pass = all(

    group[
        "weight_decay"
    ]
    ==
    0.0

    for group
    in toy_optimizer.param_groups
)


eps_pass = (
    toy_optimizer.defaults[
        "eps"
    ]
    ==
    ADAM_EPSILON
)


beta_pass = (
    toy_optimizer.defaults[
        "betas"
    ]
    ==
    ADAM_BETAS
)


OPTIMIZER_SANITY_PASS = all([
    lr_formula_pass,
    weight_decay_pass,
    eps_pass,
    beta_pass,
])


print("=" * 105)
print("NOTEBOOK 04 — OPTIMIZER COMPATIBILITY AUDIT")
print("=" * 105)

print(
    f"LR step 0    : {lr0:.12f}"
)

print(
    f"LR step 1    : {lr1:.12f}"
)

print(
    f"LR step 1000 : {lr1000:.12f}"
)

print()

print(
    "Legacy decay formula :",
    "PASS"
    if lr_formula_pass
    else "FAIL"
)

print(
    "Weight decay = 0     :",
    "PASS"
    if weight_decay_pass
    else "FAIL"
)

print(
    "Adam epsilon = 1e-7  :",
    "PASS"
    if eps_pass
    else "FAIL"
)

print(
    "Adam betas exact     :",
    "PASS"
    if beta_pass
    else "FAIL"
)

print("-" * 105)

print(
    "OPTIMIZER SANITY GATE:",
    "PASS"
    if OPTIMIZER_SANITY_PASS
    else "FAIL"
)

print("=" * 105)


del toy_model
del toy_optimizer

NOTEBOOK 04 — OPTIMIZER COMPATIBILITY AUDIT
LR step 0    : 0.001000000000
LR step 1    : 0.000999999000
LR step 1000 : 0.000999000999

Legacy decay formula : PASS
Weight decay = 0     : PASS
Adam epsilon = 1e-7  : PASS
Adam betas exact     : PASS
---------------------------------------------------------------------------------------------------------
OPTIMIZER SANITY GATE: PASS


In [ ]:
# ============================================================
# CELL 28 — FROZEN BASELINE LOSS FACTORY
# ============================================================

VALID_BASELINE_METHODS = [
    "Focal",
    "LDAM_DRW",
    "BalancedSoftmax",
    "LA_Loss",
]


def validate_baseline_candidate(
    method,
    candidate
):

    if (
        method
        not in
        VALID_BASELINE_METHODS
    ):

        raise ValueError(
            f"Unknown baseline method: {method}"
        )


    locked_candidates = (
        BASELINE_SEARCH_SPACE[
            method
        ][
            "candidates"
        ]
    )


    if method == "BalancedSoftmax":

        return (
            candidate
            ==
            "training_counts"
        )


    candidate_float = float(
        candidate
    )


    return any(

        np.isclose(
            candidate_float,
            float(
                locked
            )
        )

        for locked in
        locked_candidates
    )


def build_loss_bundle(
    dataset_name,
    method,
    candidate,
    device
):

    if not validate_baseline_candidate(
        method,
        candidate
    ):

        raise ValueError(
            f"Candidate {candidate} is outside "
            f"the frozen search space for {method}."
        )


    arrays = BASELINE_CLASS_ARRAYS[
        dataset_name
    ]


    counts = arrays[
        "counts"
    ]


    priors = arrays[
        "priors"
    ]


    if method == "Focal":

        criterion = FocalLoss(
            gamma=float(
                candidate
            )
        )


        hyperparameters = {
            "gamma":
                float(
                    candidate
                )
        }


        drw_weights = None


    elif method == "BalancedSoftmax":

        criterion = BalancedSoftmaxLoss(
            counts
        )


        hyperparameters = {
            "class_statistics":
                "training_counts"
        }


        drw_weights = None


    elif method == "LA_Loss":

        criterion = LogitAdjustedLoss(
            priors,
            tau=float(
                candidate
            )
        )


        hyperparameters = {
            "tau":
                float(
                    candidate
                )
        }


        drw_weights = None


    elif method == "LDAM_DRW":

        criterion = LDAMLoss(

            class_counts=
                counts,

            max_margin=
                float(
                    candidate
                ),

            scale=
                30.0
        )


        drw_weights = (
            effective_number_weights(

                counts,

                beta=0.9999,

                device=device
            )
        )


        hyperparameters = {

            "max_margin_C":
                float(
                    candidate
                ),

            "scale":
                30.0,

            "drw_start_epoch":
                80,

            "effective_number_beta":
                0.9999,
        }


    criterion = criterion.to(
        device
    )


    return {

        "dataset":
            dataset_name,

        "method":
            method,

        "candidate":
            candidate,

        "criterion":
            criterion,

        "drw_weights":
            drw_weights,

        "hyperparameters":
            hyperparameters,
    }


def compute_method_loss(
    loss_bundle,
    logits,
    targets,
    epoch
):

    method = loss_bundle[
        "method"
    ]


    criterion = loss_bundle[
        "criterion"
    ]


    if method == "LDAM_DRW":

        if int(
            epoch
        ) < 80:

            class_weights = None

        else:

            class_weights = loss_bundle[
                "drw_weights"
            ]


        return criterion(

            logits,
            targets,

            class_weights=
                class_weights
        )


    return criterion(
        logits,
        targets
    )


# ------------------------------------------------------------
# Loss-factory configuration audit
# ------------------------------------------------------------

FACTORY_TEST_CASES = {

    "Focal":
        1.0,

    "LDAM_DRW":
        0.25,

    "BalancedSoftmax":
        "training_counts",

    "LA_Loss":
        0.5,
}


LOSS_FACTORY_PASS = True


print("=" * 105)
print("NOTEBOOK 04 — LOSS FACTORY AUDIT")
print("=" * 105)


for method, candidate in (
    FACTORY_TEST_CASES.items()
):

    valid = validate_baseline_candidate(
        method,
        candidate
    )


    LOSS_FACTORY_PASS &= bool(
        valid
    )


    print(
        f"{method:17s} | "
        f"Candidate={str(candidate):15s} | "
        f"{'PASS' if valid else 'FAIL'}"
    )


print("-" * 105)

print(
    "LOSS FACTORY GATE:",
    "PASS"
    if LOSS_FACTORY_PASS
    else "FAIL"
)

print("=" * 105)

NOTEBOOK 04 — LOSS FACTORY AUDIT
Focal             | Candidate=1.0             | PASS
LDAM_DRW          | Candidate=0.25            | PASS
BalancedSoftmax   | Candidate=training_counts | PASS
LA_Loss           | Candidate=0.5             | PASS
---------------------------------------------------------------------------------------------------------
LOSS FACTORY GATE: PASS


In [ ]:
# ============================================================
# CELL 29 — TRAIN / VALIDATION EPOCH ENGINE
# ============================================================

def train_one_epoch(
    model,
    train_loader,
    optimizer,
    loss_bundle,
    epoch,
    global_step,
    device,
    max_batches=None
):

    model.train()


    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    last_lr = None


    for batch_idx, (
        x,
        y,
        _
    ) in enumerate(
        train_loader
    ):


        if (
            max_batches is not None
            and
            batch_idx >= max_batches
        ):

            break


        x = x.to(
            device,
            non_blocking=True
        )

        y = y.to(
            device,
            non_blocking=True
        )


        last_lr = (
            set_optimizer_legacy_lr(
                optimizer,
                global_step
            )
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = model(
            x
        )


        loss = compute_method_loss(

            loss_bundle,
            logits,
            y,
            epoch
        )


        if not bool(
            torch.isfinite(
                loss
            ).item()
        ):

            raise RuntimeError(
                "Non-finite training loss."
            )


        loss.backward()


        optimizer.step()


        batch_size = int(
            y.shape[0]
        )


        total_loss += (
            float(
                loss.item()
            )
            *
            batch_size
        )


        predictions = torch.argmax(
            logits,
            dim=1
        )


        total_correct += int(
            (
                predictions
                ==
                y
            )
            .sum()
            .item()
        )


        total_samples += (
            batch_size
        )


        global_step += 1


    if total_samples == 0:

        raise RuntimeError(
            "Training epoch processed zero samples."
        )


    return {

        "loss":
            float(
                total_loss
                /
                total_samples
            ),

        "accuracy":
            float(
                total_correct
                /
                total_samples
            ),

        "samples":
            int(
                total_samples
            ),

        "global_step":
            int(
                global_step
            ),

        "last_lr":
            float(
                last_lr
            ),
    }


@torch.inference_mode()
def evaluate_validation(
    model,
    val_loader,
    loss_bundle,
    epoch,
    num_classes,
    device,
    max_batches=None,
    require_all_classes=True
):

    model.eval()


    total_loss = 0.0
    total_samples = 0

    all_labels = []
    all_predictions = []


    for batch_idx, (
        x,
        y,
        _
    ) in enumerate(
        val_loader
    ):


        if (
            max_batches is not None
            and
            batch_idx >= max_batches
        ):

            break


        x = x.to(
            device,
            non_blocking=True
        )

        y = y.to(
            device,
            non_blocking=True
        )


        logits = model(
            x
        )


        loss = compute_method_loss(

            loss_bundle,
            logits,
            y,
            epoch
        )


        if not bool(
            torch.isfinite(
                loss
            ).item()
        ):

            raise RuntimeError(
                "Non-finite validation loss."
            )


        predictions = torch.argmax(
            logits,
            dim=1
        )


        batch_size = int(
            y.shape[0]
        )


        total_loss += (
            float(
                loss.item()
            )
            *
            batch_size
        )


        total_samples += (
            batch_size
        )


        all_labels.append(
            y.detach()
            .cpu()
            .numpy()
        )


        all_predictions.append(
            predictions.detach()
            .cpu()
            .numpy()
        )


    if total_samples == 0:

        raise RuntimeError(
            "Validation processed zero samples."
        )


    y_true = np.concatenate(
        all_labels
    )


    y_pred = np.concatenate(
        all_predictions
    )


    metrics = (
        compute_classification_metrics(

            y_true,
            y_pred,

            num_classes=
                num_classes,

            require_all_classes=
                require_all_classes
        )
    )


    return {

        "loss":
            float(
                total_loss
                /
                total_samples
            ),

        "oa":
            metrics[
                "oa"
            ],

        "aa":
            metrics[
                "aa"
            ],

        "kappa":
            metrics[
                "kappa"
            ],

        "macro_f1":
            metrics[
                "macro_f1"
            ],

        "class_recall":
            metrics[
                "class_recall"
            ],

        "support":
            metrics[
                "support"
            ],

        "samples":
            int(
                total_samples
            ),
    }


print(
    "Train/validation epoch engine: READY"
)

Train/validation epoch engine: READY


In [ ]:
# ============================================================
# CELL 30 — FULL LONG-TAIL TRAINING RUN ENGINE
# ============================================================

LONGTAIL_CHECKPOINT_DIR = (
    NOTEBOOK04_RUNS_DIR /
    "checkpoints"
)

LONGTAIL_HISTORY_DIR = (
    NOTEBOOK04_RUNS_DIR /
    "histories"
)

LONGTAIL_METADATA_DIR = (
    NOTEBOOK04_RUNS_DIR /
    "metadata"
)


for path in [
    LONGTAIL_CHECKPOINT_DIR,
    LONGTAIL_HISTORY_DIR,
    LONGTAIL_METADATA_DIR,
]:

    path.mkdir(
        parents=True,
        exist_ok=True
    )


def candidate_tag(
    method,
    candidate
):

    if method == "BalancedSoftmax":

        return "training_counts"


    value = float(
        candidate
    )


    return (
        f"{value:.4f}"
        .replace(
            ".",
            "p"
        )
    )


def make_baseline_run_id(
    dataset_name,
    method,
    candidate,
    seed
):

    return (

        f"{dataset_name.lower()}__"
        f"{method.lower()}__"
        f"{candidate_tag(method, candidate)}__"
        f"seed{int(seed)}"
    )


def train_longtail_run(
    dataset_name,
    method,
    candidate,
    seed,
    verbose=True
):

    if not NOTEBOOK04_PIPELINE_GATE_PASS:

        raise RuntimeError(
            "Notebook 04 pipeline gate has not passed."
        )


    if not validate_baseline_candidate(
        method,
        candidate
    ):

        raise ValueError(
            "Candidate outside locked protocol."
        )


    seed = int(
        seed
    )


    if (
        seed
        not in
        TRAINING_SEEDS
    ):

        raise ValueError(
            "Seed outside frozen training seeds."
        )


    run_id = make_baseline_run_id(

        dataset_name,
        method,
        candidate,
        seed
    )


    checkpoint_path = (

        LONGTAIL_CHECKPOINT_DIR /
        f"{run_id}.pt"
    )


    history_csv_path = (

        LONGTAIL_HISTORY_DIR /
        f"{run_id}_history.csv"
    )


    metadata_path = (

        LONGTAIL_METADATA_DIR /
        f"{run_id}_metadata.json"
    )


    # --------------------------------------------------------
    # Never silently overwrite a completed experiment.
    # --------------------------------------------------------

    existing_artifacts = [

        checkpoint_path,
        history_csv_path,
        metadata_path,
    ]


    if any(
        p.exists()
        for p in existing_artifacts
    ):

        raise RuntimeError(

            f"Run artifacts already exist for:\n"
            f"{run_id}\n"
            f"Refusing silent overwrite."
        )


    set_global_seed(
        seed
    )


    cfg = DATASETS[
        dataset_name
    ]


    model = HybridSN(

        spectral_channels=
            cfg[
                "pca_components"
            ],

        patch_size=
            cfg[
                "patch_size"
            ],

        num_classes=
            cfg[
                "num_classes"
            ],

        dropout=
            0.4

    ).to(
        DEVICE
    )


    train_loader, val_loader = (
        make_train_val_loaders(
            dataset_name,
            seed
        )
    )


    loss_bundle = build_loss_bundle(

        dataset_name,
        method,
        candidate,
        DEVICE
    )


    optimizer = (
        make_notebook04_optimizer(
            model
        )
    )


    global_step = 0

    best_record = None
    best_state_dict = None

    history = []


    run_start = time.time()


    if verbose:

        print("=" * 120)

        print(
            f"TRAINING RUN | "
            f"{dataset_name} | "
            f"{method} | "
            f"candidate={candidate} | "
            f"seed={seed}"
        )

        print("=" * 120)


    for epoch in range(
        1,
        NUM_EPOCHS + 1
    ):


        epoch_start = time.time()


        train_result = train_one_epoch(

            model=
                model,

            train_loader=
                train_loader,

            optimizer=
                optimizer,

            loss_bundle=
                loss_bundle,

            epoch=
                epoch,

            global_step=
                global_step,

            device=
                DEVICE
        )


        global_step = (
            train_result[
                "global_step"
            ]
        )


        val_result = (
            evaluate_validation(

                model=
                    model,

                val_loader=
                    val_loader,

                loss_bundle=
                    loss_bundle,

                epoch=
                    epoch,

                num_classes=
                    cfg[
                        "num_classes"
                    ],

                device=
                    DEVICE,

                require_all_classes=
                    True
            )
        )


        candidate_record = {

            "epoch":
                int(
                    epoch
                ),

            "aa":
                float(
                    val_result[
                        "aa"
                    ]
                ),

            "macro_f1":
                float(
                    val_result[
                        "macro_f1"
                    ]
                ),

            "oa":
                float(
                    val_result[
                        "oa"
                    ]
                ),

            "kappa":
                float(
                    val_result[
                        "kappa"
                    ]
                ),

            "loss":
                float(
                    val_result[
                        "loss"
                    ]
                ),
        }


        is_best = (
            is_better_longtail_checkpoint(

                candidate_record,

                best_record
            )
        )


        if is_best:

            best_record = (
                candidate_record.copy()
            )


            best_state_dict = {

                key:
                    value.detach()
                    .cpu()
                    .clone()

                for key, value
                in model.state_dict().items()
            }


        epoch_seconds = (
            time.time()
            -
            epoch_start
        )


        history.append({

            "epoch":
                int(
                    epoch
                ),

            "train_loss":
                float(
                    train_result[
                        "loss"
                    ]
                ),

            "train_accuracy":
                float(
                    train_result[
                        "accuracy"
                    ]
                ),

            "validation_loss":
                float(
                    val_result[
                        "loss"
                    ]
                ),

            "validation_oa":
                float(
                    val_result[
                        "oa"
                    ]
                ),

            "validation_aa":
                float(
                    val_result[
                        "aa"
                    ]
                ),

            "validation_kappa":
                float(
                    val_result[
                        "kappa"
                    ]
                ),

            "validation_macro_f1":
                float(
                    val_result[
                        "macro_f1"
                    ]
                ),

            "global_step":
                int(
                    global_step
                ),

            "last_lr":
                float(
                    train_result[
                        "last_lr"
                    ]
                ),

            "epoch_seconds":
                float(
                    epoch_seconds
                ),

            "is_best":
                bool(
                    is_best
                ),
        })


        if verbose and (
            epoch == 1
            or
            epoch % 10 == 0
            or
            is_best
            or
            epoch == NUM_EPOCHS
        ):

            marker = (
                " BEST"
                if is_best
                else ""
            )


            print(

                f"Epoch {epoch:3d} | "

                f"TrainLoss "
                f"{train_result['loss']:.4f} | "

                f"ValLoss "
                f"{val_result['loss']:.4f} | "

                f"ValOA "
                f"{val_result['oa']:.6f} | "

                f"ValAA "
                f"{val_result['aa']:.6f} | "

                f"F1 "
                f"{val_result['macro_f1']:.6f} | "

                f"{epoch_seconds:.1f}s"
                f"{marker}"
            )


    if (
        best_record is None
        or
        best_state_dict is None
    ):

        raise RuntimeError(
            "Training produced no valid checkpoint."
        )


    total_seconds = (
        time.time()
        -
        run_start
    )


    # --------------------------------------------------------
    # Freeze best checkpoint
    # --------------------------------------------------------

    checkpoint_payload = {

        "project":
            "CSE498R LTLC",

        "notebook":
            "04_LongTail_Training_Baselines",

        "run_id":
            run_id,

        "dataset":
            dataset_name,

        "method":
            method,

        "candidate":
            candidate,

        "seed":
            seed,

        "best_epoch":
            int(
                best_record[
                    "epoch"
                ]
            ),

        "best_validation":
            best_record,

        "state_dict":
            best_state_dict,

        "training_hyperparameters":
            loss_bundle[
                "hyperparameters"
            ],

        "protocol_sha256":
            protocol_sha256,

        "split_sha256":
            DATASETS[
                dataset_name
            ][
                "split_sha256"
            ],

        "pca_cube_sha256":
            DATASETS[
                dataset_name
            ][
                "normalized_cube_sha256"
            ],

        "official_test_used":
            False,
    }


    torch.save(
        checkpoint_payload,
        checkpoint_path
    )


    checkpoint_sha256 = (
        sha256_file(
            checkpoint_path
        )
    )


    # --------------------------------------------------------
    # Freeze history
    # --------------------------------------------------------

    history_df = pd.DataFrame(
        history
    )


    history_df.to_csv(
        history_csv_path,
        index=False
    )


    # --------------------------------------------------------
    # Freeze metadata
    # --------------------------------------------------------

    metadata = {

        "run_id":
            run_id,

        "dataset":
            dataset_name,

        "method":
            method,

        "candidate":
            candidate,

        "seed":
            seed,

        "epochs_completed":
            NUM_EPOCHS,

        "best_epoch":
            int(
                best_record[
                    "epoch"
                ]
            ),

        "best_validation":
            best_record,

        "total_training_seconds":
            float(
                total_seconds
            ),

        "checkpoint_path":
            str(
                checkpoint_path
            ),

        "checkpoint_sha256":
            checkpoint_sha256,

        "history_path":
            str(
                history_csv_path
            ),

        "history_sha256":
            sha256_file(
                history_csv_path
            ),

        "protocol_sha256":
            protocol_sha256,

        "official_test_used":
            False,
    }


    with open(
        metadata_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            metadata,
            f,
            indent=2
        )


    metadata[
        "metadata_sha256"
    ] = sha256_file(
        metadata_path
    )


    if verbose:

        print("-" * 120)

        print(
            f"Best epoch : "
            f"{best_record['epoch']}"
        )

        print(
            f"Best Val AA: "
            f"{best_record['aa']:.6f}"
        )

        print(
            f"Best Val OA: "
            f"{best_record['oa']:.6f}"
        )

        print(
            f"Best Val F1: "
            f"{best_record['macro_f1']:.6f}"
        )

        print(
            f"Time       : "
            f"{total_seconds / 60:.2f} min"
        )

        print(
            f"SHA-256    : "
            f"{checkpoint_sha256}"
        )

        print("=" * 120)


    del model
    del optimizer
    del loss_bundle

    gc.collect()

    torch.cuda.empty_cache()


    return metadata


print(
    "Full 100-epoch long-tail training engine: READY"
)

print(
    "No full training run has been launched."
)

Full 100-epoch long-tail training engine: READY
No full training run has been launched.


In [ ]:
# ============================================================
# CELL 31 — 12-CASE GPU TRAINING SMOKE TEST
# ============================================================

SMOKE_CANDIDATES = {

    "Focal":
        1.0,

    "LDAM_DRW":
        0.25,

    "BalancedSoftmax":
        "training_counts",

    "LA_Loss":
        0.5,
}


SMOKE_RESULTS = []


print("=" * 125)
print("NOTEBOOK 04 — GPU BACKPROPAGATION SMOKE TEST")
print("=" * 125)


for dataset_name in DATASETS:

    cfg = DATASETS[
        dataset_name
    ]


    for method, candidate in (
        SMOKE_CANDIDATES.items()
    ):


        set_global_seed(
            42
        )


        model = HybridSN(

            spectral_channels=
                cfg[
                    "pca_components"
                ],

            patch_size=
                cfg[
                    "patch_size"
                ],

            num_classes=
                cfg[
                    "num_classes"
                ],

            dropout=
                0.4

        ).to(
            DEVICE
        )


        optimizer = (
            make_notebook04_optimizer(
                model
            )
        )


        loss_bundle = build_loss_bundle(

            dataset_name,
            method,
            candidate,
            DEVICE
        )


        train_loader, val_loader = (
            make_train_val_loaders(
                dataset_name,
                seed=42
            )
        )


        x, y, _ = next(
            iter(
                train_loader
            )
        )


        x = x.to(
            DEVICE,
            non_blocking=True
        )

        y = y.to(
            DEVICE,
            non_blocking=True
        )


        smoke_epoch = (
            80
            if method
            ==
            "LDAM_DRW"
            else
            1
        )


        model.train()


        before_weight = (
            model.classifier.weight
            .detach()
            .clone()
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        smoke_lr = (
            set_optimizer_legacy_lr(
                optimizer,
                global_step=0
            )
        )


        train_logits = model(
            x
        )


        train_loss = (
            compute_method_loss(

                loss_bundle,
                train_logits,
                y,
                smoke_epoch
            )
        )


        train_loss_finite = bool(
            torch.isfinite(
                train_loss
            ).item()
        )


        train_loss.backward()


        gradient_tensors = [

            p.grad

            for p in model.parameters()

            if p.grad is not None
        ]


        gradients_exist = (
            len(
                gradient_tensors
            )
            >
            0
        )


        gradients_finite = (
            gradients_exist
            and
            all(
                bool(
                    torch.isfinite(
                        grad
                    ).all()
                    .item()
                )

                for grad in
                gradient_tensors
            )
        )


        optimizer.step()


        after_weight = (
            model.classifier.weight
            .detach()
        )


        parameter_changed = (
            not torch.equal(
                before_weight,
                after_weight
            )
        )


        # ----------------------------------------------------
        # One validation batch
        # ----------------------------------------------------

        val_x, val_y, _ = next(
            iter(
                val_loader
            )
        )


        val_x = val_x.to(
            DEVICE,
            non_blocking=True
        )

        val_y = val_y.to(
            DEVICE,
            non_blocking=True
        )


        model.eval()


        with torch.inference_mode():

            val_logits = model(
                val_x
            )


            val_loss = (
                compute_method_loss(

                    loss_bundle,
                    val_logits,
                    val_y,
                    smoke_epoch
                )
            )


        val_loss_finite = bool(
            torch.isfinite(
                val_loss
            ).item()
        )


        logits_finite = bool(
            torch.isfinite(
                val_logits
            ).all()
            .item()
        )


        output_shape_pass = (

            tuple(
                val_logits.shape
            )

            ==

            (
                len(
                    val_y
                ),

                cfg[
                    "num_classes"
                ]
            )
        )


        overall_pass = all([

            train_loss_finite,
            gradients_exist,
            gradients_finite,
            parameter_changed,
            val_loss_finite,
            logits_finite,
            output_shape_pass,
        ])


        SMOKE_RESULTS.append({

            "dataset":
                dataset_name,

            "method":
                method,

            "candidate":
                candidate,

            "epoch_path":
                smoke_epoch,

            "train_loss":
                float(
                    train_loss.item()
                ),

            "val_loss":
                float(
                    val_loss.item()
                ),

            "lr":
                float(
                    smoke_lr
                ),

            "pass":
                bool(
                    overall_pass
                ),
        })


        print(

            f"{dataset_name:12s} | "
            f"{method:17s} | "
            f"EpochPath={smoke_epoch:3d} | "
            f"TrainLoss={train_loss.item():9.5f} | "
            f"ValLoss={val_loss.item():9.5f} | "
            f"Grad={'PASS' if gradients_finite else 'FAIL'} | "
            f"Update={'PASS' if parameter_changed else 'FAIL'} | "
            f"{'PASS' if overall_pass else 'FAIL'}"
        )


        del model
        del optimizer
        del loss_bundle
        del x
        del y
        del val_x
        del val_y
        del train_logits
        del val_logits

        gc.collect()

        torch.cuda.empty_cache()


ALL_GPU_SMOKE_PASS = all(

    row[
        "pass"
    ]

    for row
    in SMOKE_RESULTS
)


print("-" * 125)

print(
    "Cases tested:",
    len(
        SMOKE_RESULTS
    )
)

print(
    "Cases passed:",
    sum(
        int(
            row[
                "pass"
            ]
        )
        for row
        in SMOKE_RESULTS
    )
)

print()

print(
    "12-CASE GPU TRAINING SMOKE GATE:",
    "PASS"
    if ALL_GPU_SMOKE_PASS
    else "FAIL"
)

print("=" * 125)

NOTEBOOK 04 — GPU BACKPROPAGATION SMOKE TEST
Pingan       | Focal             | EpochPath=  1 | TrainLoss=  2.07405 | ValLoss=  2.07866 | Grad=PASS | Update=PASS | PASS
Pingan       | LDAM_DRW          | EpochPath= 80 | TrainLoss=  8.66464 | ValLoss=  8.46750 | Grad=PASS | Update=PASS | PASS
Pingan       | BalancedSoftmax   | EpochPath=  1 | TrainLoss=  1.64382 | ValLoss=  2.08314 | Grad=PASS | Update=PASS | PASS
Pingan       | LA_Loss           | EpochPath=  1 | TrainLoss=  1.73095 | ValLoss=  1.95936 | Grad=PASS | Update=PASS | PASS
Qingyun      | Focal             | EpochPath=  1 | TrainLoss=  1.49157 | ValLoss=  1.44161 | Grad=PASS | Update=PASS | PASS
Qingyun      | LDAM_DRW          | EpochPath= 80 | TrainLoss=  6.56264 | ValLoss=  6.94154 | Grad=PASS | Update=PASS | PASS
Qingyun      | BalancedSoftmax   | EpochPath=  1 | TrainLoss=  1.48110 | ValLoss=  1.68891 | Grad=PASS | Update=PASS | PASS
Qingyun      | LA_Loss           | EpochPath=  1 | TrainLoss=  1.52249 | ValLoss=  1.60

In [ ]:
# ============================================================
# CELL 32 — TRAINING ENGINE FREEZE + FINAL PRE-RUN GATE
# ============================================================

NOTEBOOK04_ENGINE_SPEC = {

    "project":
        "CSE498R LTLC",

    "notebook":
        "04_LongTail_Training_Baselines",

    "parent_protocol_sha256":
        protocol_sha256,

    "training_engine_version":
        "notebook04_engine_v1",

    "optimizer": {

        "name":
            "Adam",

        "base_learning_rate":
            BASE_LEARNING_RATE,

        "betas":
            list(
                ADAM_BETAS
            ),

        "epsilon":
            ADAM_EPSILON,

        "weight_decay":
            0.0,

        "legacy_decay":
            LEGACY_LR_DECAY,

        "lr_formula":
            (
                "base_lr / "
                "(1 + decay * global_step)"
            ),
    },

    "metrics": {

        "OA":
            "overall accuracy",

        "AA":
            "mean recall across all evaluation classes",

        "Kappa":
            "Cohen kappa",

        "Macro_F1":
            "macro averaged F1",

        "checkpoint_primary":
            "validation AA",
    },

    "prediction_rule":
        "argmax of raw model logits",

    "validation_loss":
        (
            "active method-specific loss for the "
            "corresponding epoch; used only as final "
            "checkpoint tie-break"
        ),

    "ldam_drw": {

        "epochs_before_80":
            "LDAM without class weights",

        "epochs_80_to_100":
            "LDAM with effective-number class weights",
    },

    "tangdaowan_public_mask_protocol": {

        "full_gt_classes":
            18,

        "evaluation_classes":
            16,

        "omitted_full_gt_classes":
            [
                17,
                18
            ],

        "reason":
            (
                "Classes 17 and 18 are absent from "
                "the public train/test masks."
            ),
    },

    "official_test": {

        "dataset_created":
            False,

        "dataloader_created":
            False,

        "used_for_training":
            False,

        "used_for_tuning":
            False,

        "used_for_checkpoint_selection":
            False,
    },
}


ENGINE_SPEC_PATH = (

    NOTEBOOK04_AUDIT_DIR /

    "training_engine_v1_locked.json"
)


if ENGINE_SPEC_PATH.exists():

    with open(
        ENGINE_SPEC_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        existing_engine_spec = (
            json.load(f)
        )


    if (
        existing_engine_spec
        !=
        NOTEBOOK04_ENGINE_SPEC
    ):

        raise RuntimeError(
            "Existing training-engine audit does not "
            "match the current engine."
        )

else:

    with open(
        ENGINE_SPEC_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            NOTEBOOK04_ENGINE_SPEC,
            f,
            indent=2,
            sort_keys=True
        )


ENGINE_SPEC_SHA256 = sha256_file(
    ENGINE_SPEC_PATH
)


NOTEBOOK04_REAL_TRAINING_GATE = {

    "protocol_frozen":
        bool(
            NOTEBOOK04_PRETRAIN_GATE_PASS
        ),

    "pipeline_frozen":
        bool(
            NOTEBOOK04_PIPELINE_GATE_PASS
        ),

    "metric_engine":
        bool(
            METRIC_ENGINE_SANITY_PASS
        ),

    "optimizer_compatibility":
        bool(
            OPTIMIZER_SANITY_PASS
        ),

    "loss_factory":
        bool(
            LOSS_FACTORY_PASS
        ),

    "gpu_backpropagation":
        bool(
            ALL_GPU_SMOKE_PASS
        ),

    "official_test_dataset_absent":
        (
            not
            OFFICIAL_TEST_DATASET_CREATED
        ),

    "official_test_loader_absent":
        (
            not
            OFFICIAL_TEST_DATALOADER_CREATED
        ),

    "training_engine_spec_saved":
        ENGINE_SPEC_PATH.exists(),
}


NOTEBOOK04_REAL_TRAINING_GATE_PASS = all(
    NOTEBOOK04_REAL_TRAINING_GATE.values()
)


print("=" * 120)
print("LTLC NOTEBOOK 04 — FINAL GATE BEFORE REAL TRAINING")
print("=" * 120)


for name, passed in (
    NOTEBOOK04_REAL_TRAINING_GATE.items()
):

    print(
        f"{'PASS' if passed else 'FAIL':5s} | "
        f"{name}"
    )


print("-" * 120)

print(
    "Protocol SHA :",
    protocol_sha256
)

print(
    "Engine SHA   :",
    ENGINE_SPEC_SHA256
)

print()

print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print("=" * 120)

print(
    "REAL TRAINING AUTHORIZATION:",
    "PASS"
    if NOTEBOOK04_REAL_TRAINING_GATE_PASS
    else "FAIL"
)

print("=" * 120)


if not NOTEBOOK04_REAL_TRAINING_GATE_PASS:

    raise RuntimeError(
        "Do not launch long-tail baseline training."
    )

LTLC NOTEBOOK 04 — FINAL GATE BEFORE REAL TRAINING
PASS  | protocol_frozen
PASS  | pipeline_frozen
PASS  | metric_engine
PASS  | optimizer_compatibility
PASS  | loss_factory
PASS  | gpu_backpropagation
PASS  | official_test_dataset_absent
PASS  | official_test_loader_absent
PASS  | training_engine_spec_saved
------------------------------------------------------------------------------------------------------------------------
Protocol SHA : 053179d263b118bea1da4a7c02c128c2c160cb13398ebc4adaa2027577c0d577
Engine SHA   : 6054df419915dcbee1e2d6320098dc9d50c26c2e6225fa4b77f6737209588807

Official-test dataset created    : False
Official-test DataLoader created : False
REAL TRAINING AUTHORIZATION: PASS


In [ ]:
# ============================================================
# CELL 33 — RECOVER FROZEN RUN 01
# TANGDAOWAN FOCAL γ=1.0 | SEED 42
# ============================================================

RUN01_ID = make_baseline_run_id(
    "Tangdaowan",
    "Focal",
    1.0,
    42
)

RUN01_METADATA_PATH = (
    LONGTAIL_METADATA_DIR /
    f"{RUN01_ID}_metadata.json"
)

RUN01_HISTORY_PATH = (
    LONGTAIL_HISTORY_DIR /
    f"{RUN01_ID}_history.csv"
)

RUN01_CHECKPOINT_PATH = (
    LONGTAIL_CHECKPOINT_DIR /
    f"{RUN01_ID}.pt"
)


if not all([
    RUN01_METADATA_PATH.exists(),
    RUN01_HISTORY_PATH.exists(),
    RUN01_CHECKPOINT_PATH.exists(),
]):
    raise FileNotFoundError(
        "Frozen Run-01 artifacts are incomplete."
    )


with open(
    RUN01_METADATA_PATH,
    "r",
    encoding="utf-8"
) as f:
    TANGDAOWAN_FOCAL_G1_META = json.load(f)


run01_history = pd.read_csv(
    RUN01_HISTORY_PATH
)


print("=" * 120)
print("RECOVERED FROZEN RUN 01 — NO RETRAINING")
print("=" * 120)

print(
    "Dataset          :",
    TANGDAOWAN_FOCAL_G1_META["dataset"]
)

print(
    "Method           :",
    TANGDAOWAN_FOCAL_G1_META["method"]
)

print(
    "Candidate gamma  :",
    TANGDAOWAN_FOCAL_G1_META["candidate"]
)

print(
    "Seed             :",
    TANGDAOWAN_FOCAL_G1_META["seed"]
)

print(
    "Epochs completed :",
    len(run01_history)
)

print(
    "Best epoch       :",
    TANGDAOWAN_FOCAL_G1_META["best_epoch"]
)

print(
    "Best Val AA      :",
    f"{TANGDAOWAN_FOCAL_G1_META['best_validation']['aa']:.6f}"
)

print(
    "Best Val Macro-F1:",
    f"{TANGDAOWAN_FOCAL_G1_META['best_validation']['macro_f1']:.6f}"
)

print(
    "Best Val OA      :",
    f"{TANGDAOWAN_FOCAL_G1_META['best_validation']['oa']:.6f}"
)

print(
    "Checkpoint SHA   :",
    TANGDAOWAN_FOCAL_G1_META["checkpoint_sha256"]
)

print(
    "Official test used:",
    TANGDAOWAN_FOCAL_G1_META["official_test_used"]
)

print()
print("Training action   : NONE — existing frozen artifacts loaded")
print("=" * 120)

RECOVERED FROZEN RUN 01 — NO RETRAINING
Dataset          : Tangdaowan
Method           : Focal
Candidate gamma  : 1.0
Seed             : 42
Epochs completed : 100
Best epoch       : 83
Best Val AA      : 0.999821
Best Val Macro-F1: 0.999734
Best Val OA      : 0.999623
Checkpoint SHA   : 0399967267fa4f27595341ef83f722462c06c0ff4418046c2ad1fb97b3e50ed5
Official test used: False

Training action   : NONE — existing frozen artifacts loaded


In [ ]:
# ============================================================
# CELL 34 — INDEPENDENT AUDIT:
# TANGDAOWAN FOCAL γ=1.0 | SEED 42
# ============================================================

RUN01_ID = make_baseline_run_id(
    "Tangdaowan",
    "Focal",
    1.0,
    42
)


RUN01_CHECKPOINT_PATH = (

    LONGTAIL_CHECKPOINT_DIR /
    f"{RUN01_ID}.pt"
)


RUN01_HISTORY_PATH = (

    LONGTAIL_HISTORY_DIR /
    f"{RUN01_ID}_history.csv"
)


RUN01_METADATA_PATH = (

    LONGTAIL_METADATA_DIR /
    f"{RUN01_ID}_metadata.json"
)


# ------------------------------------------------------------
# 1. File existence
# ------------------------------------------------------------

files_exist_pass = all(
    path.exists()
    for path in [
        RUN01_CHECKPOINT_PATH,
        RUN01_HISTORY_PATH,
        RUN01_METADATA_PATH,
    ]
)


if not files_exist_pass:

    raise FileNotFoundError(
        "One or more Run-01 artifacts are missing."
    )


# ------------------------------------------------------------
# 2. Reload independently from disk
# ------------------------------------------------------------

history_df = pd.read_csv(
    RUN01_HISTORY_PATH
)


with open(
    RUN01_METADATA_PATH,
    "r",
    encoding="utf-8"
) as f:

    metadata = json.load(f)


checkpoint = torch.load(

    RUN01_CHECKPOINT_PATH,

    map_location="cpu",

    weights_only=False
)


# ------------------------------------------------------------
# 3. Basic identity audit
# ------------------------------------------------------------

identity_pass = all([

    metadata[
        "dataset"
    ]
    ==
    "Tangdaowan",

    metadata[
        "method"
    ]
    ==
    "Focal",

    np.isclose(
        float(
            metadata[
                "candidate"
            ]
        ),
        1.0
    ),

    int(
        metadata[
            "seed"
        ]
    )
    ==
    42,

    checkpoint[
        "dataset"
    ]
    ==
    "Tangdaowan",

    checkpoint[
        "method"
    ]
    ==
    "Focal",

    np.isclose(
        float(
            checkpoint[
                "candidate"
            ]
        ),
        1.0
    ),

    int(
        checkpoint[
            "seed"
        ]
    )
    ==
    42,
])


# ------------------------------------------------------------
# 4. Exactly 100 epochs
# ------------------------------------------------------------

epoch_count_pass = (
    len(
        history_df
    )
    ==
    100
)


epoch_sequence_pass = np.array_equal(

    history_df[
        "epoch"
    ].to_numpy(
        dtype=np.int64
    ),

    np.arange(
        1,
        101,
        dtype=np.int64
    )
)


# ------------------------------------------------------------
# 5. Finite complete history
# ------------------------------------------------------------

numeric_columns = [

    "train_loss",
    "train_accuracy",
    "validation_loss",
    "validation_oa",
    "validation_aa",
    "validation_kappa",
    "validation_macro_f1",
    "global_step",
    "last_lr",
    "epoch_seconds",
]


history_finite_pass = bool(

    np.isfinite(
        history_df[
            numeric_columns
        ].to_numpy(
            dtype=np.float64
        )
    ).all()
)


# ------------------------------------------------------------
# 6. Independently recompute the selected best epoch
# ------------------------------------------------------------

recomputed_best = None


for _, row in (
    history_df.iterrows()
):

    candidate_record = {

        "epoch":
            int(
                row[
                    "epoch"
                ]
            ),

        "aa":
            float(
                row[
                    "validation_aa"
                ]
            ),

        "macro_f1":
            float(
                row[
                    "validation_macro_f1"
                ]
            ),

        "oa":
            float(
                row[
                    "validation_oa"
                ]
            ),

        "kappa":
            float(
                row[
                    "validation_kappa"
                ]
            ),

        "loss":
            float(
                row[
                    "validation_loss"
                ]
            ),
    }


    if is_better_longtail_checkpoint(

        candidate_record,

        recomputed_best
    ):

        recomputed_best = (
            candidate_record.copy()
        )


best_epoch_pass = all([

    int(
        recomputed_best[
            "epoch"
        ]
    )
    ==
    int(
        metadata[
            "best_epoch"
        ]
    ),

    int(
        recomputed_best[
            "epoch"
        ]
    )
    ==
    int(
        checkpoint[
            "best_epoch"
        ]
    ),
])


best_metrics_pass = all([

    np.isclose(

        recomputed_best[
            "aa"
        ],

        float(
            metadata[
                "best_validation"
            ][
                "aa"
            ]
        ),

        atol=1e-12
    ),

    np.isclose(

        recomputed_best[
            "macro_f1"
        ],

        float(
            metadata[
                "best_validation"
            ][
                "macro_f1"
            ]
        ),

        atol=1e-12
    ),

    np.isclose(

        recomputed_best[
            "oa"
        ],

        float(
            metadata[
                "best_validation"
            ][
                "oa"
            ]
        ),

        atol=1e-12
    ),
])


# ------------------------------------------------------------
# 7. Frozen parent-artifact hashes
# ------------------------------------------------------------

protocol_hash_pass = (

    checkpoint[
        "protocol_sha256"
    ]

    ==
    protocol_sha256
)


split_hash_pass = (

    checkpoint[
        "split_sha256"
    ]

    ==
    DATASETS[
        "Tangdaowan"
    ][
        "split_sha256"
    ]
)


pca_hash_pass = (

    checkpoint[
        "pca_cube_sha256"
    ]

    ==
    DATASETS[
        "Tangdaowan"
    ][
        "normalized_cube_sha256"
    ]
)


# ------------------------------------------------------------
# 8. Saved artifact hashes
# ------------------------------------------------------------

checkpoint_hash_pass = (

    sha256_file(
        RUN01_CHECKPOINT_PATH
    )

    ==

    metadata[
        "checkpoint_sha256"
    ]
)


history_hash_pass = (

    sha256_file(
        RUN01_HISTORY_PATH
    )

    ==

    metadata[
        "history_sha256"
    ]
)


# ------------------------------------------------------------
# 9. Test leakage gate
# ------------------------------------------------------------

test_sealed_pass = all([

    metadata[
        "official_test_used"
    ]
    is False,

    checkpoint[
        "official_test_used"
    ]
    is False,

    OFFICIAL_TEST_DATASET_CREATED
    is False,

    OFFICIAL_TEST_DATALOADER_CREATED
    is False,
])


# ------------------------------------------------------------
# 10. Model state completeness
# ------------------------------------------------------------

reference_model = HybridSN(

    spectral_channels=
        DATASETS[
            "Tangdaowan"
        ][
            "pca_components"
        ],

    patch_size=
        DATASETS[
            "Tangdaowan"
        ][
            "patch_size"
        ],

    num_classes=
        DATASETS[
            "Tangdaowan"
        ][
            "num_classes"
        ],

    dropout=
        0.4
)


reference_keys = set(
    reference_model.state_dict().keys()
)


checkpoint_keys = set(
    checkpoint[
        "state_dict"
    ].keys()
)


state_dict_pass = (
    reference_keys
    ==
    checkpoint_keys
)


del reference_model


# ------------------------------------------------------------
# Final audit
# ------------------------------------------------------------

RUN01_AUDIT_CHECKS = {

    "artifact_files":
        files_exist_pass,

    "run_identity":
        identity_pass,

    "100_epochs":
        epoch_count_pass,

    "epoch_sequence":
        epoch_sequence_pass,

    "finite_history":
        history_finite_pass,

    "best_epoch_recomputed":
        best_epoch_pass,

    "best_metrics_recomputed":
        best_metrics_pass,

    "protocol_hash":
        protocol_hash_pass,

    "split_hash":
        split_hash_pass,

    "pca_hash":
        pca_hash_pass,

    "checkpoint_hash":
        checkpoint_hash_pass,

    "history_hash":
        history_hash_pass,

    "state_dict":
        state_dict_pass,

    "official_test_sealed":
        test_sealed_pass,
}


RUN01_AUDIT_PASS = all(
    RUN01_AUDIT_CHECKS.values()
)


print("=" * 120)
print("RUN 01 — CLEAN-ROOM ARTIFACT AUDIT")
print("=" * 120)


for name, passed in (
    RUN01_AUDIT_CHECKS.items()
):

    print(
        f"{'PASS' if passed else 'FAIL':5s} | "
        f"{name}"
    )


print("-" * 120)

print(
    "Recomputed best epoch :",
    recomputed_best[
        "epoch"
    ]
)

print(
    "Recomputed best Val AA:",
    f"{recomputed_best['aa']:.6f}"
)

print(
    "Recomputed best F1    :",
    f"{recomputed_best['macro_f1']:.6f}"
)

print(
    "Recomputed best Val OA:",
    f"{recomputed_best['oa']:.6f}"
)

print()

print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print("=" * 120)

print(
    "RUN 01 ARTIFACT GATE:",
    "PASS"
    if RUN01_AUDIT_PASS
    else "FAIL"
)

print("=" * 120)


if not RUN01_AUDIT_PASS:

    raise RuntimeError(
        "Run 01 audit failed. Stop before Run 02."
    )

RUN 01 — CLEAN-ROOM ARTIFACT AUDIT
PASS  | artifact_files
PASS  | run_identity
PASS  | 100_epochs
PASS  | epoch_sequence
PASS  | finite_history
PASS  | best_epoch_recomputed
PASS  | best_metrics_recomputed
PASS  | protocol_hash
PASS  | split_hash
PASS  | pca_hash
PASS  | checkpoint_hash
PASS  | history_hash
PASS  | state_dict
PASS  | official_test_sealed
------------------------------------------------------------------------------------------------------------------------
Recomputed best epoch : 83
Recomputed best Val AA: 0.999821
Recomputed best F1    : 0.999734
Recomputed best Val OA: 0.999623

Official-test dataset created    : False
Official-test DataLoader created : False
RUN 01 ARTIFACT GATE: PASS


In [ ]:
# ============================================================
# CELL 35 — TANGDAOWAN FOCAL γ=2.0 | SEED 42
# ============================================================

assert NOTEBOOK04_REAL_TRAINING_GATE_PASS
assert RUN01_AUDIT_PASS

assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


TANGDAOWAN_FOCAL_G2_META = train_longtail_run(

    dataset_name="Tangdaowan",

    method="Focal",

    candidate=2.0,

    seed=42,

    verbose=True
)


print()
print("=" * 120)
print("RUN 02 COMPLETION SUMMARY")
print("=" * 120)

print(
    "Dataset          :",
    TANGDAOWAN_FOCAL_G2_META["dataset"]
)

print(
    "Method           :",
    TANGDAOWAN_FOCAL_G2_META["method"]
)

print(
    "Candidate gamma  :",
    TANGDAOWAN_FOCAL_G2_META["candidate"]
)

print(
    "Seed             :",
    TANGDAOWAN_FOCAL_G2_META["seed"]
)

print(
    "Epochs completed :",
    TANGDAOWAN_FOCAL_G2_META["epochs_completed"]
)

print(
    "Best epoch       :",
    TANGDAOWAN_FOCAL_G2_META["best_epoch"]
)

print(
    "Best Val AA      :",
    f"{TANGDAOWAN_FOCAL_G2_META['best_validation']['aa']:.6f}"
)

print(
    "Best Val Macro-F1:",
    f"{TANGDAOWAN_FOCAL_G2_META['best_validation']['macro_f1']:.6f}"
)

print(
    "Best Val OA      :",
    f"{TANGDAOWAN_FOCAL_G2_META['best_validation']['oa']:.6f}"
)

print(
    "Official test used:",
    TANGDAOWAN_FOCAL_G2_META["official_test_used"]
)

print("=" * 120)

TRAINING RUN | Tangdaowan | Focal | candidate=2.0 | seed=42
Epoch   1 | TrainLoss 1.5690 | ValLoss 0.8551 | ValOA 0.573125 | ValAA 0.222891 | F1 0.170232 | 3.3s BEST
Epoch   2 | TrainLoss 0.5653 | ValLoss 0.2961 | ValOA 0.790614 | ValAA 0.475833 | F1 0.462875 | 2.4s BEST
Epoch   3 | TrainLoss 0.2936 | ValLoss 0.2290 | ValOA 0.828025 | ValAA 0.506508 | F1 0.512771 | 2.8s BEST
Epoch   4 | TrainLoss 0.2285 | ValLoss 0.1739 | ValOA 0.841123 | ValAA 0.588402 | F1 0.566770 | 2.5s BEST
Epoch   5 | TrainLoss 0.1929 | ValLoss 0.1361 | ValOA 0.861760 | ValAA 0.602016 | F1 0.605833 | 2.3s BEST
Epoch   7 | TrainLoss 0.1309 | ValLoss 0.0867 | ValOA 0.901527 | ValAA 0.699095 | F1 0.715923 | 2.3s BEST
Epoch   8 | TrainLoss 0.1151 | ValLoss 0.0798 | ValOA 0.919996 | ValAA 0.705448 | F1 0.712949 | 2.6s BEST
Epoch   9 | TrainLoss 0.0929 | ValLoss 0.0527 | ValOA 0.942895 | ValAA 0.795830 | F1 0.798940 | 2.8s BEST
Epoch  10 | TrainLoss 0.0773 | ValLoss 0.0567 | ValOA 0.928194 | ValAA 0.789013 | F1 0.77283

In [ ]:
# ============================================================
# CELL 36 — GENERIC LONG-TAIL RUN AUDITOR + RUN 02 AUDIT
# ============================================================

def baseline_candidate_equal(
    observed,
    expected
):

    if isinstance(
        expected,
        str
    ):

        return str(observed) == expected


    return np.isclose(
        float(observed),
        float(expected)
    )


def audit_longtail_run(
    dataset_name,
    method,
    candidate,
    seed
):

    run_id = make_baseline_run_id(

        dataset_name,
        method,
        candidate,
        seed
    )


    checkpoint_path = (
        LONGTAIL_CHECKPOINT_DIR /
        f"{run_id}.pt"
    )

    history_path = (
        LONGTAIL_HISTORY_DIR /
        f"{run_id}_history.csv"
    )

    metadata_path = (
        LONGTAIL_METADATA_DIR /
        f"{run_id}_metadata.json"
    )


    # --------------------------------------------------------
    # 1. Required files
    # --------------------------------------------------------

    artifact_files_pass = all(
        p.exists()
        for p in [
            checkpoint_path,
            history_path,
            metadata_path,
        ]
    )


    if not artifact_files_pass:

        raise FileNotFoundError(
            f"Incomplete artifacts for {run_id}"
        )


    # --------------------------------------------------------
    # 2. Independent disk reload
    # --------------------------------------------------------

    history_df = pd.read_csv(
        history_path
    )


    with open(
        metadata_path,
        "r",
        encoding="utf-8"
    ) as f:

        metadata = json.load(f)


    checkpoint = torch.load(

        checkpoint_path,

        map_location="cpu",

        weights_only=False
    )


    # --------------------------------------------------------
    # 3. Identity
    # --------------------------------------------------------

    identity_pass = all([

        metadata["dataset"]
        ==
        dataset_name,

        metadata["method"]
        ==
        method,

        baseline_candidate_equal(
            metadata["candidate"],
            candidate
        ),

        int(metadata["seed"])
        ==
        int(seed),

        checkpoint["dataset"]
        ==
        dataset_name,

        checkpoint["method"]
        ==
        method,

        baseline_candidate_equal(
            checkpoint["candidate"],
            candidate
        ),

        int(checkpoint["seed"])
        ==
        int(seed),
    ])


    # --------------------------------------------------------
    # 4. Epoch completeness
    # --------------------------------------------------------

    epoch_count_pass = (
        len(history_df)
        ==
        NUM_EPOCHS
    )


    epoch_sequence_pass = np.array_equal(

        history_df[
            "epoch"
        ].to_numpy(
            dtype=np.int64
        ),

        np.arange(
            1,
            NUM_EPOCHS + 1,
            dtype=np.int64
        )
    )


    # --------------------------------------------------------
    # 5. Finite training history
    # --------------------------------------------------------

    numeric_columns = [

        "train_loss",
        "train_accuracy",
        "validation_loss",
        "validation_oa",
        "validation_aa",
        "validation_kappa",
        "validation_macro_f1",
        "global_step",
        "last_lr",
        "epoch_seconds",
    ]


    finite_history_pass = bool(

        np.isfinite(

            history_df[
                numeric_columns
            ].to_numpy(
                dtype=np.float64
            )

        ).all()
    )


    # --------------------------------------------------------
    # 6. Independently reconstruct best checkpoint
    # --------------------------------------------------------

    recomputed_best = None


    for _, row in history_df.iterrows():

        candidate_record = {

            "epoch":
                int(
                    row["epoch"]
                ),

            "aa":
                float(
                    row["validation_aa"]
                ),

            "macro_f1":
                float(
                    row["validation_macro_f1"]
                ),

            "oa":
                float(
                    row["validation_oa"]
                ),

            "kappa":
                float(
                    row["validation_kappa"]
                ),

            "loss":
                float(
                    row["validation_loss"]
                ),
        }


        if is_better_longtail_checkpoint(
            candidate_record,
            recomputed_best
        ):

            recomputed_best = (
                candidate_record.copy()
            )


    best_epoch_pass = all([

        recomputed_best["epoch"]
        ==
        int(
            metadata["best_epoch"]
        ),

        recomputed_best["epoch"]
        ==
        int(
            checkpoint["best_epoch"]
        ),
    ])


    best_metrics_pass = all([

        np.isclose(
            recomputed_best["aa"],
            float(
                metadata[
                    "best_validation"
                ]["aa"]
            ),
            atol=1e-12
        ),

        np.isclose(
            recomputed_best["macro_f1"],
            float(
                metadata[
                    "best_validation"
                ]["macro_f1"]
            ),
            atol=1e-12
        ),

        np.isclose(
            recomputed_best["oa"],
            float(
                metadata[
                    "best_validation"
                ]["oa"]
            ),
            atol=1e-12
        ),

        np.isclose(
            recomputed_best["loss"],
            float(
                metadata[
                    "best_validation"
                ]["loss"]
            ),
            atol=1e-12
        ),
    ])


    # --------------------------------------------------------
    # 7. Parent hashes
    # --------------------------------------------------------

    protocol_hash_pass = (

        checkpoint[
            "protocol_sha256"
        ]
        ==
        protocol_sha256
    )


    split_hash_pass = (

        checkpoint[
            "split_sha256"
        ]
        ==
        DATASETS[
            dataset_name
        ][
            "split_sha256"
        ]
    )


    pca_hash_pass = (

        checkpoint[
            "pca_cube_sha256"
        ]
        ==
        DATASETS[
            dataset_name
        ][
            "normalized_cube_sha256"
        ]
    )


    # --------------------------------------------------------
    # 8. Saved artifact hashes
    # --------------------------------------------------------

    checkpoint_hash_pass = (

        sha256_file(
            checkpoint_path
        )
        ==
        metadata[
            "checkpoint_sha256"
        ]
    )


    history_hash_pass = (

        sha256_file(
            history_path
        )
        ==
        metadata[
            "history_sha256"
        ]
    )


    # --------------------------------------------------------
    # 9. Official test must still be untouched
    # --------------------------------------------------------

    official_test_sealed_pass = all([

        metadata[
            "official_test_used"
        ]
        is False,

        checkpoint[
            "official_test_used"
        ]
        is False,

        OFFICIAL_TEST_DATASET_CREATED
        is False,

        OFFICIAL_TEST_DATALOADER_CREATED
        is False,
    ])


    # --------------------------------------------------------
    # 10. State-dict architecture completeness
    # --------------------------------------------------------

    cfg = DATASETS[
        dataset_name
    ]


    reference_model = HybridSN(

        spectral_channels=
            cfg[
                "pca_components"
            ],

        patch_size=
            cfg[
                "patch_size"
            ],

        num_classes=
            cfg[
                "num_classes"
            ],

        dropout=0.4
    )


    state_dict_pass = (

        set(
            reference_model
            .state_dict()
            .keys()
        )

        ==

        set(
            checkpoint[
                "state_dict"
            ].keys()
        )
    )


    del reference_model


    checks = {

        "artifact_files":
            artifact_files_pass,

        "run_identity":
            identity_pass,

        "100_epochs":
            epoch_count_pass,

        "epoch_sequence":
            epoch_sequence_pass,

        "finite_history":
            finite_history_pass,

        "best_epoch_recomputed":
            best_epoch_pass,

        "best_metrics_recomputed":
            best_metrics_pass,

        "protocol_hash":
            protocol_hash_pass,

        "split_hash":
            split_hash_pass,

        "pca_hash":
            pca_hash_pass,

        "checkpoint_hash":
            checkpoint_hash_pass,

        "history_hash":
            history_hash_pass,

        "state_dict":
            state_dict_pass,

        "official_test_sealed":
            official_test_sealed_pass,
    }


    audit_pass = all(
        checks.values()
    )


    return {

        "run_id":
            run_id,

        "checks":
            checks,

        "pass":
            bool(
                audit_pass
            ),

        "metadata":
            metadata,

        "recomputed_best":
            recomputed_best,

        "checkpoint_path":
            str(
                checkpoint_path
            ),

        "checkpoint_sha256":
            sha256_file(
                checkpoint_path
            ),
    }


# ============================================================
# AUDIT RUN 02
# ============================================================

RUN02_AUDIT = audit_longtail_run(

    dataset_name="Tangdaowan",

    method="Focal",

    candidate=2.0,

    seed=42
)


RUN02_AUDIT_PASS = (
    RUN02_AUDIT[
        "pass"
    ]
)


print("=" * 120)
print("RUN 02 — CLEAN-ROOM ARTIFACT AUDIT")
print("=" * 120)


for name, passed in (
    RUN02_AUDIT[
        "checks"
    ].items()
):

    print(
        f"{'PASS' if passed else 'FAIL':5s} | "
        f"{name}"
    )


best = RUN02_AUDIT[
    "recomputed_best"
]


print("-" * 120)

print(
    "Recomputed best epoch :",
    best["epoch"]
)

print(
    "Recomputed best Val AA:",
    f"{best['aa']:.6f}"
)

print(
    "Recomputed best F1    :",
    f"{best['macro_f1']:.6f}"
)

print(
    "Recomputed best Val OA:",
    f"{best['oa']:.6f}"
)

print()

print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print("=" * 120)

print(
    "RUN 02 ARTIFACT GATE:",
    "PASS"
    if RUN02_AUDIT_PASS
    else "FAIL"
)

print("=" * 120)


if not RUN02_AUDIT_PASS:

    raise RuntimeError(
        "Run 02 audit failed."
    )

RUN 02 — CLEAN-ROOM ARTIFACT AUDIT
PASS  | artifact_files
PASS  | run_identity
PASS  | 100_epochs
PASS  | epoch_sequence
PASS  | finite_history
PASS  | best_epoch_recomputed
PASS  | best_metrics_recomputed
PASS  | protocol_hash
PASS  | split_hash
PASS  | pca_hash
PASS  | checkpoint_hash
PASS  | history_hash
PASS  | state_dict
PASS  | official_test_sealed
------------------------------------------------------------------------------------------------------------------------
Recomputed best epoch : 97
Recomputed best Val AA: 0.999599
Recomputed best F1    : 0.998672
Recomputed best Val OA: 0.998775

Official-test dataset created    : False
Official-test DataLoader created : False
RUN 02 ARTIFACT GATE: PASS


In [ ]:
# ============================================================
# CELL 37 — TANGDAOWAN FOCAL HYPERPARAMETER SELECTION
# ============================================================

assert RUN01_AUDIT_PASS
assert RUN02_AUDIT_PASS

assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


# ------------------------------------------------------------
# Reload both candidate metadata directly from disk
# ------------------------------------------------------------

def load_run_metadata(
    dataset_name,
    method,
    candidate,
    seed
):

    run_id = make_baseline_run_id(

        dataset_name,
        method,
        candidate,
        seed
    )


    path = (

        LONGTAIL_METADATA_DIR /
        f"{run_id}_metadata.json"
    )


    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


g1_meta = load_run_metadata(
    "Tangdaowan",
    "Focal",
    1.0,
    42
)


g2_meta = load_run_metadata(
    "Tangdaowan",
    "Focal",
    2.0,
    42
)


candidate_records = []


for meta in [
    g1_meta,
    g2_meta,
]:

    candidate_records.append({

        "gamma":
            float(
                meta["candidate"]
            ),

        "best_epoch":
            int(
                meta["best_epoch"]
            ),

        "validation_aa":
            float(
                meta[
                    "best_validation"
                ]["aa"]
            ),

        "validation_macro_f1":
            float(
                meta[
                    "best_validation"
                ]["macro_f1"]
            ),

        "validation_oa":
            float(
                meta[
                    "best_validation"
                ]["oa"]
            ),

        "validation_loss":
            float(
                meta[
                    "best_validation"
                ]["loss"]
            ),

        "checkpoint_sha256":
            meta[
                "checkpoint_sha256"
            ],
    })


FOCAL_TANGDAOWAN_TUNING_DF = (

    pd.DataFrame(
        candidate_records
    )

    .sort_values(
        "gamma"
    )

    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Frozen candidate-selection rule:
#
# 1. higher validation AA
# 2. higher validation Macro-F1
# 3. higher validation OA
# 4. smaller/simpler gamma
#
# Validation loss is NOT a tuning tie-break.
# ------------------------------------------------------------

def better_focal_candidate(
    candidate,
    best
):

    if best is None:

        return True


    if (
        candidate[
            "validation_aa"
        ]
        >
        best[
            "validation_aa"
        ]
        +
        CHECKPOINT_TOL
    ):

        return True


    if not math.isclose(

        candidate[
            "validation_aa"
        ],

        best[
            "validation_aa"
        ],

        abs_tol=
            CHECKPOINT_TOL
    ):

        return False


    if (
        candidate[
            "validation_macro_f1"
        ]
        >
        best[
            "validation_macro_f1"
        ]
        +
        CHECKPOINT_TOL
    ):

        return True


    if not math.isclose(

        candidate[
            "validation_macro_f1"
        ],

        best[
            "validation_macro_f1"
        ],

        abs_tol=
            CHECKPOINT_TOL
    ):

        return False


    if (
        candidate[
            "validation_oa"
        ]
        >
        best[
            "validation_oa"
        ]
        +
        CHECKPOINT_TOL
    ):

        return True


    if not math.isclose(

        candidate[
            "validation_oa"
        ],

        best[
            "validation_oa"
        ],

        abs_tol=
            CHECKPOINT_TOL
    ):

        return False


    # Final tie:
    # smaller/simpler gamma
    return (
        candidate[
            "gamma"
        ]
        <
        best[
            "gamma"
        ]
    )


selected = None


for record in candidate_records:

    if better_focal_candidate(
        record,
        selected
    ):

        selected = (
            record.copy()
        )


TANGDAOWAN_FOCAL_SELECTED_GAMMA = float(
    selected[
        "gamma"
    ]
)


# ------------------------------------------------------------
# Freeze tuning decision
# ------------------------------------------------------------

TANGDAOWAN_FOCAL_SELECTION = {

    "dataset":
        "Tangdaowan",

    "method":
        "Focal",

    "tuning_seed":
        42,

    "candidate_gammas":
        [
            1.0,
            2.0
        ],

    "selection_primary":
        "validation_AA",

    "tie_1":
        "validation_Macro_F1",

    "tie_2":
        "validation_OA",

    "final_tie":
        "smaller_gamma",

    "selected_gamma":
        TANGDAOWAN_FOCAL_SELECTED_GAMMA,

    "selected_best_epoch":
        int(
            selected[
                "best_epoch"
            ]
        ),

    "selected_validation_aa":
        float(
            selected[
                "validation_aa"
            ]
        ),

    "selected_validation_macro_f1":
        float(
            selected[
                "validation_macro_f1"
            ]
        ),

    "selected_validation_oa":
        float(
            selected[
                "validation_oa"
            ]
        ),

    "candidate_results":
        candidate_records,

    "protocol_sha256":
        protocol_sha256,

    "engine_sha256":
        ENGINE_SPEC_SHA256,

    "official_test_used":
        False,
}


TANGDAOWAN_FOCAL_SELECTION_PATH = (

    NOTEBOOK04_AUDIT_DIR /

    "tangdaowan_focal_hyperparameter_selection.json"
)


if (
    TANGDAOWAN_FOCAL_SELECTION_PATH.exists()
):

    with open(
        TANGDAOWAN_FOCAL_SELECTION_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        existing_selection = (
            json.load(f)
        )


    if (
        existing_selection
        !=
        TANGDAOWAN_FOCAL_SELECTION
    ):

        raise RuntimeError(
            "Existing Tangdaowan Focal selection "
            "does not match current validation-only result."
        )

else:

    with open(
        TANGDAOWAN_FOCAL_SELECTION_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            TANGDAOWAN_FOCAL_SELECTION,
            f,
            indent=2,
            sort_keys=True
        )


TANGDAOWAN_FOCAL_SELECTION_SHA = (
    sha256_file(
        TANGDAOWAN_FOCAL_SELECTION_PATH
    )
)


print("=" * 120)
print("TANGDAOWAN — FOCAL VALIDATION-ONLY TUNING")
print("=" * 120)

display(

    FOCAL_TANGDAOWAN_TUNING_DF[
        [
            "gamma",
            "best_epoch",
            "validation_aa",
            "validation_macro_f1",
            "validation_oa",
        ]
    ]
)

print()

print(
    "SELECTED GAMMA:",
    TANGDAOWAN_FOCAL_SELECTED_GAMMA
)

print(
    "Selected Val AA:",
    f"{selected['validation_aa']:.6f}"
)

print(
    "Selected F1    :",
    f"{selected['validation_macro_f1']:.6f}"
)

print(
    "Selected Val OA:",
    f"{selected['validation_oa']:.6f}"
)

print()

print(
    "Official test used:",
    False
)

print(
    "Selection file:",
    TANGDAOWAN_FOCAL_SELECTION_PATH
)

print(
    "Selection SHA :",
    TANGDAOWAN_FOCAL_SELECTION_SHA
)

print("=" * 120)

print(
    "TANGDAOWAN FOCAL TUNING GATE: PASS"
)

print("=" * 120)

TANGDAOWAN — FOCAL VALIDATION-ONLY TUNING


,gamma,best_epoch,validation_aa,validation_macro_f1,validation_oa
0,1.0,83,0.999821,0.999734,0.999623
1,2.0,97,0.999599,0.998672,0.998775



SELECTED GAMMA: 1.0
Selected Val AA: 0.999821
Selected F1    : 0.999734
Selected Val OA: 0.999623

Official test used: False
Selection file: /content/drive/MyDrive/CSE498/LTLC/audit/notebook04_longtail/tangdaowan_focal_hyperparameter_selection.json
Selection SHA : bbba7bd8bd081f80a37d8792d0122179f85b12ab8b9303b7ba50d455e44cbbb9
TANGDAOWAN FOCAL TUNING GATE: PASS


In [ ]:
# ============================================================
# CELL 38 — TANGDAOWAN FOCAL SELECTED γ=1.0
# TRAIN SEEDS 123 AND 3407
# ============================================================

assert TANGDAOWAN_FOCAL_SELECTED_GAMMA == 1.0

assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


def run_or_recover_longtail(
    dataset_name,
    method,
    candidate,
    seed
):

    run_id = make_baseline_run_id(
        dataset_name,
        method,
        candidate,
        seed
    )

    checkpoint_path = (
        LONGTAIL_CHECKPOINT_DIR /
        f"{run_id}.pt"
    )

    history_path = (
        LONGTAIL_HISTORY_DIR /
        f"{run_id}_history.csv"
    )

    metadata_path = (
        LONGTAIL_METADATA_DIR /
        f"{run_id}_metadata.json"
    )


    existence = [
        checkpoint_path.exists(),
        history_path.exists(),
        metadata_path.exists(),
    ]


    # --------------------------------------------------------
    # Case 1 — no artifacts: train normally
    # --------------------------------------------------------

    if not any(existence):

        print()
        print("=" * 120)
        print(
            f"NEW RUN REQUIRED: "
            f"{dataset_name} | {method} | "
            f"candidate={candidate} | seed={seed}"
        )
        print("=" * 120)

        return train_longtail_run(
            dataset_name=dataset_name,
            method=method,
            candidate=candidate,
            seed=seed,
            verbose=True
        )


    # --------------------------------------------------------
    # Case 2 — complete artifacts: verify and recover
    # --------------------------------------------------------

    if all(existence):

        audit = audit_longtail_run(
            dataset_name,
            method,
            candidate,
            seed
        )

        if not audit["pass"]:

            raise RuntimeError(
                f"Existing completed run failed audit: {run_id}"
            )


        print()
        print("=" * 120)
        print(
            f"VALID FROZEN RUN RECOVERED: {run_id}"
        )
        print("=" * 120)

        print(
            "Best epoch :",
            audit["recomputed_best"]["epoch"]
        )

        print(
            "Best Val AA:",
            f"{audit['recomputed_best']['aa']:.6f}"
        )

        print(
            "Training action: NONE"
        )

        print("=" * 120)


        return audit["metadata"]


    # --------------------------------------------------------
    # Case 3 — partial artifacts: STOP
    # --------------------------------------------------------

    raise RuntimeError(
        f"Partial experiment artifacts detected for {run_id}. "
        f"Do not retrain or delete automatically."
    )


TANGDAOWAN_FOCAL_SEED123_META = (
    run_or_recover_longtail(
        "Tangdaowan",
        "Focal",
        1.0,
        123
    )
)


TANGDAOWAN_FOCAL_SEED3407_META = (
    run_or_recover_longtail(
        "Tangdaowan",
        "Focal",
        1.0,
        3407
    )
)


print()
print("=" * 120)
print("TANGDAOWAN FOCAL — REMAINING SEEDS COMPLETE")
print("=" * 120)

for meta in [
    TANGDAOWAN_FOCAL_SEED123_META,
    TANGDAOWAN_FOCAL_SEED3407_META,
]:

    print(
        f"Seed {int(meta['seed']):4d} | "
        f"BestEpoch={int(meta['best_epoch']):3d} | "
        f"ValAA={float(meta['best_validation']['aa']):.6f} | "
        f"F1={float(meta['best_validation']['macro_f1']):.6f} | "
        f"ValOA={float(meta['best_validation']['oa']):.6f}"
    )

print("=" * 120)


NEW RUN REQUIRED: Tangdaowan | Focal | candidate=1.0 | seed=123
TRAINING RUN | Tangdaowan | Focal | candidate=1.0 | seed=123
Epoch   1 | TrainLoss 1.8367 | ValLoss 1.5442 | ValOA 0.413400 | ValAA 0.157877 | F1 0.119553 | 4.0s BEST
Epoch   2 | TrainLoss 1.2360 | ValLoss 0.7233 | ValOA 0.685262 | ValAA 0.330135 | F1 0.295457 | 3.5s BEST
Epoch   3 | TrainLoss 0.6126 | ValLoss 0.3854 | ValOA 0.805597 | ValAA 0.467590 | F1 0.464622 | 2.4s BEST
Epoch   4 | TrainLoss 0.3689 | ValLoss 0.2792 | ValOA 0.830946 | ValAA 0.494482 | F1 0.486014 | 2.3s BEST
Epoch   5 | TrainLoss 0.2832 | ValLoss 0.2257 | ValOA 0.846495 | ValAA 0.539141 | F1 0.542313 | 2.3s BEST
Epoch   6 | TrainLoss 0.2253 | ValLoss 0.1544 | ValOA 0.906238 | ValAA 0.725290 | F1 0.723939 | 2.3s BEST
Epoch   7 | TrainLoss 0.1828 | ValLoss 0.1247 | ValOA 0.925179 | ValAA 0.751709 | F1 0.741258 | 2.6s BEST
Epoch   8 | TrainLoss 0.1442 | ValLoss 0.1051 | ValOA 0.928854 | ValAA 0.784914 | F1 0.771748 | 2.8s BEST
Epoch   9 | TrainLoss 0.11

In [ ]:
# ============================================================
# CELL 39 — TANGDAOWAN FOCAL THREE-SEED CLEAN-ROOM AUDIT
# ============================================================

TANGDAOWAN_FOCAL_AUDITS = {}


print("=" * 125)
print("TANGDAOWAN FOCAL γ=1.0 — THREE-SEED ARTIFACT AUDIT")
print("=" * 125)


for seed in TRAINING_SEEDS:

    audit = audit_longtail_run(
        dataset_name="Tangdaowan",
        method="Focal",
        candidate=1.0,
        seed=seed
    )


    TANGDAOWAN_FOCAL_AUDITS[
        seed
    ] = audit


    best = audit[
        "recomputed_best"
    ]


    print(
        f"Seed {seed:4d} | "
        f"Epoch={best['epoch']:3d} | "
        f"ValAA={best['aa']:.6f} | "
        f"F1={best['macro_f1']:.6f} | "
        f"ValOA={best['oa']:.6f} | "
        f"Artifact={'PASS' if audit['pass'] else 'FAIL'}"
    )


ALL_TANGDAOWAN_FOCAL_SEEDS_PASS = all(

    audit[
        "pass"
    ]

    for audit in
    TANGDAOWAN_FOCAL_AUDITS.values()
)


print("-" * 125)

print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print()

print(
    "TANGDAOWAN FOCAL 3-SEED GATE:",
    "PASS"
    if ALL_TANGDAOWAN_FOCAL_SEEDS_PASS
    else "FAIL"
)

print("=" * 125)


if not ALL_TANGDAOWAN_FOCAL_SEEDS_PASS:

    raise RuntimeError(
        "Tangdaowan Focal three-seed audit failed."
    )

TANGDAOWAN FOCAL γ=1.0 — THREE-SEED ARTIFACT AUDIT
Seed   42 | Epoch= 83 | ValAA=0.999821 | F1=0.999734 | ValOA=0.999623 | Artifact=PASS
Seed  123 | Epoch= 94 | ValAA=0.999907 | F1=0.999932 | ValOA=0.999812 | Artifact=PASS
Seed 3407 | Epoch= 70 | ValAA=0.999598 | F1=0.999423 | ValOA=0.999435 | Artifact=PASS
-----------------------------------------------------------------------------------------------------------------------------
Official-test dataset created    : False
Official-test DataLoader created : False

TANGDAOWAN FOCAL 3-SEED GATE: PASS


In [ ]:
# ============================================================
# CELL 40 — FREEZE TANGDAOWAN FOCAL THREE-SEED RESULT
# ============================================================

assert ALL_TANGDAOWAN_FOCAL_SEEDS_PASS


records = []


for seed in TRAINING_SEEDS:

    audit = TANGDAOWAN_FOCAL_AUDITS[
        seed
    ]

    best = audit[
        "recomputed_best"
    ]


    records.append({

        "dataset":
            "Tangdaowan",

        "method":
            "Focal",

        "gamma":
            1.0,

        "seed":
            int(seed),

        "best_epoch":
            int(
                best["epoch"]
            ),

        "validation_aa":
            float(
                best["aa"]
            ),

        "validation_macro_f1":
            float(
                best["macro_f1"]
            ),

        "validation_oa":
            float(
                best["oa"]
            ),

        "validation_kappa":
            float(
                best["kappa"]
            ),

        "validation_loss":
            float(
                best["loss"]
            ),

        "checkpoint_sha256":
            audit[
                "checkpoint_sha256"
            ],
    })


TANGDAOWAN_FOCAL_3SEED_DF = pd.DataFrame(
    records
)


summary = {

    "validation_aa_mean":
        float(
            TANGDAOWAN_FOCAL_3SEED_DF[
                "validation_aa"
            ].mean()
        ),

    "validation_aa_std":
        float(
            TANGDAOWAN_FOCAL_3SEED_DF[
                "validation_aa"
            ].std(
                ddof=1
            )
        ),

    "validation_macro_f1_mean":
        float(
            TANGDAOWAN_FOCAL_3SEED_DF[
                "validation_macro_f1"
            ].mean()
        ),

    "validation_macro_f1_std":
        float(
            TANGDAOWAN_FOCAL_3SEED_DF[
                "validation_macro_f1"
            ].std(
                ddof=1
            )
        ),

    "validation_oa_mean":
        float(
            TANGDAOWAN_FOCAL_3SEED_DF[
                "validation_oa"
            ].mean()
        ),

    "validation_oa_std":
        float(
            TANGDAOWAN_FOCAL_3SEED_DF[
                "validation_oa"
            ].std(
                ddof=1
            )
        ),
}


TANGDAOWAN_FOCAL_MANIFEST = {

    "dataset":
        "Tangdaowan",

    "method":
        "Focal",

    "selected_gamma":
        1.0,

    "selection_source":
        "seed42_validation_only",

    "training_seeds":
        TRAINING_SEEDS,

    "runs":
        records,

    "summary":
        summary,

    "protocol_sha256":
        protocol_sha256,

    "engine_sha256":
        ENGINE_SPEC_SHA256,

    "hyperparameter_selection_sha256":
        TANGDAOWAN_FOCAL_SELECTION_SHA,

    "official_test_used":
        False,
}


TANGDAOWAN_FOCAL_MANIFEST_PATH = (

    NOTEBOOK04_AUDIT_DIR /
    "tangdaowan_focal_three_seed_manifest.json"
)


with open(
    TANGDAOWAN_FOCAL_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        TANGDAOWAN_FOCAL_MANIFEST,
        f,
        indent=2,
        sort_keys=True
    )


TANGDAOWAN_FOCAL_MANIFEST_SHA = sha256_file(
    TANGDAOWAN_FOCAL_MANIFEST_PATH
)


print("=" * 120)
print("TANGDAOWAN FOCAL — FROZEN THREE-SEED VALIDATION RESULT")
print("=" * 120)

display(
    TANGDAOWAN_FOCAL_3SEED_DF[
        [
            "seed",
            "best_epoch",
            "validation_aa",
            "validation_macro_f1",
            "validation_oa",
        ]
    ]
)

print()

print(
    "Val AA       : "
    f"{summary['validation_aa_mean']:.6f} ± "
    f"{summary['validation_aa_std']:.6f}"
)

print(
    "Val Macro-F1 : "
    f"{summary['validation_macro_f1_mean']:.6f} ± "
    f"{summary['validation_macro_f1_std']:.6f}"
)

print(
    "Val OA       : "
    f"{summary['validation_oa_mean']:.6f} ± "
    f"{summary['validation_oa_std']:.6f}"
)

print()

print(
    "Selected gamma:",
    1.0
)

print(
    "Manifest SHA :",
    TANGDAOWAN_FOCAL_MANIFEST_SHA
)

print(
    "Official test used:",
    False
)

print("=" * 120)

print(
    "TANGDAOWAN FOCAL FREEZE GATE: PASS"
)

print("=" * 120)

TANGDAOWAN FOCAL — FROZEN THREE-SEED VALIDATION RESULT


,seed,best_epoch,validation_aa,validation_macro_f1,validation_oa
0,42,83,0.999821,0.999734,0.999623
1,123,94,0.999907,0.999932,0.999812
2,3407,70,0.999598,0.999423,0.999435



Val AA       : 0.999776 ± 0.000159
Val Macro-F1 : 0.999696 ± 0.000257
Val OA       : 0.999623 ± 0.000188

Selected gamma: 1.0
Manifest SHA : 4d085bf564adb71db708e9f579fb9a3363908b7d130d0cb9eaf0828da30acf4e
Official test used: False
TANGDAOWAN FOCAL FREEZE GATE: PASS


In [ ]:
# ============================================================
# CELL 41 — TANGDAOWAN LDAM-DRW TUNING RUNS
# C=0.25 AND C=0.50 | SEED 42
# ============================================================

assert TANGDAOWAN_FOCAL_MANIFEST_PATH.exists()

assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


TANGDAOWAN_LDAM_C025_META = (
    run_or_recover_longtail(
        dataset_name="Tangdaowan",
        method="LDAM_DRW",
        candidate=0.25,
        seed=42
    )
)


TANGDAOWAN_LDAM_C050_META = (
    run_or_recover_longtail(
        dataset_name="Tangdaowan",
        method="LDAM_DRW",
        candidate=0.50,
        seed=42
    )
)


print()
print("=" * 125)
print("TANGDAOWAN LDAM-DRW — TUNING RUNS COMPLETE")
print("=" * 125)


for meta in [
    TANGDAOWAN_LDAM_C025_META,
    TANGDAOWAN_LDAM_C050_META,
]:

    print(
        f"C={float(meta['candidate']):.2f} | "
        f"Seed={int(meta['seed'])} | "
        f"BestEpoch={int(meta['best_epoch']):3d} | "
        f"ValAA={float(meta['best_validation']['aa']):.6f} | "
        f"F1={float(meta['best_validation']['macro_f1']):.6f} | "
        f"ValOA={float(meta['best_validation']['oa']):.6f}"
    )


print("=" * 125)


NEW RUN REQUIRED: Tangdaowan | LDAM_DRW | candidate=0.25 | seed=42
TRAINING RUN | Tangdaowan | LDAM_DRW | candidate=0.25 | seed=42
Epoch   1 | TrainLoss 3.4425 | ValLoss 1.8578 | ValOA 0.755089 | ValAA 0.397065 | F1 0.386147 | 5.8s BEST
Epoch   2 | TrainLoss 1.5189 | ValLoss 1.1377 | ValOA 0.853279 | ValAA 0.550826 | F1 0.537101 | 3.7s BEST
Epoch   3 | TrainLoss 1.0249 | ValLoss 0.8544 | ValOA 0.896344 | ValAA 0.625297 | F1 0.630870 | 2.4s BEST
Epoch   4 | TrainLoss 0.6496 | ValLoss 0.3755 | ValOA 0.958538 | ValAA 0.769683 | F1 0.786995 | 2.4s BEST
Epoch   5 | TrainLoss 0.4431 | ValLoss 0.2712 | ValOA 0.969563 | ValAA 0.837317 | F1 0.832820 | 2.4s BEST
Epoch   6 | TrainLoss 0.3196 | ValLoss 0.2046 | ValOA 0.976913 | ValAA 0.854815 | F1 0.840767 | 2.9s BEST
Epoch   8 | TrainLoss 0.2238 | ValLoss 0.2159 | ValOA 0.977101 | ValAA 0.871684 | F1 0.877552 | 2.4s BEST
Epoch   9 | TrainLoss 0.1680 | ValLoss 0.1129 | ValOA 0.987750 | ValAA 0.912188 | F1 0.919475 | 2.5s BEST
Epoch  10 | TrainLos

In [ ]:
# ============================================================
# CELL 42 — TANGDAOWAN LDAM-DRW
# TWO-CANDIDATE CLEAN-ROOM AUDIT
# ============================================================

TANGDAOWAN_LDAM_TUNING_AUDITS = {}


print("=" * 125)
print("TANGDAOWAN LDAM-DRW — SEED 42 CANDIDATE AUDIT")
print("=" * 125)


for C in [
    0.25,
    0.50,
]:

    audit = audit_longtail_run(
        dataset_name="Tangdaowan",
        method="LDAM_DRW",
        candidate=C,
        seed=42
    )


    TANGDAOWAN_LDAM_TUNING_AUDITS[
        C
    ] = audit


    best = audit[
        "recomputed_best"
    ]


    print(
        f"C={C:.2f} | "
        f"Epoch={best['epoch']:3d} | "
        f"ValAA={best['aa']:.6f} | "
        f"F1={best['macro_f1']:.6f} | "
        f"ValOA={best['oa']:.6f} | "
        f"Artifact={'PASS' if audit['pass'] else 'FAIL'}"
    )


ALL_TANGDAOWAN_LDAM_TUNING_RUNS_PASS = all(

    audit["pass"]

    for audit in
    TANGDAOWAN_LDAM_TUNING_AUDITS.values()
)


print("-" * 125)

print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print()

print(
    "TANGDAOWAN LDAM TUNING-RUN AUDIT GATE:",
    "PASS"
    if ALL_TANGDAOWAN_LDAM_TUNING_RUNS_PASS
    else "FAIL"
)

print("=" * 125)


if not ALL_TANGDAOWAN_LDAM_TUNING_RUNS_PASS:

    raise RuntimeError(
        "Tangdaowan LDAM tuning-run audit failed."
    )

TANGDAOWAN LDAM-DRW — SEED 42 CANDIDATE AUDIT
C=0.25 | Epoch= 38 | ValAA=0.999703 | F1=0.999525 | ValOA=0.999529 | Artifact=PASS
C=0.50 | Epoch= 78 | ValAA=1.000000 | F1=1.000000 | ValOA=1.000000 | Artifact=PASS
-----------------------------------------------------------------------------------------------------------------------------
Official-test dataset created    : False
Official-test DataLoader created : False

TANGDAOWAN LDAM TUNING-RUN AUDIT GATE: PASS


In [ ]:
# ============================================================
# CELL 43 — TANGDAOWAN LDAM-DRW
# VALIDATION-ONLY MARGIN SELECTION
# ============================================================

assert ALL_TANGDAOWAN_LDAM_TUNING_RUNS_PASS

assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


ldam_candidate_records = []


for C in [
    0.25,
    0.50,
]:

    audit = (
        TANGDAOWAN_LDAM_TUNING_AUDITS[
            C
        ]
    )

    best = audit[
        "recomputed_best"
    ]


    ldam_candidate_records.append({

        "max_margin_C":
            float(C),

        "best_epoch":
            int(
                best["epoch"]
            ),

        "validation_aa":
            float(
                best["aa"]
            ),

        "validation_macro_f1":
            float(
                best["macro_f1"]
            ),

        "validation_oa":
            float(
                best["oa"]
            ),

        "validation_loss":
            float(
                best["loss"]
            ),

        "checkpoint_sha256":
            audit[
                "checkpoint_sha256"
            ],
    })


TANGDAOWAN_LDAM_TUNING_DF = pd.DataFrame(
    ldam_candidate_records
)


def better_numeric_tuning_candidate(
    candidate,
    best,
    parameter_key
):

    """
    Frozen hyperparameter selection:

    1. higher validation AA
    2. higher validation Macro-F1
    3. higher validation OA
    4. smaller/simpler numeric hyperparameter
    """

    if best is None:

        return True


    # ------------------------------------------
    # 1. Validation AA
    # ------------------------------------------

    if (
        candidate["validation_aa"]
        >
        best["validation_aa"]
        +
        CHECKPOINT_TOL
    ):

        return True


    if not math.isclose(
        candidate["validation_aa"],
        best["validation_aa"],
        abs_tol=CHECKPOINT_TOL
    ):

        return False


    # ------------------------------------------
    # 2. Macro-F1
    # ------------------------------------------

    if (
        candidate["validation_macro_f1"]
        >
        best["validation_macro_f1"]
        +
        CHECKPOINT_TOL
    ):

        return True


    if not math.isclose(
        candidate["validation_macro_f1"],
        best["validation_macro_f1"],
        abs_tol=CHECKPOINT_TOL
    ):

        return False


    # ------------------------------------------
    # 3. Validation OA
    # ------------------------------------------

    if (
        candidate["validation_oa"]
        >
        best["validation_oa"]
        +
        CHECKPOINT_TOL
    ):

        return True


    if not math.isclose(
        candidate["validation_oa"],
        best["validation_oa"],
        abs_tol=CHECKPOINT_TOL
    ):

        return False


    # ------------------------------------------
    # 4. Simpler/smaller parameter
    # ------------------------------------------

    return (
        float(
            candidate[
                parameter_key
            ]
        )
        <
        float(
            best[
                parameter_key
            ]
        )
    )


selected_ldam = None


for record in ldam_candidate_records:

    if better_numeric_tuning_candidate(
        candidate=record,
        best=selected_ldam,
        parameter_key="max_margin_C"
    ):

        selected_ldam = (
            record.copy()
        )


TANGDAOWAN_LDAM_SELECTED_C = float(
    selected_ldam[
        "max_margin_C"
    ]
)


TANGDAOWAN_LDAM_SELECTION = {

    "dataset":
        "Tangdaowan",

    "method":
        "LDAM_DRW",

    "tuning_seed":
        42,

    "candidate_max_margin_C":
        [
            0.25,
            0.50
        ],

    "scale":
        30.0,

    "drw_start_epoch":
        80,

    "effective_number_beta":
        0.9999,

    "selection_primary":
        "validation_AA",

    "tie_1":
        "validation_Macro_F1",

    "tie_2":
        "validation_OA",

    "final_tie":
        "smaller_max_margin_C",

    "selected_max_margin_C":
        TANGDAOWAN_LDAM_SELECTED_C,

    "selected_best_epoch":
        int(
            selected_ldam[
                "best_epoch"
            ]
        ),

    "selected_validation_aa":
        float(
            selected_ldam[
                "validation_aa"
            ]
        ),

    "selected_validation_macro_f1":
        float(
            selected_ldam[
                "validation_macro_f1"
            ]
        ),

    "selected_validation_oa":
        float(
            selected_ldam[
                "validation_oa"
            ]
        ),

    "candidate_results":
        ldam_candidate_records,

    "protocol_sha256":
        protocol_sha256,

    "engine_sha256":
        ENGINE_SPEC_SHA256,

    "official_test_used":
        False,
}


TANGDAOWAN_LDAM_SELECTION_PATH = (

    NOTEBOOK04_AUDIT_DIR /

    "tangdaowan_ldam_drw_hyperparameter_selection.json"
)


if TANGDAOWAN_LDAM_SELECTION_PATH.exists():

    with open(
        TANGDAOWAN_LDAM_SELECTION_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        existing = json.load(f)


    if existing != TANGDAOWAN_LDAM_SELECTION:

        raise RuntimeError(
            "Existing Tangdaowan LDAM selection "
            "does not match current validation result."
        )

else:

    with open(
        TANGDAOWAN_LDAM_SELECTION_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            TANGDAOWAN_LDAM_SELECTION,
            f,
            indent=2,
            sort_keys=True
        )


TANGDAOWAN_LDAM_SELECTION_SHA = sha256_file(
    TANGDAOWAN_LDAM_SELECTION_PATH
)


print("=" * 125)
print("TANGDAOWAN — LDAM-DRW VALIDATION-ONLY TUNING")
print("=" * 125)


display(

    TANGDAOWAN_LDAM_TUNING_DF[
        [
            "max_margin_C",
            "best_epoch",
            "validation_aa",
            "validation_macro_f1",
            "validation_oa",
        ]
    ]
)


print()

print(
    "SELECTED C:",
    TANGDAOWAN_LDAM_SELECTED_C
)

print(
    "Selected Val AA:",
    f"{selected_ldam['validation_aa']:.6f}"
)

print(
    "Selected F1    :",
    f"{selected_ldam['validation_macro_f1']:.6f}"
)

print(
    "Selected Val OA:",
    f"{selected_ldam['validation_oa']:.6f}"
)

print()

print(
    "Official test used:",
    False
)

print(
    "Selection SHA:",
    TANGDAOWAN_LDAM_SELECTION_SHA
)

print("=" * 125)

print(
    "TANGDAOWAN LDAM TUNING GATE: PASS"
)

print("=" * 125)

TANGDAOWAN — LDAM-DRW VALIDATION-ONLY TUNING


,max_margin_C,best_epoch,validation_aa,validation_macro_f1,validation_oa
0,0.25,38,0.999703,0.999525,0.999529
1,0.50,78,1.000000,1.000000,1.000000



SELECTED C: 0.5
Selected Val AA: 1.000000
Selected F1    : 1.000000
Selected Val OA: 1.000000

Official test used: False
Selection SHA: 39811a58a263e27cb94a69980b35572f5011b5831ee5106d4ce782d1a0804f99
TANGDAOWAN LDAM TUNING GATE: PASS


In [ ]:
# ============================================================
# CELL 44 — TANGDAOWAN LDAM-DRW
# CHECKPOINT-vs-DRW SCHEDULE AUDIT
# ============================================================

assert TANGDAOWAN_LDAM_SELECTED_C == 0.50


selected_audit = (
    TANGDAOWAN_LDAM_TUNING_AUDITS[
        0.50
    ]
)

selected_best = (
    selected_audit[
        "recomputed_best"
    ]
)

selected_best_epoch = int(
    selected_best[
        "epoch"
    ]
)

DRW_START_EPOCH = 80


selected_checkpoint_uses_drw = (
    selected_best_epoch
    >=
    DRW_START_EPOCH
)


# ------------------------------------------------------------
# Reload history to inspect transition around epoch 80
# ------------------------------------------------------------

run_id = make_baseline_run_id(
    "Tangdaowan",
    "LDAM_DRW",
    0.50,
    42
)

history_path = (
    LONGTAIL_HISTORY_DIR /
    f"{run_id}_history.csv"
)

history_df = pd.read_csv(
    history_path
)


transition_rows = history_df[
    history_df["epoch"].isin(
        [78, 79, 80, 81, 100]
    )
][
    [
        "epoch",
        "validation_aa",
        "validation_macro_f1",
        "validation_oa",
        "validation_loss",
    ]
]


LDAM_DRW_CHECKPOINT_NOTE = {

    "dataset":
        "Tangdaowan",

    "method":
        "LDAM_DRW",

    "selected_C":
        0.50,

    "seed":
        42,

    "best_epoch":
        selected_best_epoch,

    "drw_start_epoch":
        DRW_START_EPOCH,

    "selected_checkpoint_after_drw_start":
        bool(
            selected_checkpoint_uses_drw
        ),

    "interpretation":
        (
            "The run followed the frozen LDAM-DRW schedule "
            "through 100 epochs, but validation-AA checkpoint "
            "selection chose a checkpoint before DRW activation."
        ),

    "checkpoint_rule_changed":
        False,

    "official_test_used":
        False,
}


LDAM_DRW_CHECKPOINT_NOTE_PATH = (

    NOTEBOOK04_AUDIT_DIR /
    "tangdaowan_ldam_drw_checkpoint_schedule_note.json"
)


with open(
    LDAM_DRW_CHECKPOINT_NOTE_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        LDAM_DRW_CHECKPOINT_NOTE,
        f,
        indent=2,
        sort_keys=True
    )


LDAM_DRW_CHECKPOINT_NOTE_SHA = sha256_file(
    LDAM_DRW_CHECKPOINT_NOTE_PATH
)


print("=" * 120)
print("TANGDAOWAN LDAM-DRW — CHECKPOINT/SCHEDULE AUDIT")
print("=" * 120)

display(
    transition_rows
)

print()

print(
    "Selected C            :",
    0.50
)

print(
    "Selected best epoch   :",
    selected_best_epoch
)

print(
    "DRW starts at epoch   :",
    DRW_START_EPOCH
)

print(
    "Selected checkpoint "
    "uses DRW            :",
    selected_checkpoint_uses_drw
)

print()

print(
    "Checkpoint rule changed:",
    False
)

print(
    "Official test used     :",
    False
)

print(
    "Audit-note SHA         :",
    LDAM_DRW_CHECKPOINT_NOTE_SHA
)

print("=" * 120)

print(
    "LDAM-DRW SCHEDULE DISCLOSURE GATE: PASS"
)

print("=" * 120)

TANGDAOWAN LDAM-DRW — CHECKPOINT/SCHEDULE AUDIT


,epoch,validation_aa,validation_macro_f1,validation_oa,validation_loss
77,78,1.000000,1.000000,1.000000,0.001763
78,79,0.999768,0.997470,0.998963,0.013079
79,80,0.942734,0.838559,0.932152,1.365481
80,81,0.978255,0.878032,0.936864,1.122536
99,100,0.998920,0.994217,0.996796,0.027099



Selected C            : 0.5
Selected best epoch   : 78
DRW starts at epoch   : 80
Selected checkpoint uses DRW            : False

Checkpoint rule changed: False
Official test used     : False
Audit-note SHA         : d936c660b91a21ddb9e9f9acf0df1e73ce7d3938f9e1dad47f7b8b1233c8309f
LDAM-DRW SCHEDULE DISCLOSURE GATE: PASS


In [ ]:
# ============================================================
# CELL 45 — TANGDAOWAN LDAM-DRW SELECTED C=0.50
# SEEDS 123 AND 3407
# ============================================================

assert TANGDAOWAN_LDAM_SELECTED_C == 0.50

assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


TANGDAOWAN_LDAM_SEED123_META = (
    run_or_recover_longtail(
        dataset_name="Tangdaowan",
        method="LDAM_DRW",
        candidate=0.50,
        seed=123
    )
)


TANGDAOWAN_LDAM_SEED3407_META = (
    run_or_recover_longtail(
        dataset_name="Tangdaowan",
        method="LDAM_DRW",
        candidate=0.50,
        seed=3407
    )
)


print()
print("=" * 125)
print("TANGDAOWAN LDAM-DRW — REMAINING SEEDS COMPLETE")
print("=" * 125)


for meta in [
    TANGDAOWAN_LDAM_SEED123_META,
    TANGDAOWAN_LDAM_SEED3407_META,
]:

    best_epoch = int(
        meta[
            "best_epoch"
        ]
    )

    uses_drw = (
        best_epoch
        >=
        80
    )


    print(
        f"Seed {int(meta['seed']):4d} | "
        f"BestEpoch={best_epoch:3d} | "
        f"ValAA={float(meta['best_validation']['aa']):.6f} | "
        f"F1={float(meta['best_validation']['macro_f1']):.6f} | "
        f"ValOA={float(meta['best_validation']['oa']):.6f} | "
        f"BestCheckpointUsesDRW={uses_drw}"
    )


print("=" * 125)


NEW RUN REQUIRED: Tangdaowan | LDAM_DRW | candidate=0.5 | seed=123
TRAINING RUN | Tangdaowan | LDAM_DRW | candidate=0.5 | seed=123
Epoch   1 | TrainLoss 6.3982 | ValLoss 4.1024 | ValOA 0.699585 | ValAA 0.341639 | F1 0.315065 | 2.5s BEST
Epoch   2 | TrainLoss 3.2936 | ValLoss 2.5414 | ValOA 0.805692 | ValAA 0.485732 | F1 0.470843 | 2.4s BEST
Epoch   3 | TrainLoss 2.3389 | ValLoss 2.0136 | ValOA 0.848191 | ValAA 0.523878 | F1 0.514570 | 2.5s BEST
Epoch   4 | TrainLoss 1.8347 | ValLoss 1.4257 | ValOA 0.899548 | ValAA 0.642644 | F1 0.640189 | 3.1s BEST
Epoch   5 | TrainLoss 1.3060 | ValLoss 0.9788 | ValOA 0.933283 | ValAA 0.751981 | F1 0.740010 | 2.8s BEST
Epoch   6 | TrainLoss 0.8839 | ValLoss 0.6818 | ValOA 0.956464 | ValAA 0.819314 | F1 0.791084 | 2.5s BEST
Epoch   7 | TrainLoss 0.6330 | ValLoss 0.3983 | ValOA 0.974274 | ValAA 0.846162 | F1 0.825510 | 2.5s BEST
Epoch   8 | TrainLoss 0.5039 | ValLoss 0.3258 | ValOA 0.981813 | ValAA 0.852713 | F1 0.844953 | 2.5s BEST
Epoch   9 | TrainLos

In [ ]:
# ============================================================
# CELL 46 — TANGDAOWAN LDAM-DRW
# THREE-SEED CLEAN-ROOM AUDIT
# ============================================================

TANGDAOWAN_LDAM_AUDITS = {}


print("=" * 130)
print("TANGDAOWAN LDAM-DRW C=0.50 — THREE-SEED ARTIFACT AUDIT")
print("=" * 130)


for seed in TRAINING_SEEDS:

    audit = audit_longtail_run(
        dataset_name="Tangdaowan",
        method="LDAM_DRW",
        candidate=0.50,
        seed=seed
    )


    TANGDAOWAN_LDAM_AUDITS[
        seed
    ] = audit


    best = audit[
        "recomputed_best"
    ]


    best_epoch = int(
        best[
            "epoch"
        ]
    )


    uses_drw = (
        best_epoch
        >=
        80
    )


    print(
        f"Seed {seed:4d} | "
        f"Epoch={best_epoch:3d} | "
        f"ValAA={best['aa']:.6f} | "
        f"F1={best['macro_f1']:.6f} | "
        f"ValOA={best['oa']:.6f} | "
        f"UsesDRW={'YES' if uses_drw else 'NO ':3s} | "
        f"Artifact={'PASS' if audit['pass'] else 'FAIL'}"
    )


ALL_TANGDAOWAN_LDAM_SEEDS_PASS = all(

    audit[
        "pass"
    ]

    for audit in
    TANGDAOWAN_LDAM_AUDITS.values()
)


print("-" * 130)

print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print()

print(
    "TANGDAOWAN LDAM-DRW 3-SEED GATE:",
    "PASS"
    if ALL_TANGDAOWAN_LDAM_SEEDS_PASS
    else "FAIL"
)

print("=" * 130)


if not ALL_TANGDAOWAN_LDAM_SEEDS_PASS:

    raise RuntimeError(
        "Tangdaowan LDAM-DRW three-seed audit failed."
    )

TANGDAOWAN LDAM-DRW C=0.50 — THREE-SEED ARTIFACT AUDIT
Seed   42 | Epoch= 78 | ValAA=1.000000 | F1=1.000000 | ValOA=1.000000 | UsesDRW=NO  | Artifact=PASS
Seed  123 | Epoch= 78 | ValAA=0.999541 | F1=0.999512 | ValOA=0.999340 | UsesDRW=NO  | Artifact=PASS
Seed 3407 | Epoch= 79 | ValAA=0.999937 | F1=0.999651 | ValOA=0.999717 | UsesDRW=NO  | Artifact=PASS
----------------------------------------------------------------------------------------------------------------------------------
Official-test dataset created    : False
Official-test DataLoader created : False

TANGDAOWAN LDAM-DRW 3-SEED GATE: PASS


In [ ]:
# ============================================================
# CELL 47 — FREEZE TANGDAOWAN LDAM-DRW
# THREE-SEED VALIDATION RESULT
# ============================================================

assert ALL_TANGDAOWAN_LDAM_SEEDS_PASS


records = []


for seed in TRAINING_SEEDS:

    audit = (
        TANGDAOWAN_LDAM_AUDITS[
            seed
        ]
    )

    best = audit[
        "recomputed_best"
    ]

    best_epoch = int(
        best[
            "epoch"
        ]
    )


    records.append({

        "dataset":
            "Tangdaowan",

        "method":
            "LDAM_DRW",

        "max_margin_C":
            0.50,

        "seed":
            int(seed),

        "best_epoch":
            best_epoch,

        "best_checkpoint_uses_drw":
            bool(
                best_epoch >= 80
            ),

        "validation_aa":
            float(
                best["aa"]
            ),

        "validation_macro_f1":
            float(
                best["macro_f1"]
            ),

        "validation_oa":
            float(
                best["oa"]
            ),

        "validation_kappa":
            float(
                best["kappa"]
            ),

        "validation_loss":
            float(
                best["loss"]
            ),

        "checkpoint_sha256":
            audit[
                "checkpoint_sha256"
            ],
    })


TANGDAOWAN_LDAM_3SEED_DF = pd.DataFrame(
    records
)


summary = {

    "validation_aa_mean":
        float(
            TANGDAOWAN_LDAM_3SEED_DF[
                "validation_aa"
            ].mean()
        ),

    "validation_aa_std":
        float(
            TANGDAOWAN_LDAM_3SEED_DF[
                "validation_aa"
            ].std(
                ddof=1
            )
        ),

    "validation_macro_f1_mean":
        float(
            TANGDAOWAN_LDAM_3SEED_DF[
                "validation_macro_f1"
            ].mean()
        ),

    "validation_macro_f1_std":
        float(
            TANGDAOWAN_LDAM_3SEED_DF[
                "validation_macro_f1"
            ].std(
                ddof=1
            )
        ),

    "validation_oa_mean":
        float(
            TANGDAOWAN_LDAM_3SEED_DF[
                "validation_oa"
            ].mean()
        ),

    "validation_oa_std":
        float(
            TANGDAOWAN_LDAM_3SEED_DF[
                "validation_oa"
            ].std(
                ddof=1
            )
        ),

    "selected_checkpoints_after_drw_start":
        int(
            TANGDAOWAN_LDAM_3SEED_DF[
                "best_checkpoint_uses_drw"
            ].sum()
        ),
}


TANGDAOWAN_LDAM_MANIFEST = {

    "dataset":
        "Tangdaowan",

    "method":
        "LDAM_DRW",

    "selected_max_margin_C":
        0.50,

    "scale":
        30.0,

    "drw_start_epoch":
        80,

    "effective_number_beta":
        0.9999,

    "selection_source":
        "seed42_validation_only",

    "training_seeds":
        TRAINING_SEEDS,

    "runs":
        records,

    "summary":
        summary,

    "protocol_sha256":
        protocol_sha256,

    "engine_sha256":
        ENGINE_SPEC_SHA256,

    "hyperparameter_selection_sha256":
        TANGDAOWAN_LDAM_SELECTION_SHA,

    "schedule_disclosure_sha256":
        LDAM_DRW_CHECKPOINT_NOTE_SHA,

    "official_test_used":
        False,
}


TANGDAOWAN_LDAM_MANIFEST_PATH = (

    NOTEBOOK04_AUDIT_DIR /
    "tangdaowan_ldam_drw_three_seed_manifest.json"
)


with open(
    TANGDAOWAN_LDAM_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        TANGDAOWAN_LDAM_MANIFEST,
        f,
        indent=2,
        sort_keys=True
    )


TANGDAOWAN_LDAM_MANIFEST_SHA = (
    sha256_file(
        TANGDAOWAN_LDAM_MANIFEST_PATH
    )
)


print("=" * 125)
print("TANGDAOWAN LDAM-DRW — FROZEN THREE-SEED VALIDATION RESULT")
print("=" * 125)


display(

    TANGDAOWAN_LDAM_3SEED_DF[
        [
            "seed",
            "best_epoch",
            "best_checkpoint_uses_drw",
            "validation_aa",
            "validation_macro_f1",
            "validation_oa",
        ]
    ]
)


print()

print(
    "Val AA       : "
    f"{summary['validation_aa_mean']:.6f} ± "
    f"{summary['validation_aa_std']:.6f}"
)

print(
    "Val Macro-F1 : "
    f"{summary['validation_macro_f1_mean']:.6f} ± "
    f"{summary['validation_macro_f1_std']:.6f}"
)

print(
    "Val OA       : "
    f"{summary['validation_oa_mean']:.6f} ± "
    f"{summary['validation_oa_std']:.6f}"
)

print()

print(
    "Selected C:",
    0.50
)

print(
    "Checkpoints after DRW activation:",
    f"{summary['selected_checkpoints_after_drw_start']}/3"
)

print(
    "Manifest SHA:",
    TANGDAOWAN_LDAM_MANIFEST_SHA
)

print(
    "Official test used:",
    False
)

print("=" * 125)

print(
    "TANGDAOWAN LDAM-DRW FREEZE GATE: PASS"
)

print("=" * 125)

TANGDAOWAN LDAM-DRW — FROZEN THREE-SEED VALIDATION RESULT


,seed,best_epoch,best_checkpoint_uses_drw,validation_aa,validation_macro_f1,validation_oa
0,42,78,False,1.000000,1.000000,1.000000
1,123,78,False,0.999541,0.999512,0.999340
2,3407,79,False,0.999937,0.999651,0.999717



Val AA       : 0.999826 ± 0.000249
Val Macro-F1 : 0.999721 ± 0.000251
Val OA       : 0.999686 ± 0.000331

Selected C: 0.5
Checkpoints after DRW activation: 0/3
Manifest SHA: 12155bc18fcb85031cafa5a695728a1355913fc21e27072d79eaaafe9b7f9611
Official test used: False
TANGDAOWAN LDAM-DRW FREEZE GATE: PASS


In [ ]:
# ============================================================
# CELL 48 — TANGDAOWAN BALANCED SOFTMAX
# ALL THREE SEEDS
# ============================================================

assert TANGDAOWAN_LDAM_MANIFEST_PATH.exists()

assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


TANGDAOWAN_BS_METADATA = {}


print("=" * 125)
print("TANGDAOWAN BALANCED SOFTMAX — THREE-SEED TRAINING")
print("=" * 125)


for seed in TRAINING_SEEDS:

    meta = run_or_recover_longtail(

        dataset_name=
            "Tangdaowan",

        method=
            "BalancedSoftmax",

        candidate=
            "training_counts",

        seed=
            seed
    )


    TANGDAOWAN_BS_METADATA[
        seed
    ] = meta


print()
print("=" * 125)
print("TANGDAOWAN BALANCED SOFTMAX — TRAINING COMPLETE")
print("=" * 125)


for seed in TRAINING_SEEDS:

    meta = TANGDAOWAN_BS_METADATA[
        seed
    ]


    print(
        f"Seed {seed:4d} | "
        f"BestEpoch={int(meta['best_epoch']):3d} | "
        f"ValAA={float(meta['best_validation']['aa']):.6f} | "
        f"F1={float(meta['best_validation']['macro_f1']):.6f} | "
        f"ValOA={float(meta['best_validation']['oa']):.6f}"
    )


print()

print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print("=" * 125)

TANGDAOWAN BALANCED SOFTMAX — THREE-SEED TRAINING

NEW RUN REQUIRED: Tangdaowan | BalancedSoftmax | candidate=training_counts | seed=42
TRAINING RUN | Tangdaowan | BalancedSoftmax | candidate=training_counts | seed=42
Epoch   1 | TrainLoss 1.8062 | ValLoss 0.9698 | ValOA 0.598662 | ValAA 0.344173 | F1 0.316053 | 4.6s BEST
Epoch   2 | TrainLoss 0.7453 | ValLoss 0.5176 | ValOA 0.716359 | ValAA 0.645765 | F1 0.533873 | 3.3s BEST
Epoch   3 | TrainLoss 0.4856 | ValLoss 0.3708 | ValOA 0.774406 | ValAA 0.746228 | F1 0.595329 | 2.2s BEST
Epoch   4 | TrainLoss 0.3864 | ValLoss 0.2918 | ValOA 0.793913 | ValAA 0.856995 | F1 0.653320 | 2.2s BEST
Epoch   6 | TrainLoss 0.2312 | ValLoss 0.1575 | ValOA 0.896626 | ValAA 0.904211 | F1 0.739902 | 2.4s BEST
Epoch   8 | TrainLoss 0.1333 | ValLoss 0.0964 | ValOA 0.949208 | ValAA 0.966344 | F1 0.864380 | 2.3s BEST
Epoch  10 | TrainLoss 0.0972 | ValLoss 0.0719 | ValOA 0.958632 | ValAA 0.971586 | F1 0.858965 | 2.2s BEST
Epoch  13 | TrainLoss 0.0558 | ValLoss 0

In [ ]:
# ============================================================
# CELL 49 — TANGDAOWAN BALANCED SOFTMAX
# THREE-SEED CLEAN-ROOM AUDIT
# ============================================================

TANGDAOWAN_BS_AUDITS = {}


print("=" * 130)
print("TANGDAOWAN BALANCED SOFTMAX — THREE-SEED ARTIFACT AUDIT")
print("=" * 130)


for seed in TRAINING_SEEDS:

    audit = audit_longtail_run(

        dataset_name=
            "Tangdaowan",

        method=
            "BalancedSoftmax",

        candidate=
            "training_counts",

        seed=
            seed
    )


    TANGDAOWAN_BS_AUDITS[
        seed
    ] = audit


    best = audit[
        "recomputed_best"
    ]


    print(
        f"Seed {seed:4d} | "
        f"Epoch={int(best['epoch']):3d} | "
        f"ValAA={best['aa']:.6f} | "
        f"F1={best['macro_f1']:.6f} | "
        f"ValOA={best['oa']:.6f} | "
        f"Artifact={'PASS' if audit['pass'] else 'FAIL'}"
    )


ALL_TANGDAOWAN_BS_SEEDS_PASS = all(

    audit[
        "pass"
    ]

    for audit in
    TANGDAOWAN_BS_AUDITS.values()
)


print("-" * 130)

print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print()

print(
    "TANGDAOWAN BALANCED SOFTMAX 3-SEED GATE:",
    "PASS"
    if ALL_TANGDAOWAN_BS_SEEDS_PASS
    else "FAIL"
)

print("=" * 130)


if not ALL_TANGDAOWAN_BS_SEEDS_PASS:

    raise RuntimeError(
        "Tangdaowan Balanced Softmax "
        "three-seed audit failed."
    )

TANGDAOWAN BALANCED SOFTMAX — THREE-SEED ARTIFACT AUDIT
Seed   42 | Epoch= 83 | ValAA=0.999979 | F1=0.999919 | ValOA=0.999906 | Artifact=PASS
Seed  123 | Epoch= 81 | ValAA=0.999771 | F1=0.999262 | ValOA=0.999246 | Artifact=PASS
Seed 3407 | Epoch= 99 | ValAA=0.999937 | F1=0.999686 | ValOA=0.999717 | Artifact=PASS
----------------------------------------------------------------------------------------------------------------------------------
Official-test dataset created    : False
Official-test DataLoader created : False

TANGDAOWAN BALANCED SOFTMAX 3-SEED GATE: PASS


In [ ]:
# ============================================================
# CELL 50 — FREEZE TANGDAOWAN BALANCED SOFTMAX
# THREE-SEED VALIDATION RESULT
# ============================================================

assert ALL_TANGDAOWAN_BS_SEEDS_PASS


records = []


for seed in TRAINING_SEEDS:

    audit = (
        TANGDAOWAN_BS_AUDITS[
            seed
        ]
    )

    best = audit[
        "recomputed_best"
    ]


    records.append({

        "dataset":
            "Tangdaowan",

        "method":
            "BalancedSoftmax",

        "class_statistics":
            "training_counts",

        "seed":
            int(seed),

        "best_epoch":
            int(
                best[
                    "epoch"
                ]
            ),

        "validation_aa":
            float(
                best[
                    "aa"
                ]
            ),

        "validation_macro_f1":
            float(
                best[
                    "macro_f1"
                ]
            ),

        "validation_oa":
            float(
                best[
                    "oa"
                ]
            ),

        "validation_kappa":
            float(
                best[
                    "kappa"
                ]
            ),

        "validation_loss":
            float(
                best[
                    "loss"
                ]
            ),

        "checkpoint_sha256":
            audit[
                "checkpoint_sha256"
            ],
    })


TANGDAOWAN_BS_3SEED_DF = pd.DataFrame(
    records
)


bs_summary = {

    "validation_aa_mean":
        float(
            TANGDAOWAN_BS_3SEED_DF[
                "validation_aa"
            ].mean()
        ),

    "validation_aa_std":
        float(
            TANGDAOWAN_BS_3SEED_DF[
                "validation_aa"
            ].std(
                ddof=1
            )
        ),

    "validation_macro_f1_mean":
        float(
            TANGDAOWAN_BS_3SEED_DF[
                "validation_macro_f1"
            ].mean()
        ),

    "validation_macro_f1_std":
        float(
            TANGDAOWAN_BS_3SEED_DF[
                "validation_macro_f1"
            ].std(
                ddof=1
            )
        ),

    "validation_oa_mean":
        float(
            TANGDAOWAN_BS_3SEED_DF[
                "validation_oa"
            ].mean()
        ),

    "validation_oa_std":
        float(
            TANGDAOWAN_BS_3SEED_DF[
                "validation_oa"
            ].std(
                ddof=1
            )
        ),
}


TANGDAOWAN_BS_MANIFEST = {

    "dataset":
        "Tangdaowan",

    "method":
        "BalancedSoftmax",

    "class_statistics":
        "training_counts",

    "hyperparameter_tuning_required":
        False,

    "training_seeds":
        TRAINING_SEEDS,

    "runs":
        records,

    "summary":
        bs_summary,

    "protocol_sha256":
        protocol_sha256,

    "engine_sha256":
        ENGINE_SPEC_SHA256,

    "official_test_used":
        False,
}


TANGDAOWAN_BS_MANIFEST_PATH = (

    NOTEBOOK04_AUDIT_DIR /

    "tangdaowan_balanced_softmax_three_seed_manifest.json"
)


with open(
    TANGDAOWAN_BS_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        TANGDAOWAN_BS_MANIFEST,
        f,
        indent=2,
        sort_keys=True
    )


TANGDAOWAN_BS_MANIFEST_SHA = (
    sha256_file(
        TANGDAOWAN_BS_MANIFEST_PATH
    )
)


print("=" * 125)
print("TANGDAOWAN BALANCED SOFTMAX — FROZEN THREE-SEED VALIDATION RESULT")
print("=" * 125)


display(

    TANGDAOWAN_BS_3SEED_DF[
        [
            "seed",
            "best_epoch",
            "validation_aa",
            "validation_macro_f1",
            "validation_oa",
        ]
    ]
)


print()

print(
    "Val AA       : "
    f"{bs_summary['validation_aa_mean']:.6f} ± "
    f"{bs_summary['validation_aa_std']:.6f}"
)

print(
    "Val Macro-F1 : "
    f"{bs_summary['validation_macro_f1_mean']:.6f} ± "
    f"{bs_summary['validation_macro_f1_std']:.6f}"
)

print(
    "Val OA       : "
    f"{bs_summary['validation_oa_mean']:.6f} ± "
    f"{bs_summary['validation_oa_std']:.6f}"
)

print()

print(
    "Class statistics:",
    "training counts only"
)

print(
    "Hyperparameter tuning:",
    "NONE"
)

print(
    "Manifest SHA:",
    TANGDAOWAN_BS_MANIFEST_SHA
)

print(
    "Official test used:",
    False
)

print("=" * 125)

print(
    "TANGDAOWAN BALANCED SOFTMAX FREEZE GATE: PASS"
)

print("=" * 125)

TANGDAOWAN BALANCED SOFTMAX — FROZEN THREE-SEED VALIDATION RESULT


,seed,best_epoch,validation_aa,validation_macro_f1,validation_oa
0,42,83,0.999979,0.999919,0.999906
1,123,81,0.999771,0.999262,0.999246
2,3407,99,0.999937,0.999686,0.999717



Val AA       : 0.999895 ± 0.000110
Val Macro-F1 : 0.999622 ± 0.000334
Val OA       : 0.999623 ± 0.000340

Class statistics: training counts only
Hyperparameter tuning: NONE
Manifest SHA: 9698291fad402aa7795cb2a6d972552bc5e8d54357f5a913fe11428b07285591
Official test used: False
TANGDAOWAN BALANCED SOFTMAX FREEZE GATE: PASS


In [ ]:
# ============================================================
# CELL 51 — TANGDAOWAN LA-LOSS TUNING
# RESTART-SAFE RECOVERY / TRAINING
# tau=0.5 AND tau=1.0 | SEED 42
# ============================================================

assert NOTEBOOK04_REAL_TRAINING_GATE_PASS

assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


# ------------------------------------------------------------
# Previous baseline must already be frozen
# ------------------------------------------------------------

TANGDAOWAN_BS_MANIFEST_PATH = (
    NOTEBOOK04_AUDIT_DIR /
    "tangdaowan_balanced_softmax_three_seed_manifest.json"
)


if not TANGDAOWAN_BS_MANIFEST_PATH.exists():

    raise FileNotFoundError(
        "Tangdaowan Balanced Softmax manifest is missing."
    )


# ------------------------------------------------------------
# Restart-safe helper
# ------------------------------------------------------------

def recover_or_train_la_run(
    tau
):

    run_id = make_baseline_run_id(
        "Tangdaowan",
        "LA_Loss",
        tau,
        42
    )


    checkpoint_path = (
        LONGTAIL_CHECKPOINT_DIR /
        f"{run_id}.pt"
    )

    history_path = (
        LONGTAIL_HISTORY_DIR /
        f"{run_id}_history.csv"
    )

    metadata_path = (
        LONGTAIL_METADATA_DIR /
        f"{run_id}_metadata.json"
    )


    existence = {
        "checkpoint":
            checkpoint_path.exists(),

        "history":
            history_path.exists(),

        "metadata":
            metadata_path.exists(),
    }


    print()
    print("=" * 125)

    print(
        f"TANGDAOWAN LA-LOSS τ={tau:.1f} | "
        f"SEED 42"
    )

    print("=" * 125)

    print(
        "Checkpoint exists:",
        existence["checkpoint"]
    )

    print(
        "History exists   :",
        existence["history"]
    )

    print(
        "Metadata exists  :",
        existence["metadata"]
    )


    # ========================================================
    # CASE 1 — COMPLETE EXISTING RUN
    # ========================================================

    if all(
        existence.values()
    ):

        print()
        print(
            "Complete artifact set detected."
        )

        print(
            "Running independent clean-room audit..."
        )


        audit = audit_longtail_run(

            dataset_name=
                "Tangdaowan",

            method=
                "LA_Loss",

            candidate=
                tau,

            seed=
                42
        )


        if not audit[
            "pass"
        ]:

            raise RuntimeError(
                f"Existing τ={tau} run failed audit. "
                "Do not delete or retrain yet."
            )


        best = audit[
            "recomputed_best"
        ]


        print()

        print(
            "Existing run audit : PASS"
        )

        print(
            "Training action     : NONE"
        )

        print(
            "Best epoch          :",
            best["epoch"]
        )

        print(
            "Best Val AA         :",
            f"{best['aa']:.6f}"
        )

        print(
            "Best Macro-F1       :",
            f"{best['macro_f1']:.6f}"
        )

        print(
            "Best Val OA         :",
            f"{best['oa']:.6f}"
        )

        print("=" * 125)


        return audit[
            "metadata"
        ]


    # ========================================================
    # CASE 2 — NO ARTIFACTS
    # ========================================================

    elif not any(
        existence.values()
    ):

        print()

        print(
            "No previous artifacts detected."
        )

        print(
            "Training new 100-epoch run..."
        )

        print("=" * 125)


        return train_longtail_run(

            dataset_name=
                "Tangdaowan",

            method=
                "LA_Loss",

            candidate=
                tau,

            seed=
                42,

            verbose=
                True
        )


    # ========================================================
    # CASE 3 — PARTIAL ARTIFACT SET
    # ========================================================

    else:

        print()

        print(
            "PARTIAL ARTIFACT SET DETECTED"
        )

        print(
            existence
        )


        raise RuntimeError(

            f"τ={tau} has partial experiment artifacts. "
            "STOP — do not delete or retrain automatically."
        )


# ============================================================
# τ = 0.5
# ============================================================

TANGDAOWAN_LA_T05_META = (
    recover_or_train_la_run(
        0.5
    )
)


# ============================================================
# τ = 1.0
# ============================================================

TANGDAOWAN_LA_T10_META = (
    recover_or_train_la_run(
        1.0
    )
)


# ============================================================
# FINAL TUNING-RUN SUMMARY
# ============================================================

print()
print("=" * 125)
print("TANGDAOWAN LA-LOSS — TUNING RUNS AVAILABLE")
print("=" * 125)


for meta in [
    TANGDAOWAN_LA_T05_META,
    TANGDAOWAN_LA_T10_META,
]:

    print(

        f"tau={float(meta['candidate']):.1f} | "
        f"Seed={int(meta['seed'])} | "
        f"BestEpoch={int(meta['best_epoch']):3d} | "
        f"ValAA="
        f"{float(meta['best_validation']['aa']):.6f} | "
        f"F1="
        f"{float(meta['best_validation']['macro_f1']):.6f} | "
        f"ValOA="
        f"{float(meta['best_validation']['oa']):.6f}"
    )


print()

print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print("=" * 125)


TANGDAOWAN LA-LOSS τ=0.5 | SEED 42
Checkpoint exists: True
History exists   : True
Metadata exists  : True

Complete artifact set detected.
Running independent clean-room audit...

Existing run audit : PASS
Training action     : NONE
Best epoch          : 92
Best Val AA         : 0.999916
Best Macro-F1       : 0.997854
Best Val OA         : 0.999623

TANGDAOWAN LA-LOSS τ=1.0 | SEED 42
Checkpoint exists: False
History exists   : False
Metadata exists  : False

No previous artifacts detected.
Training new 100-epoch run...
TRAINING RUN | Tangdaowan | LA_Loss | candidate=1.0 | seed=42
Epoch   1 | TrainLoss 1.8674 | ValLoss 1.0386 | ValOA 0.605447 | ValAA 0.372933 | F1 0.335938 | 5.9s BEST
Epoch   2 | TrainLoss 0.7357 | ValLoss 0.5254 | ValOA 0.636072 | ValAA 0.673615 | F1 0.523679 | 2.6s BEST
Epoch   3 | TrainLoss 0.4985 | ValLoss 0.4515 | ValOA 0.727101 | ValAA 0.760642 | F1 0.566428 | 2.2s BEST
Epoch   4 | TrainLoss 0.3979 | ValLoss 0.2980 | ValOA 0.772051 | ValAA 0.828958 | F1 0.640728

In [ ]:
# ============================================================
# CELL 52 — TANGDAOWAN LA-LOSS
# TWO-CANDIDATE CLEAN-ROOM AUDIT
# ============================================================

TANGDAOWAN_LA_TUNING_AUDITS = {}


print("=" * 125)
print("TANGDAOWAN LA-LOSS — SEED 42 CANDIDATE AUDIT")
print("=" * 125)


for tau in [
    0.5,
    1.0,
]:

    audit = audit_longtail_run(

        dataset_name=
            "Tangdaowan",

        method=
            "LA_Loss",

        candidate=
            tau,

        seed=
            42
    )


    TANGDAOWAN_LA_TUNING_AUDITS[
        tau
    ] = audit


    best = audit[
        "recomputed_best"
    ]


    print(

        f"tau={tau:.1f} | "
        f"Epoch={best['epoch']:3d} | "
        f"ValAA={best['aa']:.6f} | "
        f"F1={best['macro_f1']:.6f} | "
        f"ValOA={best['oa']:.6f} | "
        f"Artifact="
        f"{'PASS' if audit['pass'] else 'FAIL'}"
    )


ALL_TANGDAOWAN_LA_TUNING_PASS = all(

    audit[
        "pass"
    ]

    for audit in
    TANGDAOWAN_LA_TUNING_AUDITS.values()
)


print("-" * 125)

print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print()

print(
    "TANGDAOWAN LA-LOSS TUNING-RUN AUDIT GATE:",
    "PASS"
    if ALL_TANGDAOWAN_LA_TUNING_PASS
    else "FAIL"
)

print("=" * 125)


if not ALL_TANGDAOWAN_LA_TUNING_PASS:

    raise RuntimeError(
        "Tangdaowan LA-loss tuning audit failed."
    )

TANGDAOWAN LA-LOSS — SEED 42 CANDIDATE AUDIT
tau=0.5 | Epoch= 92 | ValAA=0.999916 | F1=0.997854 | ValOA=0.999623 | Artifact=PASS
tau=1.0 | Epoch= 93 | ValAA=1.000000 | F1=1.000000 | ValOA=1.000000 | Artifact=PASS
-----------------------------------------------------------------------------------------------------------------------------
Official-test dataset created    : False
Official-test DataLoader created : False

TANGDAOWAN LA-LOSS TUNING-RUN AUDIT GATE: PASS


In [ ]:
# ============================================================
# CELL 53 — TANGDAOWAN LA-LOSS
# VALIDATION-ONLY tau SELECTION
# ============================================================

assert ALL_TANGDAOWAN_LA_TUNING_PASS

assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


la_candidate_records = []


for tau in [
    0.5,
    1.0,
]:

    audit = (
        TANGDAOWAN_LA_TUNING_AUDITS[
            tau
        ]
    )


    best = audit[
        "recomputed_best"
    ]


    la_candidate_records.append({

        "tau":
            float(
                tau
            ),

        "best_epoch":
            int(
                best[
                    "epoch"
                ]
            ),

        "validation_aa":
            float(
                best[
                    "aa"
                ]
            ),

        "validation_macro_f1":
            float(
                best[
                    "macro_f1"
                ]
            ),

        "validation_oa":
            float(
                best[
                    "oa"
                ]
            ),

        "validation_loss":
            float(
                best[
                    "loss"
                ]
            ),

        "checkpoint_sha256":
            audit[
                "checkpoint_sha256"
            ],
    })


TANGDAOWAN_LA_TUNING_DF = pd.DataFrame(
    la_candidate_records
)


def better_la_candidate(
    candidate,
    best
):

    if best is None:

        return True


    # --------------------------------------------------------
    # 1. Validation AA
    # --------------------------------------------------------

    if (
        candidate[
            "validation_aa"
        ]
        >
        best[
            "validation_aa"
        ]
        +
        CHECKPOINT_TOL
    ):

        return True


    if not math.isclose(

        candidate[
            "validation_aa"
        ],

        best[
            "validation_aa"
        ],

        abs_tol=
            CHECKPOINT_TOL
    ):

        return False


    # --------------------------------------------------------
    # 2. Macro-F1
    # --------------------------------------------------------

    if (
        candidate[
            "validation_macro_f1"
        ]
        >
        best[
            "validation_macro_f1"
        ]
        +
        CHECKPOINT_TOL
    ):

        return True


    if not math.isclose(

        candidate[
            "validation_macro_f1"
        ],

        best[
            "validation_macro_f1"
        ],

        abs_tol=
            CHECKPOINT_TOL
    ):

        return False


    # --------------------------------------------------------
    # 3. Validation OA
    # --------------------------------------------------------

    if (
        candidate[
            "validation_oa"
        ]
        >
        best[
            "validation_oa"
        ]
        +
        CHECKPOINT_TOL
    ):

        return True


    if not math.isclose(

        candidate[
            "validation_oa"
        ],

        best[
            "validation_oa"
        ],

        abs_tol=
            CHECKPOINT_TOL
    ):

        return False


    # --------------------------------------------------------
    # 4. Smaller/simpler tau
    # --------------------------------------------------------

    return (
        candidate[
            "tau"
        ]
        <
        best[
            "tau"
        ]
    )


selected_la = None


for record in la_candidate_records:

    if better_la_candidate(
        record,
        selected_la
    ):

        selected_la = (
            record.copy()
        )


TANGDAOWAN_LA_SELECTED_TAU = float(
    selected_la[
        "tau"
    ]
)


TANGDAOWAN_LA_SELECTION = {

    "dataset":
        "Tangdaowan",

    "method":
        "LA_Loss",

    "tuning_seed":
        42,

    "candidate_tau":
        [
            0.5,
            1.0
        ],

    "selection_primary":
        "validation_AA",

    "tie_1":
        "validation_Macro_F1",

    "tie_2":
        "validation_OA",

    "final_tie":
        "smaller_tau",

    "selected_tau":
        TANGDAOWAN_LA_SELECTED_TAU,

    "selected_best_epoch":
        int(
            selected_la[
                "best_epoch"
            ]
        ),

    "selected_validation_aa":
        float(
            selected_la[
                "validation_aa"
            ]
        ),

    "selected_validation_macro_f1":
        float(
            selected_la[
                "validation_macro_f1"
            ]
        ),

    "selected_validation_oa":
        float(
            selected_la[
                "validation_oa"
            ]
        ),

    "candidate_results":
        la_candidate_records,

    "protocol_sha256":
        protocol_sha256,

    "engine_sha256":
        ENGINE_SPEC_SHA256,

    "official_test_used":
        False,
}


TANGDAOWAN_LA_SELECTION_PATH = (

    NOTEBOOK04_AUDIT_DIR /

    "tangdaowan_la_loss_hyperparameter_selection.json"
)


if TANGDAOWAN_LA_SELECTION_PATH.exists():

    with open(
        TANGDAOWAN_LA_SELECTION_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        existing = json.load(f)


    if existing != TANGDAOWAN_LA_SELECTION:

        raise RuntimeError(
            "Existing Tangdaowan LA-loss selection "
            "does not match current validation result."
        )

else:

    with open(
        TANGDAOWAN_LA_SELECTION_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            TANGDAOWAN_LA_SELECTION,
            f,
            indent=2,
            sort_keys=True
        )


TANGDAOWAN_LA_SELECTION_SHA = (
    sha256_file(
        TANGDAOWAN_LA_SELECTION_PATH
    )
)


print("=" * 125)
print("TANGDAOWAN — LA-LOSS VALIDATION-ONLY TUNING")
print("=" * 125)


display(

    TANGDAOWAN_LA_TUNING_DF[
        [
            "tau",
            "best_epoch",
            "validation_aa",
            "validation_macro_f1",
            "validation_oa",
        ]
    ]
)


print()

print(
    "SELECTED TAU:",
    TANGDAOWAN_LA_SELECTED_TAU
)

print(
    "Selected Val AA:",
    f"{selected_la['validation_aa']:.6f}"
)

print(
    "Selected F1    :",
    f"{selected_la['validation_macro_f1']:.6f}"
)

print(
    "Selected Val OA:",
    f"{selected_la['validation_oa']:.6f}"
)

print()

print(
    "Official test used:",
    False
)

print(
    "Selection SHA:",
    TANGDAOWAN_LA_SELECTION_SHA
)

print("=" * 125)

print(
    "TANGDAOWAN LA-LOSS TUNING GATE: PASS"
)

print("=" * 125)

TANGDAOWAN — LA-LOSS VALIDATION-ONLY TUNING


,tau,best_epoch,validation_aa,validation_macro_f1,validation_oa
0,0.5,92,0.999916,0.997854,0.999623
1,1.0,93,1.000000,1.000000,1.000000



SELECTED TAU: 1.0
Selected Val AA: 1.000000
Selected F1    : 1.000000
Selected Val OA: 1.000000

Official test used: False
Selection SHA: d82685535fa84dabd99b9df5992c261714e14bbde2e363013bab665372064cef
TANGDAOWAN LA-LOSS TUNING GATE: PASS


In [ ]:
# ============================================================
# CELL 54 — TANGDAOWAN LA-LOSS
# SELECTED tau=1.0 | REMAINING SEEDS 123, 3407
# RESTART-SAFE
# ============================================================

assert TANGDAOWAN_LA_SELECTED_TAU == 1.0

assert TANGDAOWAN_LA_SELECTION_PATH.exists()

assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


def run_or_recover_selected_la(seed):

    tau = TANGDAOWAN_LA_SELECTED_TAU

    run_id = make_baseline_run_id(
        "Tangdaowan",
        "LA_Loss",
        tau,
        seed
    )


    checkpoint_path = (
        LONGTAIL_CHECKPOINT_DIR /
        f"{run_id}.pt"
    )

    history_path = (
        LONGTAIL_HISTORY_DIR /
        f"{run_id}_history.csv"
    )

    metadata_path = (
        LONGTAIL_METADATA_DIR /
        f"{run_id}_metadata.json"
    )


    existence = {

        "checkpoint":
            checkpoint_path.exists(),

        "history":
            history_path.exists(),

        "metadata":
            metadata_path.exists(),
    }


    print()
    print("=" * 125)

    print(
        f"TANGDAOWAN LA-LOSS | "
        f"tau={tau:.1f} | seed={seed}"
    )

    print("=" * 125)


    # ========================================================
    # COMPLETE EXISTING RUN
    # ========================================================

    if all(existence.values()):

        print(
            "Complete artifact set detected."
        )

        print(
            "Action: clean-room audit + recovery"
        )


        audit = audit_longtail_run(

            dataset_name=
                "Tangdaowan",

            method=
                "LA_Loss",

            candidate=
                tau,

            seed=
                seed
        )


        if not audit["pass"]:

            raise RuntimeError(
                f"Existing LA-loss seed {seed} "
                "failed artifact audit."
            )


        return audit["metadata"]


    # ========================================================
    # NO EXISTING RUN
    # ========================================================

    elif not any(existence.values()):

        print(
            "No previous artifacts detected."
        )

        print(
            "Action: new 100-epoch training run"
        )


        return train_longtail_run(

            dataset_name=
                "Tangdaowan",

            method=
                "LA_Loss",

            candidate=
                tau,

            seed=
                seed,

            verbose=
                True
        )


    # ========================================================
    # PARTIAL ARTIFACTS
    # ========================================================

    else:

        print(
            "Partial artifact state:",
            existence
        )


        raise RuntimeError(

            f"Partial artifacts detected for "
            f"Tangdaowan LA-loss seed {seed}. "
            "STOP — no automatic overwrite."
        )


# ============================================================
# Remaining seeds
# ============================================================

TANGDAOWAN_LA_SEED123_META = (
    run_or_recover_selected_la(
        123
    )
)


TANGDAOWAN_LA_SEED3407_META = (
    run_or_recover_selected_la(
        3407
    )
)


print()
print("=" * 125)

print(
    "TANGDAOWAN LA-LOSS — "
    "REMAINING SEEDS COMPLETE"
)

print("=" * 125)


for meta in [

    TANGDAOWAN_LA_SEED123_META,
    TANGDAOWAN_LA_SEED3407_META,

]:

    print(

        f"Seed {int(meta['seed']):4d} | "
        f"BestEpoch={int(meta['best_epoch']):3d} | "
        f"ValAA="
        f"{float(meta['best_validation']['aa']):.6f} | "
        f"F1="
        f"{float(meta['best_validation']['macro_f1']):.6f} | "
        f"ValOA="
        f"{float(meta['best_validation']['oa']):.6f}"
    )


print()

print(
    "Selected tau:",
    TANGDAOWAN_LA_SELECTED_TAU
)

print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print("=" * 125)


TANGDAOWAN LA-LOSS | tau=1.0 | seed=123
No previous artifacts detected.
Action: new 100-epoch training run
TRAINING RUN | Tangdaowan | LA_Loss | candidate=1.0 | seed=123
Epoch   1 | TrainLoss 1.9832 | ValLoss 1.7866 | ValOA 0.296551 | ValAA 0.226668 | F1 0.125340 | 5.7s BEST
Epoch   2 | TrainLoss 1.7177 | ValLoss 1.5882 | ValOA 0.298342 | ValAA 0.266143 | F1 0.153022 | 3.6s BEST
Epoch   3 | TrainLoss 1.4127 | ValLoss 1.0271 | ValOA 0.432812 | ValAA 0.383953 | F1 0.276781 | 2.2s BEST
Epoch   4 | TrainLoss 0.9476 | ValLoss 0.8100 | ValOA 0.544101 | ValAA 0.480726 | F1 0.381005 | 2.3s BEST
Epoch   5 | TrainLoss 0.6743 | ValLoss 0.5240 | ValOA 0.698172 | ValAA 0.624493 | F1 0.505665 | 2.8s BEST
Epoch   6 | TrainLoss 0.5432 | ValLoss 0.4607 | ValOA 0.686204 | ValAA 0.706177 | F1 0.571373 | 2.4s BEST
Epoch   8 | TrainLoss 0.4327 | ValLoss 0.3810 | ValOA 0.729269 | ValAA 0.771833 | F1 0.587013 | 2.2s BEST
Epoch   9 | TrainLoss 0.4075 | ValLoss 0.3643 | ValOA 0.760837 | ValAA 0.793406 | F1 0.

In [ ]:
# ============================================================
# CELL 55 — TANGDAOWAN LA-LOSS tau=1.0
# THREE-SEED ARTIFACT AUDIT
# ============================================================

TANGDAOWAN_LA_3SEED_AUDITS = {}


print("=" * 130)

print(
    "TANGDAOWAN LA-LOSS tau=1.0 — "
    "THREE-SEED ARTIFACT AUDIT"
)

print("=" * 130)


for seed in TRAINING_SEEDS:

    audit = audit_longtail_run(

        dataset_name=
            "Tangdaowan",

        method=
            "LA_Loss",

        candidate=
            TANGDAOWAN_LA_SELECTED_TAU,

        seed=
            seed
    )


    TANGDAOWAN_LA_3SEED_AUDITS[
        seed
    ] = audit


    best = audit[
        "recomputed_best"
    ]


    print(

        f"Seed {seed:4d} | "
        f"Epoch={int(best['epoch']):3d} | "
        f"ValAA={best['aa']:.6f} | "
        f"F1={best['macro_f1']:.6f} | "
        f"ValOA={best['oa']:.6f} | "
        f"Artifact="
        f"{'PASS' if audit['pass'] else 'FAIL'}"
    )


ALL_TANGDAOWAN_LA_SEEDS_PASS = all(

    audit["pass"]

    for audit in
    TANGDAOWAN_LA_3SEED_AUDITS.values()
)


print("-" * 130)

print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print()

print(

    "TANGDAOWAN LA-LOSS 3-SEED GATE:",

    "PASS"
    if ALL_TANGDAOWAN_LA_SEEDS_PASS
    else "FAIL"
)

print("=" * 130)


if not ALL_TANGDAOWAN_LA_SEEDS_PASS:

    raise RuntimeError(
        "Tangdaowan LA-loss "
        "three-seed audit failed."
    )

TANGDAOWAN LA-LOSS tau=1.0 — THREE-SEED ARTIFACT AUDIT
Seed   42 | Epoch= 93 | ValAA=1.000000 | F1=1.000000 | ValOA=1.000000 | Artifact=PASS
Seed  123 | Epoch= 75 | ValAA=0.999807 | F1=0.999578 | ValOA=0.999529 | Artifact=PASS
Seed 3407 | Epoch= 60 | ValAA=0.999916 | F1=0.999694 | ValOA=0.999623 | Artifact=PASS
----------------------------------------------------------------------------------------------------------------------------------
Official-test dataset created    : False
Official-test DataLoader created : False

TANGDAOWAN LA-LOSS 3-SEED GATE: PASS


In [ ]:
# ============================================================
# CELL 56 — FREEZE TANGDAOWAN LA-LOSS
# THREE-SEED VALIDATION RESULT
# ============================================================

assert ALL_TANGDAOWAN_LA_SEEDS_PASS

assert TANGDAOWAN_LA_SELECTED_TAU == 1.0

assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


records = []


for seed in TRAINING_SEEDS:

    audit = (
        TANGDAOWAN_LA_3SEED_AUDITS[
            seed
        ]
    )


    best = audit[
        "recomputed_best"
    ]


    records.append({

        "dataset":
            "Tangdaowan",

        "method":
            "LA_Loss",

        "tau":
            float(
                TANGDAOWAN_LA_SELECTED_TAU
            ),

        "seed":
            int(seed),

        "best_epoch":
            int(
                best["epoch"]
            ),

        "validation_aa":
            float(
                best["aa"]
            ),

        "validation_macro_f1":
            float(
                best["macro_f1"]
            ),

        "validation_oa":
            float(
                best["oa"]
            ),

        "validation_kappa":
            float(
                best["kappa"]
            ),

        "validation_loss":
            float(
                best["loss"]
            ),

        "checkpoint_sha256":
            audit[
                "checkpoint_sha256"
            ],
    })


TANGDAOWAN_LA_3SEED_DF = (
    pd.DataFrame(
        records
    )
)


la_summary = {

    "validation_aa_mean":
        float(
            TANGDAOWAN_LA_3SEED_DF[
                "validation_aa"
            ].mean()
        ),

    "validation_aa_std":
        float(
            TANGDAOWAN_LA_3SEED_DF[
                "validation_aa"
            ].std(
                ddof=1
            )
        ),

    "validation_macro_f1_mean":
        float(
            TANGDAOWAN_LA_3SEED_DF[
                "validation_macro_f1"
            ].mean()
        ),

    "validation_macro_f1_std":
        float(
            TANGDAOWAN_LA_3SEED_DF[
                "validation_macro_f1"
            ].std(
                ddof=1
            )
        ),

    "validation_oa_mean":
        float(
            TANGDAOWAN_LA_3SEED_DF[
                "validation_oa"
            ].mean()
        ),

    "validation_oa_std":
        float(
            TANGDAOWAN_LA_3SEED_DF[
                "validation_oa"
            ].std(
                ddof=1
            )
        ),
}


TANGDAOWAN_LA_MANIFEST = {

    "dataset":
        "Tangdaowan",

    "method":
        "LA_Loss",

    "selected_tau":
        float(
            TANGDAOWAN_LA_SELECTED_TAU
        ),

    "candidate_tau":
        [
            0.5,
            1.0
        ],

    "selection_metric":
        "validation_AA",

    "selection_sha256":
        TANGDAOWAN_LA_SELECTION_SHA,

    "training_seeds":
        TRAINING_SEEDS,

    "runs":
        records,

    "summary":
        la_summary,

    "protocol_sha256":
        protocol_sha256,

    "engine_sha256":
        ENGINE_SPEC_SHA256,

    "balanced_softmax_equivalence_note":
        (
            "Under the frozen loss definitions, "
            "LA-loss with tau=1.0 and Balanced Softmax "
            "use equivalent prior-logit adjustments up to "
            "a class-independent additive constant. "
            "Both remain reported because both baselines "
            "were predeclared in the protocol."
        ),

    "official_test_used":
        False,
}


TANGDAOWAN_LA_MANIFEST_PATH = (

    NOTEBOOK04_AUDIT_DIR /

    "tangdaowan_la_loss_three_seed_manifest.json"
)


if TANGDAOWAN_LA_MANIFEST_PATH.exists():

    with open(
        TANGDAOWAN_LA_MANIFEST_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        existing = json.load(f)


    if existing != TANGDAOWAN_LA_MANIFEST:

        raise RuntimeError(

            "Existing Tangdaowan LA-loss manifest "
            "does not match the current frozen result."
        )

else:

    with open(
        TANGDAOWAN_LA_MANIFEST_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            TANGDAOWAN_LA_MANIFEST,
            f,
            indent=2,
            sort_keys=True
        )


TANGDAOWAN_LA_MANIFEST_SHA = (
    sha256_file(
        TANGDAOWAN_LA_MANIFEST_PATH
    )
)


print("=" * 125)

print(
    "TANGDAOWAN LA-LOSS — "
    "FROZEN THREE-SEED VALIDATION RESULT"
)

print("=" * 125)


display(

    TANGDAOWAN_LA_3SEED_DF[
        [
            "seed",
            "best_epoch",
            "validation_aa",
            "validation_macro_f1",
            "validation_oa",
        ]
    ]
)


print()


print(
    "Val AA       : "
    f"{la_summary['validation_aa_mean']:.6f} ± "
    f"{la_summary['validation_aa_std']:.6f}"
)

print(
    "Val Macro-F1 : "
    f"{la_summary['validation_macro_f1_mean']:.6f} ± "
    f"{la_summary['validation_macro_f1_std']:.6f}"
)

print(
    "Val OA       : "
    f"{la_summary['validation_oa_mean']:.6f} ± "
    f"{la_summary['validation_oa_std']:.6f}"
)


print()

print(
    "Selected tau:",
    TANGDAOWAN_LA_SELECTED_TAU
)

print(
    "Selection SHA:",
    TANGDAOWAN_LA_SELECTION_SHA
)

print(
    "Manifest SHA:",
    TANGDAOWAN_LA_MANIFEST_SHA
)

print(
    "Official test used:",
    False
)


print("=" * 125)

print(
    "TANGDAOWAN LA-LOSS FREEZE GATE: PASS"
)

print("=" * 125)

TANGDAOWAN LA-LOSS — FROZEN THREE-SEED VALIDATION RESULT


,seed,best_epoch,validation_aa,validation_macro_f1,validation_oa
0,42,93,1.000000,1.000000,1.000000
1,123,75,0.999807,0.999578,0.999529
2,3407,60,0.999916,0.999694,0.999623



Val AA       : 0.999908 ± 0.000097
Val Macro-F1 : 0.999757 ± 0.000218
Val OA       : 0.999717 ± 0.000249

Selected tau: 1.0
Selection SHA: d82685535fa84dabd99b9df5992c261714e14bbde2e363013bab665372064cef
Manifest SHA: e6dae7bdabc08dc62662b07ec203863320a2bb2c34eaf8a0159fc69be5846a68
Official test used: False
TANGDAOWAN LA-LOSS FREEZE GATE: PASS


In [ ]:
# ============================================================
# CELL 57 — GENERIC RESTART-SAFE RUNNER
# + QINGYUN FOCAL TUNING | SEED 42
# ============================================================

assert NOTEBOOK04_REAL_TRAINING_GATE_PASS
assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


# ============================================================
# GENERIC RESTART-SAFE RUNNER
# ============================================================

def run_or_recover_longtail(
    dataset_name,
    method,
    candidate,
    seed,
    verbose=True
):

    run_id = make_baseline_run_id(
        dataset_name,
        method,
        candidate,
        seed
    )

    checkpoint_path = (
        LONGTAIL_CHECKPOINT_DIR /
        f"{run_id}.pt"
    )

    history_path = (
        LONGTAIL_HISTORY_DIR /
        f"{run_id}_history.csv"
    )

    metadata_path = (
        LONGTAIL_METADATA_DIR /
        f"{run_id}_metadata.json"
    )


    existence = {
        "checkpoint": checkpoint_path.exists(),
        "history": history_path.exists(),
        "metadata": metadata_path.exists(),
    }


    print()
    print("=" * 125)

    print(
        f"{dataset_name} | {method} | "
        f"candidate={candidate} | seed={seed}"
    )

    print("=" * 125)


    # --------------------------------------------------------
    # COMPLETE EXISTING RUN
    # --------------------------------------------------------

    if all(existence.values()):

        print(
            "Complete artifact set detected."
        )

        print(
            "Action: audit + recover"
        )


        audit = audit_longtail_run(
            dataset_name=dataset_name,
            method=method,
            candidate=candidate,
            seed=seed
        )


        if not audit["pass"]:

            raise RuntimeError(
                f"Existing run failed audit:\n{run_id}"
            )


        return audit["metadata"]


    # --------------------------------------------------------
    # NO EXISTING ARTIFACTS
    # --------------------------------------------------------

    elif not any(existence.values()):

        print(
            "No previous artifacts detected."
        )

        print(
            "Action: new 100-epoch training run"
        )


        return train_longtail_run(
            dataset_name=dataset_name,
            method=method,
            candidate=candidate,
            seed=seed,
            verbose=verbose
        )


    # --------------------------------------------------------
    # PARTIAL ARTIFACTS
    # --------------------------------------------------------

    else:

        print(
            "Partial artifact state:",
            existence
        )


        raise RuntimeError(
            f"Partial artifacts detected for {run_id}. "
            "STOP — no automatic deletion or overwrite."
        )


# ============================================================
# QINGYUN FOCAL — TUNING
# ============================================================

print("=" * 125)
print("QINGYUN FOCAL — VALIDATION-ONLY TUNING")
print("=" * 125)

print("Candidate gamma values :", [1.0, 2.0])
print("Tuning seed            :", 42)
print("Checkpoint primary     : Validation AA")
print("Official test used     :", False)

print("=" * 125)


QINGYUN_FOCAL_G1_META = run_or_recover_longtail(
    dataset_name="Qingyun",
    method="Focal",
    candidate=1.0,
    seed=42
)


QINGYUN_FOCAL_G2_META = run_or_recover_longtail(
    dataset_name="Qingyun",
    method="Focal",
    candidate=2.0,
    seed=42
)


print()
print("=" * 125)
print("QINGYUN FOCAL — TUNING RUNS AVAILABLE")
print("=" * 125)


for meta in [
    QINGYUN_FOCAL_G1_META,
    QINGYUN_FOCAL_G2_META
]:

    print(
        f"gamma={float(meta['candidate']):.1f} | "
        f"Seed={int(meta['seed'])} | "
        f"BestEpoch={int(meta['best_epoch']):3d} | "
        f"ValAA={float(meta['best_validation']['aa']):.6f} | "
        f"F1={float(meta['best_validation']['macro_f1']):.6f} | "
        f"ValOA={float(meta['best_validation']['oa']):.6f}"
    )


print()

print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print("=" * 125)

QINGYUN FOCAL — VALIDATION-ONLY TUNING
Candidate gamma values : [1.0, 2.0]
Tuning seed            : 42
Checkpoint primary     : Validation AA
Official test used     : False

Qingyun | Focal | candidate=1.0 | seed=42
No previous artifacts detected.
Action: new 100-epoch training run
TRAINING RUN | Qingyun | Focal | candidate=1.0 | seed=42
Epoch   1 | TrainLoss 0.4864 | ValLoss 0.1531 | ValOA 0.918808 | ValAA 0.788064 | F1 0.771877 | 9.8s BEST
Epoch   2 | TrainLoss 0.1413 | ValLoss 0.1128 | ValOA 0.934690 | ValAA 0.803836 | F1 0.810341 | 8.7s BEST
Epoch   3 | TrainLoss 0.0961 | ValLoss 0.0740 | ValOA 0.955529 | ValAA 0.857031 | F1 0.875948 | 6.6s BEST
Epoch   4 | TrainLoss 0.0689 | ValLoss 0.0759 | ValOA 0.963977 | ValAA 0.878897 | F1 0.900037 | 6.1s BEST
Epoch   5 | TrainLoss 0.0564 | ValLoss 0.0411 | ValOA 0.977171 | ValAA 0.934869 | F1 0.949886 | 5.5s BEST
Epoch   6 | TrainLoss 0.0400 | ValLoss 0.0310 | ValOA 0.983594 | ValAA 0.945253 | F1 0.957317 | 6.5s BEST
Epoch   8 | TrainLoss 0.

In [ ]:
# ============================================================
# CELL 58 — QINGYUN FOCAL
# CANDIDATE AUDIT + VALIDATION-ONLY SELECTION
# ============================================================

QINGYUN_FOCAL_TUNING_AUDITS = {}


print("=" * 125)
print("QINGYUN FOCAL — SEED 42 CANDIDATE AUDIT")
print("=" * 125)


for gamma in [1.0, 2.0]:

    audit = audit_longtail_run(
        dataset_name="Qingyun",
        method="Focal",
        candidate=gamma,
        seed=42
    )

    QINGYUN_FOCAL_TUNING_AUDITS[
        gamma
    ] = audit


    best = audit["recomputed_best"]


    print(
        f"gamma={gamma:.1f} | "
        f"Epoch={int(best['epoch']):3d} | "
        f"ValAA={best['aa']:.6f} | "
        f"F1={best['macro_f1']:.6f} | "
        f"ValOA={best['oa']:.6f} | "
        f"Artifact={'PASS' if audit['pass'] else 'FAIL'}"
    )


ALL_QINGYUN_FOCAL_TUNING_PASS = all(
    audit["pass"]
    for audit in
    QINGYUN_FOCAL_TUNING_AUDITS.values()
)


if not ALL_QINGYUN_FOCAL_TUNING_PASS:

    raise RuntimeError(
        "Qingyun Focal tuning artifact audit failed."
    )


# ============================================================
# FROZEN SELECTION RULE
# 1. AA
# 2. Macro-F1
# 3. OA
# 4. smaller gamma
# ============================================================

records = []


for gamma in [1.0, 2.0]:

    best = (
        QINGYUN_FOCAL_TUNING_AUDITS[
            gamma
        ]["recomputed_best"]
    )


    records.append({

        "gamma": float(gamma),

        "best_epoch":
            int(best["epoch"]),

        "validation_aa":
            float(best["aa"]),

        "validation_macro_f1":
            float(best["macro_f1"]),

        "validation_oa":
            float(best["oa"]),

        "validation_loss":
            float(best["loss"]),

        "checkpoint_sha256":
            QINGYUN_FOCAL_TUNING_AUDITS[
                gamma
            ]["checkpoint_sha256"],
    })


QINGYUN_FOCAL_TUNING_DF = pd.DataFrame(
    records
)


def focal_candidate_better(
    candidate,
    current_best
):

    if current_best is None:
        return True


    if (
        candidate["validation_aa"]
        >
        current_best["validation_aa"]
        + CHECKPOINT_TOL
    ):
        return True

    if not math.isclose(
        candidate["validation_aa"],
        current_best["validation_aa"],
        abs_tol=CHECKPOINT_TOL
    ):
        return False


    if (
        candidate["validation_macro_f1"]
        >
        current_best["validation_macro_f1"]
        + CHECKPOINT_TOL
    ):
        return True

    if not math.isclose(
        candidate["validation_macro_f1"],
        current_best["validation_macro_f1"],
        abs_tol=CHECKPOINT_TOL
    ):
        return False


    if (
        candidate["validation_oa"]
        >
        current_best["validation_oa"]
        + CHECKPOINT_TOL
    ):
        return True

    if not math.isclose(
        candidate["validation_oa"],
        current_best["validation_oa"],
        abs_tol=CHECKPOINT_TOL
    ):
        return False


    return (
        candidate["gamma"]
        <
        current_best["gamma"]
    )


selected = None


for record in records:

    if focal_candidate_better(
        record,
        selected
    ):
        selected = record.copy()


QINGYUN_FOCAL_SELECTED_GAMMA = float(
    selected["gamma"]
)


QINGYUN_FOCAL_SELECTION = {

    "dataset":
        "Qingyun",

    "method":
        "Focal",

    "tuning_seed":
        42,

    "candidate_gamma":
        [1.0, 2.0],

    "selection_primary":
        "validation_AA",

    "tie_1":
        "validation_Macro_F1",

    "tie_2":
        "validation_OA",

    "final_tie":
        "smaller_gamma",

    "selected_gamma":
        QINGYUN_FOCAL_SELECTED_GAMMA,

    "candidate_results":
        records,

    "protocol_sha256":
        protocol_sha256,

    "engine_sha256":
        ENGINE_SPEC_SHA256,

    "official_test_used":
        False,
}


QINGYUN_FOCAL_SELECTION_PATH = (
    NOTEBOOK04_AUDIT_DIR /
    "qingyun_focal_hyperparameter_selection.json"
)


if QINGYUN_FOCAL_SELECTION_PATH.exists():

    with open(
        QINGYUN_FOCAL_SELECTION_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        existing = json.load(f)

    if existing != QINGYUN_FOCAL_SELECTION:

        raise RuntimeError(
            "Existing Qingyun Focal selection "
            "does not match current validation result."
        )

else:

    with open(
        QINGYUN_FOCAL_SELECTION_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            QINGYUN_FOCAL_SELECTION,
            f,
            indent=2,
            sort_keys=True
        )


QINGYUN_FOCAL_SELECTION_SHA = sha256_file(
    QINGYUN_FOCAL_SELECTION_PATH
)


print()
print("=" * 125)
print("QINGYUN FOCAL — VALIDATION-ONLY SELECTION")
print("=" * 125)


display(
    QINGYUN_FOCAL_TUNING_DF[
        [
            "gamma",
            "best_epoch",
            "validation_aa",
            "validation_macro_f1",
            "validation_oa",
        ]
    ]
)


print()

print(
    "SELECTED GAMMA:",
    QINGYUN_FOCAL_SELECTED_GAMMA
)

print(
    "Selection SHA:",
    QINGYUN_FOCAL_SELECTION_SHA
)

print(
    "Official test used:",
    False
)

print("=" * 125)
print("QINGYUN FOCAL TUNING GATE: PASS")
print("=" * 125)

QINGYUN FOCAL — SEED 42 CANDIDATE AUDIT
gamma=1.0 | Epoch= 85 | ValAA=0.999853 | F1=0.999288 | ValOA=0.999791 | Artifact=PASS
gamma=2.0 | Epoch= 83 | ValAA=0.999278 | F1=0.999082 | ValOA=0.999476 | Artifact=PASS

QINGYUN FOCAL — VALIDATION-ONLY SELECTION


,gamma,best_epoch,validation_aa,validation_macro_f1,validation_oa
0,1.0,85,0.999853,0.999288,0.999791
1,2.0,83,0.999278,0.999082,0.999476



SELECTED GAMMA: 1.0
Selection SHA: 4dfb09ea3ac1af6afac32d13e1c26d2c33193172ff30dbe6901077194664e801
Official test used: False
QINGYUN FOCAL TUNING GATE: PASS


In [ ]:
# ============================================================
# CELL 59 — QINGYUN FOCAL
# SELECTED gamma=1.0 | SEEDS 123 AND 3407
# RESTART-SAFE
# ============================================================

assert QINGYUN_FOCAL_SELECTED_GAMMA == 1.0
assert QINGYUN_FOCAL_SELECTION_PATH.exists()

assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


QINGYUN_FOCAL_REMAINING_META = {}


print("=" * 125)
print("QINGYUN FOCAL gamma=1.0 — REMAINING SEEDS")
print("=" * 125)


for seed in [123, 3407]:

    meta = run_or_recover_longtail(
        dataset_name="Qingyun",
        method="Focal",
        candidate=QINGYUN_FOCAL_SELECTED_GAMMA,
        seed=seed,
        verbose=True
    )

    QINGYUN_FOCAL_REMAINING_META[
        seed
    ] = meta


print()
print("=" * 125)
print("QINGYUN FOCAL — REMAINING SEEDS COMPLETE")
print("=" * 125)


for seed in [123, 3407]:

    meta = QINGYUN_FOCAL_REMAINING_META[
        seed
    ]

    print(
        f"Seed {seed:4d} | "
        f"BestEpoch={int(meta['best_epoch']):3d} | "
        f"ValAA={float(meta['best_validation']['aa']):.6f} | "
        f"F1={float(meta['best_validation']['macro_f1']):.6f} | "
        f"ValOA={float(meta['best_validation']['oa']):.6f}"
    )


print()
print(
    "Selected gamma:",
    QINGYUN_FOCAL_SELECTED_GAMMA
)

print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print("=" * 125)

QINGYUN FOCAL gamma=1.0 — REMAINING SEEDS

Qingyun | Focal | candidate=1.0 | seed=123
No previous artifacts detected.
Action: new 100-epoch training run
TRAINING RUN | Qingyun | Focal | candidate=1.0 | seed=123
Epoch   1 | TrainLoss 0.6155 | ValLoss 0.1756 | ValOA 0.910186 | ValAA 0.777082 | F1 0.771253 | 9.1s BEST
Epoch   2 | TrainLoss 0.1564 | ValLoss 0.1191 | ValOA 0.935528 | ValAA 0.823871 | F1 0.835763 | 5.9s BEST
Epoch   4 | TrainLoss 0.0802 | ValLoss 0.0498 | ValOA 0.970923 | ValAA 0.939325 | F1 0.940814 | 6.4s BEST
Epoch   6 | TrainLoss 0.0420 | ValLoss 0.0463 | ValOA 0.975949 | ValAA 0.955339 | F1 0.955244 | 6.3s BEST
Epoch   7 | TrainLoss 0.0363 | ValLoss 0.0227 | ValOA 0.986386 | ValAA 0.960370 | F1 0.972303 | 5.5s BEST
Epoch   8 | TrainLoss 0.0269 | ValLoss 0.0180 | ValOA 0.989039 | ValAA 0.972516 | F1 0.979016 | 5.9s BEST
Epoch  10 | TrainLoss 0.0198 | ValLoss 0.0171 | ValOA 0.990436 | ValAA 0.970663 | F1 0.979721 | 5.8s
Epoch  13 | TrainLoss 0.0128 | ValLoss 0.0090 | ValO

In [ ]:
# ============================================================
# CELL 60 — QINGYUN FOCAL
# RUNTIME RECOVERY
# + THREE-SEED AUDIT
# + FROZEN MANIFEST
# ============================================================


# ============================================================
# PART A — RECOVER FROZEN CELL 58 SELECTION FROM DRIVE
# ============================================================

QINGYUN_FOCAL_SELECTION_PATH = (
    NOTEBOOK04_AUDIT_DIR /
    "qingyun_focal_hyperparameter_selection.json"
)


assert QINGYUN_FOCAL_SELECTION_PATH.exists(), (
    "Frozen Qingyun Focal selection JSON is missing."
)


with open(
    QINGYUN_FOCAL_SELECTION_PATH,
    "r",
    encoding="utf-8"
) as f:

    QINGYUN_FOCAL_SELECTION = json.load(f)


QINGYUN_FOCAL_SELECTED_GAMMA = float(
    QINGYUN_FOCAL_SELECTION[
        "selected_gamma"
    ]
)


QINGYUN_FOCAL_SELECTION_SHA = sha256_file(
    QINGYUN_FOCAL_SELECTION_PATH
)


# ------------------------------------------------------------
# Frozen selection integrity checks
# ------------------------------------------------------------

assert QINGYUN_FOCAL_SELECTED_GAMMA == 1.0

assert (
    QINGYUN_FOCAL_SELECTION[
        "official_test_used"
    ]
    is False
)


print("=" * 125)
print("QINGYUN FOCAL — RUNTIME SELECTION RECOVERY")
print("=" * 125)

print(
    "Recovered selected gamma :",
    QINGYUN_FOCAL_SELECTED_GAMMA
)

print(
    "Recovered selection SHA  :",
    QINGYUN_FOCAL_SELECTION_SHA
)

print(
    "Official test used        :",
    QINGYUN_FOCAL_SELECTION[
        "official_test_used"
    ]
)

print(
    "QINGYUN FOCAL RUNTIME RECOVERY: PASS"
)

print("=" * 125)


# ============================================================
# PART B — SAFETY CHECKS
# ============================================================

assert QINGYUN_FOCAL_SELECTED_GAMMA == 1.0

assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


# ============================================================
# PART C — THREE-SEED CLEAN-ROOM ARTIFACT AUDIT
# ============================================================

QINGYUN_FOCAL_3SEED_AUDITS = {}


print()
print("=" * 130)
print(
    "QINGYUN FOCAL gamma=1.0 — "
    "THREE-SEED ARTIFACT AUDIT"
)
print("=" * 130)


for seed in TRAINING_SEEDS:

    audit = audit_longtail_run(

        dataset_name=
            "Qingyun",

        method=
            "Focal",

        candidate=
            QINGYUN_FOCAL_SELECTED_GAMMA,

        seed=
            seed
    )


    QINGYUN_FOCAL_3SEED_AUDITS[
        seed
    ] = audit


    best = audit[
        "recomputed_best"
    ]


    print(

        f"Seed {seed:4d} | "
        f"Epoch={int(best['epoch']):3d} | "
        f"ValAA={best['aa']:.6f} | "
        f"F1={best['macro_f1']:.6f} | "
        f"ValOA={best['oa']:.6f} | "
        f"Artifact="
        f"{'PASS' if audit['pass'] else 'FAIL'}"
    )


ALL_QINGYUN_FOCAL_SEEDS_PASS = all(

    audit[
        "pass"
    ]

    for audit in
    QINGYUN_FOCAL_3SEED_AUDITS.values()
)


print("-" * 130)


print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print()


print(
    "QINGYUN FOCAL 3-SEED AUDIT GATE:",
    "PASS"
    if ALL_QINGYUN_FOCAL_SEEDS_PASS
    else "FAIL"
)


print("=" * 130)


if not ALL_QINGYUN_FOCAL_SEEDS_PASS:

    raise RuntimeError(
        "Qingyun Focal three-seed audit failed."
    )


# ============================================================
# PART D — BUILD THREE-SEED FROZEN RESULT
# ============================================================

records = []


for seed in TRAINING_SEEDS:

    audit = (
        QINGYUN_FOCAL_3SEED_AUDITS[
            seed
        ]
    )


    best = audit[
        "recomputed_best"
    ]


    records.append({

        "dataset":
            "Qingyun",

        "method":
            "Focal",

        "gamma":
            float(
                QINGYUN_FOCAL_SELECTED_GAMMA
            ),

        "seed":
            int(seed),

        "best_epoch":
            int(
                best[
                    "epoch"
                ]
            ),

        "validation_aa":
            float(
                best[
                    "aa"
                ]
            ),

        "validation_macro_f1":
            float(
                best[
                    "macro_f1"
                ]
            ),

        "validation_oa":
            float(
                best[
                    "oa"
                ]
            ),

        "validation_kappa":
            float(
                best[
                    "kappa"
                ]
            ),

        "validation_loss":
            float(
                best[
                    "loss"
                ]
            ),

        "checkpoint_sha256":
            audit[
                "checkpoint_sha256"
            ],
    })


QINGYUN_FOCAL_3SEED_DF = pd.DataFrame(
    records
)


# ============================================================
# PART E — COMPUTE THREE-SEED SUMMARY
# ============================================================

focal_summary = {

    "validation_aa_mean":
        float(
            QINGYUN_FOCAL_3SEED_DF[
                "validation_aa"
            ].mean()
        ),

    "validation_aa_std":
        float(
            QINGYUN_FOCAL_3SEED_DF[
                "validation_aa"
            ].std(
                ddof=1
            )
        ),

    "validation_macro_f1_mean":
        float(
            QINGYUN_FOCAL_3SEED_DF[
                "validation_macro_f1"
            ].mean()
        ),

    "validation_macro_f1_std":
        float(
            QINGYUN_FOCAL_3SEED_DF[
                "validation_macro_f1"
            ].std(
                ddof=1
            )
        ),

    "validation_oa_mean":
        float(
            QINGYUN_FOCAL_3SEED_DF[
                "validation_oa"
            ].mean()
        ),

    "validation_oa_std":
        float(
            QINGYUN_FOCAL_3SEED_DF[
                "validation_oa"
            ].std(
                ddof=1
            )
        ),
}


# ============================================================
# PART F — BUILD FROZEN MANIFEST
# ============================================================

QINGYUN_FOCAL_MANIFEST = {

    "dataset":
        "Qingyun",

    "method":
        "Focal",

    "selected_gamma":
        float(
            QINGYUN_FOCAL_SELECTED_GAMMA
        ),

    "candidate_gamma":
        [
            1.0,
            2.0
        ],

    "selection_metric":
        "validation_AA",

    "selection_sha256":
        QINGYUN_FOCAL_SELECTION_SHA,

    "training_seeds":
        TRAINING_SEEDS,

    "runs":
        records,

    "summary":
        focal_summary,

    "protocol_sha256":
        protocol_sha256,

    "engine_sha256":
        ENGINE_SPEC_SHA256,

    "official_test_used":
        False,
}


QINGYUN_FOCAL_MANIFEST_PATH = (

    NOTEBOOK04_AUDIT_DIR /

    "qingyun_focal_three_seed_manifest.json"
)


# ============================================================
# PART G — SAFE MANIFEST WRITE / RELOAD CHECK
# ============================================================

if QINGYUN_FOCAL_MANIFEST_PATH.exists():

    with open(
        QINGYUN_FOCAL_MANIFEST_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        existing = json.load(f)


    if existing != QINGYUN_FOCAL_MANIFEST:

        raise RuntimeError(

            "Existing Qingyun Focal manifest "
            "does not match the current frozen result."
        )

else:

    with open(
        QINGYUN_FOCAL_MANIFEST_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            QINGYUN_FOCAL_MANIFEST,
            f,
            indent=2,
            sort_keys=True
        )


QINGYUN_FOCAL_MANIFEST_SHA = sha256_file(
    QINGYUN_FOCAL_MANIFEST_PATH
)


# ============================================================
# PART H — FINAL FROZEN OUTPUT
# ============================================================

print()
print("=" * 125)

print(
    "QINGYUN FOCAL — "
    "FROZEN THREE-SEED VALIDATION RESULT"
)

print("=" * 125)


display(

    QINGYUN_FOCAL_3SEED_DF[
        [
            "seed",
            "best_epoch",
            "validation_aa",
            "validation_macro_f1",
            "validation_oa",
        ]
    ]
)


print()


print(
    "Val AA       : "
    f"{focal_summary['validation_aa_mean']:.6f} ± "
    f"{focal_summary['validation_aa_std']:.6f}"
)


print(
    "Val Macro-F1 : "
    f"{focal_summary['validation_macro_f1_mean']:.6f} ± "
    f"{focal_summary['validation_macro_f1_std']:.6f}"
)


print(
    "Val OA       : "
    f"{focal_summary['validation_oa_mean']:.6f} ± "
    f"{focal_summary['validation_oa_std']:.6f}"
)


print()


print(
    "Selected gamma:",
    QINGYUN_FOCAL_SELECTED_GAMMA
)


print(
    "Selection SHA:",
    QINGYUN_FOCAL_SELECTION_SHA
)


print(
    "Manifest SHA:",
    QINGYUN_FOCAL_MANIFEST_SHA
)


print(
    "Official test used:",
    False
)


print("=" * 125)

print(
    "QINGYUN FOCAL FREEZE GATE: PASS"
)

print("=" * 125)


QINGYUN FOCAL — RUNTIME SELECTION RECOVERY
Recovered selected gamma : 1.0
Recovered selection SHA  : 4dfb09ea3ac1af6afac32d13e1c26d2c33193172ff30dbe6901077194664e801
Official test used        : False
QINGYUN FOCAL RUNTIME RECOVERY: PASS

QINGYUN FOCAL gamma=1.0 — THREE-SEED ARTIFACT AUDIT
Seed   42 | Epoch= 85 | ValAA=0.999853 | F1=0.999288 | ValOA=0.999791 | Artifact=PASS
Seed  123 | Epoch= 65 | ValAA=0.999724 | F1=0.998765 | ValOA=0.999546 | Artifact=PASS
Seed 3407 | Epoch= 83 | ValAA=0.999554 | F1=0.999555 | ValOA=0.999895 | Artifact=PASS
----------------------------------------------------------------------------------------------------------------------------------
Official-test dataset created    : False
Official-test DataLoader created : False

QINGYUN FOCAL 3-SEED AUDIT GATE: PASS

QINGYUN FOCAL — FROZEN THREE-SEED VALIDATION RESULT


,seed,best_epoch,validation_aa,validation_macro_f1,validation_oa
0,42,85,0.999853,0.999288,0.999791
1,123,65,0.999724,0.998765,0.999546
2,3407,83,0.999554,0.999555,0.999895



Val AA       : 0.999710 ± 0.000150
Val Macro-F1 : 0.999203 ± 0.000402
Val OA       : 0.999744 ± 0.000179

Selected gamma: 1.0
Selection SHA: 4dfb09ea3ac1af6afac32d13e1c26d2c33193172ff30dbe6901077194664e801
Manifest SHA: 4ff7f933412fab5b33b51a7b6848de0c2147ffbab4ccbfa30bf8fcba8baa4886
Official test used: False
QINGYUN FOCAL FREEZE GATE: PASS


In [ ]:
# ============================================================
# CELL 61 — QINGYUN LDAM-DRW
# VALIDATION-ONLY TUNING | C=0.25, 0.50 | SEED 42
# RESTART-SAFE
# ============================================================

assert NOTEBOOK04_REAL_TRAINING_GATE_PASS

assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


# ============================================================
# Confirm Qingyun Focal is already frozen
# ============================================================

QINGYUN_FOCAL_MANIFEST_PATH = (
    NOTEBOOK04_AUDIT_DIR /
    "qingyun_focal_three_seed_manifest.json"
)

assert QINGYUN_FOCAL_MANIFEST_PATH.exists(), (
    "Frozen Qingyun Focal manifest is missing."
)


# ============================================================
# GENERIC RESTART-SAFE RUNNER
# ============================================================

def run_or_recover_longtail(
    dataset_name,
    method,
    candidate,
    seed,
    verbose=True
):

    run_id = make_baseline_run_id(
        dataset_name,
        method,
        candidate,
        seed
    )

    checkpoint_path = (
        LONGTAIL_CHECKPOINT_DIR /
        f"{run_id}.pt"
    )

    history_path = (
        LONGTAIL_HISTORY_DIR /
        f"{run_id}_history.csv"
    )

    metadata_path = (
        LONGTAIL_METADATA_DIR /
        f"{run_id}_metadata.json"
    )

    existence = {
        "checkpoint": checkpoint_path.exists(),
        "history": history_path.exists(),
        "metadata": metadata_path.exists(),
    }


    print()
    print("=" * 125)

    print(
        f"{dataset_name} | {method} | "
        f"candidate={candidate} | seed={seed}"
    )

    print("=" * 125)


    # --------------------------------------------------------
    # COMPLETE RUN → AUDIT + RECOVER
    # --------------------------------------------------------

    if all(existence.values()):

        print("Complete artifact set detected.")
        print("Action: audit + recover")

        audit = audit_longtail_run(
            dataset_name=dataset_name,
            method=method,
            candidate=candidate,
            seed=seed
        )

        if not audit["pass"]:

            raise RuntimeError(
                f"Existing run failed audit:\n{run_id}"
            )

        return audit["metadata"]


    # --------------------------------------------------------
    # NO ARTIFACTS → TRAIN
    # --------------------------------------------------------

    elif not any(existence.values()):

        print("No previous artifacts detected.")
        print("Action: new 100-epoch training run")

        return train_longtail_run(
            dataset_name=dataset_name,
            method=method,
            candidate=candidate,
            seed=seed,
            verbose=verbose
        )


    # --------------------------------------------------------
    # PARTIAL ARTIFACT STATE → STOP
    # --------------------------------------------------------

    else:

        print(
            "Partial artifact state:",
            existence
        )

        raise RuntimeError(
            f"Partial artifacts detected for {run_id}. "
            "STOP — no automatic overwrite."
        )


# ============================================================
# TUNING RUNS
# ============================================================

print("=" * 125)
print("QINGYUN LDAM-DRW — VALIDATION-ONLY TUNING")
print("=" * 125)

print("Candidate C values  :", [0.25, 0.50])
print("Scale               :", 30.0)
print("DRW start epoch     :", 80)
print("DRW beta            :", 0.9999)
print("Tuning seed         :", 42)
print("Checkpoint primary  : Validation AA")
print("Official test used  :", False)

print("=" * 125)


QINGYUN_LDAM_C025_META = run_or_recover_longtail(
    dataset_name="Qingyun",
    method="LDAM_DRW",
    candidate=0.25,
    seed=42,
    verbose=True
)


QINGYUN_LDAM_C050_META = run_or_recover_longtail(
    dataset_name="Qingyun",
    method="LDAM_DRW",
    candidate=0.50,
    seed=42,
    verbose=True
)


print()
print("=" * 125)
print("QINGYUN LDAM-DRW — TUNING RUNS AVAILABLE")
print("=" * 125)


for meta in [
    QINGYUN_LDAM_C025_META,
    QINGYUN_LDAM_C050_META
]:

    best_epoch = int(
        meta["best_epoch"]
    )

    print(
        f"C={float(meta['candidate']):.2f} | "
        f"Seed={int(meta['seed'])} | "
        f"BestEpoch={best_epoch:3d} | "
        f"ValAA={float(meta['best_validation']['aa']):.6f} | "
        f"F1={float(meta['best_validation']['macro_f1']):.6f} | "
        f"ValOA={float(meta['best_validation']['oa']):.6f} | "
        f"BestCheckpointUsesDRW={best_epoch >= 80}"
    )


print()

print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print("=" * 125)

QINGYUN LDAM-DRW — VALIDATION-ONLY TUNING
Candidate C values  : [0.25, 0.5]
Scale               : 30.0
DRW start epoch     : 80
DRW beta            : 0.9999
Tuning seed         : 42
Checkpoint primary  : Validation AA
Official test used  : False

Qingyun | LDAM_DRW | candidate=0.25 | seed=42
No previous artifacts detected.
Action: new 100-epoch training run
TRAINING RUN | Qingyun | LDAM_DRW | candidate=0.25 | seed=42
Epoch   1 | TrainLoss 1.5735 | ValLoss 0.8957 | ValOA 0.913886 | ValAA 0.783482 | F1 0.778840 | 9.7s BEST
Epoch   2 | TrainLoss 0.6296 | ValLoss 0.5578 | ValOA 0.946837 | ValAA 0.829040 | F1 0.847645 | 6.5s BEST
Epoch   3 | TrainLoss 0.4101 | ValLoss 0.3528 | ValOA 0.966874 | ValAA 0.894699 | F1 0.918945 | 5.6s BEST
Epoch   4 | TrainLoss 0.2900 | ValLoss 0.2557 | ValOA 0.975984 | ValAA 0.925329 | F1 0.936755 | 6.4s BEST
Epoch   5 | TrainLoss 0.2237 | ValLoss 0.2058 | ValOA 0.981185 | ValAA 0.941734 | F1 0.958161 | 6.0s BEST
Epoch   6 | TrainLoss 0.1979 | ValLoss 0.1461 | V

In [ ]:
# ============================================================
# CELL 62 — QINGYUN LDAM-DRW
# CANDIDATE AUDIT + VALIDATION-ONLY SELECTION
# ============================================================

QINGYUN_LDAM_TUNING_AUDITS = {}


print("=" * 125)
print("QINGYUN LDAM-DRW — SEED 42 CANDIDATE AUDIT")
print("=" * 125)


for C in [0.25, 0.50]:

    audit = audit_longtail_run(
        dataset_name="Qingyun",
        method="LDAM_DRW",
        candidate=C,
        seed=42
    )

    QINGYUN_LDAM_TUNING_AUDITS[
        C
    ] = audit

    best = audit[
        "recomputed_best"
    ]

    print(
        f"C={C:.2f} | "
        f"Epoch={int(best['epoch']):3d} | "
        f"ValAA={best['aa']:.6f} | "
        f"F1={best['macro_f1']:.6f} | "
        f"ValOA={best['oa']:.6f} | "
        f"UsesDRW={'YES' if int(best['epoch']) >= 80 else 'NO '} | "
        f"Artifact={'PASS' if audit['pass'] else 'FAIL'}"
    )


ALL_QINGYUN_LDAM_TUNING_PASS = all(
    audit["pass"]
    for audit in
    QINGYUN_LDAM_TUNING_AUDITS.values()
)


if not ALL_QINGYUN_LDAM_TUNING_PASS:

    raise RuntimeError(
        "Qingyun LDAM-DRW tuning audit failed."
    )


# ============================================================
# BUILD CANDIDATE RECORDS
# ============================================================

records = []


for C in [0.25, 0.50]:

    audit = (
        QINGYUN_LDAM_TUNING_AUDITS[
            C
        ]
    )

    best = audit[
        "recomputed_best"
    ]

    records.append({

        "C":
            float(C),

        "best_epoch":
            int(best["epoch"]),

        "best_checkpoint_uses_drw":
            bool(
                int(best["epoch"]) >= 80
            ),

        "validation_aa":
            float(best["aa"]),

        "validation_macro_f1":
            float(best["macro_f1"]),

        "validation_oa":
            float(best["oa"]),

        "validation_loss":
            float(best["loss"]),

        "checkpoint_sha256":
            audit[
                "checkpoint_sha256"
            ],
    })


QINGYUN_LDAM_TUNING_DF = pd.DataFrame(
    records
)


# ============================================================
# FROZEN SELECTION RULE
# ============================================================

def ldam_candidate_better(
    candidate,
    current_best
):

    if current_best is None:
        return True


    # 1 — AA
    if (
        candidate["validation_aa"]
        >
        current_best["validation_aa"]
        + CHECKPOINT_TOL
    ):
        return True

    if not math.isclose(
        candidate["validation_aa"],
        current_best["validation_aa"],
        abs_tol=CHECKPOINT_TOL
    ):
        return False


    # 2 — Macro-F1
    if (
        candidate["validation_macro_f1"]
        >
        current_best["validation_macro_f1"]
        + CHECKPOINT_TOL
    ):
        return True

    if not math.isclose(
        candidate["validation_macro_f1"],
        current_best["validation_macro_f1"],
        abs_tol=CHECKPOINT_TOL
    ):
        return False


    # 3 — OA
    if (
        candidate["validation_oa"]
        >
        current_best["validation_oa"]
        + CHECKPOINT_TOL
    ):
        return True

    if not math.isclose(
        candidate["validation_oa"],
        current_best["validation_oa"],
        abs_tol=CHECKPOINT_TOL
    ):
        return False


    # 4 — smaller C
    return (
        candidate["C"]
        <
        current_best["C"]
    )


selected = None


for record in records:

    if ldam_candidate_better(
        record,
        selected
    ):
        selected = record.copy()


QINGYUN_LDAM_SELECTED_C = float(
    selected["C"]
)


QINGYUN_LDAM_SELECTION = {

    "dataset":
        "Qingyun",

    "method":
        "LDAM_DRW",

    "tuning_seed":
        42,

    "candidate_C":
        [0.25, 0.50],

    "scale":
        30.0,

    "drw_start_epoch":
        80,

    "drw_beta":
        0.9999,

    "selection_primary":
        "validation_AA",

    "tie_1":
        "validation_Macro_F1",

    "tie_2":
        "validation_OA",

    "final_tie":
        "smaller_C",

    "selected_C":
        QINGYUN_LDAM_SELECTED_C,

    "selected_best_epoch":
        int(
            selected["best_epoch"]
        ),

    "selected_checkpoint_uses_drw":
        bool(
            selected[
                "best_checkpoint_uses_drw"
            ]
        ),

    "candidate_results":
        records,

    "protocol_sha256":
        protocol_sha256,

    "engine_sha256":
        ENGINE_SPEC_SHA256,

    "official_test_used":
        False,
}


QINGYUN_LDAM_SELECTION_PATH = (
    NOTEBOOK04_AUDIT_DIR /
    "qingyun_ldam_drw_hyperparameter_selection.json"
)


if QINGYUN_LDAM_SELECTION_PATH.exists():

    with open(
        QINGYUN_LDAM_SELECTION_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        existing = json.load(f)

    if existing != QINGYUN_LDAM_SELECTION:

        raise RuntimeError(
            "Existing Qingyun LDAM-DRW selection "
            "does not match current validation result."
        )

else:

    with open(
        QINGYUN_LDAM_SELECTION_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            QINGYUN_LDAM_SELECTION,
            f,
            indent=2,
            sort_keys=True
        )


QINGYUN_LDAM_SELECTION_SHA = sha256_file(
    QINGYUN_LDAM_SELECTION_PATH
)


print()
print("=" * 125)
print("QINGYUN LDAM-DRW — VALIDATION-ONLY SELECTION")
print("=" * 125)


display(
    QINGYUN_LDAM_TUNING_DF[
        [
            "C",
            "best_epoch",
            "best_checkpoint_uses_drw",
            "validation_aa",
            "validation_macro_f1",
            "validation_oa",
        ]
    ]
)


print()

print(
    "SELECTED C:",
    QINGYUN_LDAM_SELECTED_C
)

print(
    "Selected best epoch:",
    int(
        selected["best_epoch"]
    )
)

print(
    "Selected checkpoint uses DRW:",
    bool(
        selected[
            "best_checkpoint_uses_drw"
        ]
    )
)

print(
    "Selection SHA:",
    QINGYUN_LDAM_SELECTION_SHA
)

print(
    "Official test used:",
    False
)

print("=" * 125)
print("QINGYUN LDAM-DRW TUNING GATE: PASS")
print("=" * 125)

QINGYUN LDAM-DRW — SEED 42 CANDIDATE AUDIT
C=0.25 | Epoch= 77 | ValAA=0.999180 | F1=0.998993 | ValOA=0.999302 | UsesDRW=NO  | Artifact=PASS
C=0.50 | Epoch= 85 | ValAA=0.999365 | F1=0.997877 | ValOA=0.999058 | UsesDRW=YES | Artifact=PASS

QINGYUN LDAM-DRW — VALIDATION-ONLY SELECTION


,C,best_epoch,best_checkpoint_uses_drw,validation_aa,validation_macro_f1,validation_oa
0,0.25,77,False,0.999180,0.998993,0.999302
1,0.50,85,True,0.999365,0.997877,0.999058



SELECTED C: 0.5
Selected best epoch: 85
Selected checkpoint uses DRW: True
Selection SHA: 3c84b2887c2c2ea6a7e35456858abe6aca49b966c1828bcc7c46dcbf4d50814b
Official test used: False
QINGYUN LDAM-DRW TUNING GATE: PASS


In [ ]:
# ============================================================
# CELL 63 — QINGYUN LDAM-DRW
# CHECKPOINT / DRW SCHEDULE DISCLOSURE AUDIT
# ============================================================

assert QINGYUN_LDAM_SELECTION_PATH.exists()

DRW_START_EPOCH = 80

selected_C = float(
    QINGYUN_LDAM_SELECTED_C
)

selected_audit = (
    QINGYUN_LDAM_TUNING_AUDITS[
        selected_C
    ]
)

best = selected_audit[
    "recomputed_best"
]

best_epoch = int(
    best["epoch"]
)

selected_checkpoint_uses_drw = bool(
    best_epoch >= DRW_START_EPOCH
)


# ============================================================
# Load complete selected-run history
# ============================================================

selected_run_id = make_baseline_run_id(
    "Qingyun",
    "LDAM_DRW",
    selected_C,
    42
)

selected_history_path = (
    LONGTAIL_HISTORY_DIR /
    f"{selected_run_id}_history.csv"
)

assert selected_history_path.exists()


history_df = pd.read_csv(
    selected_history_path
)


# Show schedule boundary and final epoch
audit_epochs = [
    78,
    79,
    80,
    81,
    100
]


schedule_rows = history_df[
    history_df["epoch"].isin(
        audit_epochs
    )
].copy()


QINGYUN_LDAM_SCHEDULE_NOTE = {

    "dataset":
        "Qingyun",

    "method":
        "LDAM_DRW",

    "selected_C":
        selected_C,

    "selected_best_epoch":
        best_epoch,

    "drw_start_epoch":
        DRW_START_EPOCH,

    "selected_checkpoint_uses_drw":
        selected_checkpoint_uses_drw,

    "checkpoint_rule_changed":
        False,

    "interpretation":
        (
            "The complete training run followed the frozen "
            "LDAM-DRW schedule. The selected checkpoint was "
            "chosen exclusively by the predeclared validation "
            "checkpoint rule. Whether the selected checkpoint "
            "contains DRW updates is reported explicitly."
        ),

    "official_test_used":
        False,
}


QINGYUN_LDAM_SCHEDULE_NOTE_PATH = (
    NOTEBOOK04_AUDIT_DIR /
    "qingyun_ldam_drw_checkpoint_schedule_note.json"
)


if QINGYUN_LDAM_SCHEDULE_NOTE_PATH.exists():

    with open(
        QINGYUN_LDAM_SCHEDULE_NOTE_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        existing = json.load(f)

    if existing != QINGYUN_LDAM_SCHEDULE_NOTE:

        raise RuntimeError(
            "Existing Qingyun LDAM schedule note "
            "does not match current frozen result."
        )

else:

    with open(
        QINGYUN_LDAM_SCHEDULE_NOTE_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            QINGYUN_LDAM_SCHEDULE_NOTE,
            f,
            indent=2,
            sort_keys=True
        )


QINGYUN_LDAM_SCHEDULE_NOTE_SHA = sha256_file(
    QINGYUN_LDAM_SCHEDULE_NOTE_PATH
)


print("=" * 125)
print("QINGYUN LDAM-DRW — CHECKPOINT/SCHEDULE AUDIT")
print("=" * 125)


if len(schedule_rows) > 0:

    display(
        schedule_rows[
            [
                "epoch",
                "validation_aa",
                "validation_macro_f1",
                "validation_oa",
                "validation_loss",
            ]
        ]
    )


print()

print(
    "Selected C                  :",
    selected_C
)

print(
    "Selected best epoch         :",
    best_epoch
)

print(
    "DRW starts at epoch         :",
    DRW_START_EPOCH
)

print(
    "Selected checkpoint uses DRW:",
    selected_checkpoint_uses_drw
)

print()

print(
    "Checkpoint rule changed:",
    False
)

print(
    "Official test used     :",
    False
)

print(
    "Audit-note SHA         :",
    QINGYUN_LDAM_SCHEDULE_NOTE_SHA
)

print("=" * 125)
print("QINGYUN LDAM-DRW SCHEDULE DISCLOSURE GATE: PASS")
print("=" * 125)

QINGYUN LDAM-DRW — CHECKPOINT/SCHEDULE AUDIT


,epoch,validation_aa,validation_macro_f1,validation_oa,validation_loss
77,78,0.996399,0.996995,0.997975,0.047069
78,79,0.997682,0.998442,0.999407,0.013769
79,80,0.991338,0.987669,0.994729,0.227271
80,81,0.996905,0.991578,0.997033,0.071346
99,100,0.982501,0.963887,0.979335,0.411755



Selected C                  : 0.5
Selected best epoch         : 85
DRW starts at epoch         : 80
Selected checkpoint uses DRW: True

Checkpoint rule changed: False
Official test used     : False
Audit-note SHA         : 7852786455bce45bd492033da7067939a07894649443c68a21685b4e9474770e
QINGYUN LDAM-DRW SCHEDULE DISCLOSURE GATE: PASS


In [ ]:
# ============================================================
# CELL 64 — QINGYUN LDAM-DRW
# SELECTED C=0.50 | SEEDS 123 AND 3407
# RESTART-SAFE
# ============================================================

assert NOTEBOOK04_REAL_TRAINING_GATE_PASS

assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


# ============================================================
# RECOVER FROZEN SELECTION IF NEEDED
# ============================================================

QINGYUN_LDAM_SELECTION_PATH = (
    NOTEBOOK04_AUDIT_DIR /
    "qingyun_ldam_drw_hyperparameter_selection.json"
)

assert QINGYUN_LDAM_SELECTION_PATH.exists()


with open(
    QINGYUN_LDAM_SELECTION_PATH,
    "r",
    encoding="utf-8"
) as f:

    QINGYUN_LDAM_SELECTION = json.load(f)


QINGYUN_LDAM_SELECTED_C = float(
    QINGYUN_LDAM_SELECTION["selected_C"]
)

QINGYUN_LDAM_SELECTION_SHA = sha256_file(
    QINGYUN_LDAM_SELECTION_PATH
)


assert QINGYUN_LDAM_SELECTED_C == 0.50


# ============================================================
# RESTART-SAFE SELECTED-RUN HELPER
# ============================================================

def run_or_recover_qingyun_ldam(seed):

    C = QINGYUN_LDAM_SELECTED_C

    run_id = make_baseline_run_id(
        "Qingyun",
        "LDAM_DRW",
        C,
        seed
    )

    checkpoint_path = (
        LONGTAIL_CHECKPOINT_DIR /
        f"{run_id}.pt"
    )

    history_path = (
        LONGTAIL_HISTORY_DIR /
        f"{run_id}_history.csv"
    )

    metadata_path = (
        LONGTAIL_METADATA_DIR /
        f"{run_id}_metadata.json"
    )

    existence = {
        "checkpoint": checkpoint_path.exists(),
        "history": history_path.exists(),
        "metadata": metadata_path.exists(),
    }


    print()
    print("=" * 125)

    print(
        f"QINGYUN LDAM-DRW | "
        f"C={C:.2f} | seed={seed}"
    )

    print("=" * 125)


    # --------------------------------------------------------
    # COMPLETE RUN EXISTS
    # --------------------------------------------------------

    if all(existence.values()):

        print("Complete artifact set detected.")
        print("Action: clean-room audit + recovery")

        audit = audit_longtail_run(
            dataset_name="Qingyun",
            method="LDAM_DRW",
            candidate=C,
            seed=seed
        )

        if not audit["pass"]:

            raise RuntimeError(
                f"Existing Qingyun LDAM seed {seed} "
                "failed artifact audit."
            )

        return audit["metadata"]


    # --------------------------------------------------------
    # NO RUN EXISTS
    # --------------------------------------------------------

    elif not any(existence.values()):

        print("No previous artifacts detected.")
        print("Action: new 100-epoch training run")

        return train_longtail_run(
            dataset_name="Qingyun",
            method="LDAM_DRW",
            candidate=C,
            seed=seed,
            verbose=True
        )


    # --------------------------------------------------------
    # PARTIAL RUN EXISTS
    # --------------------------------------------------------

    else:

        print(
            "Partial artifact state:",
            existence
        )

        raise RuntimeError(
            f"Partial artifacts detected for "
            f"Qingyun LDAM seed {seed}. "
            "STOP — no overwrite."
        )


# ============================================================
# TRAIN / RECOVER REMAINING SEEDS
# ============================================================

QINGYUN_LDAM_SEED123_META = (
    run_or_recover_qingyun_ldam(
        123
    )
)

QINGYUN_LDAM_SEED3407_META = (
    run_or_recover_qingyun_ldam(
        3407
    )
)


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 125)
print("QINGYUN LDAM-DRW — REMAINING SEEDS COMPLETE")
print("=" * 125)


for meta in [
    QINGYUN_LDAM_SEED123_META,
    QINGYUN_LDAM_SEED3407_META
]:

    best_epoch = int(
        meta["best_epoch"]
    )

    print(
        f"Seed {int(meta['seed']):4d} | "
        f"BestEpoch={best_epoch:3d} | "
        f"ValAA={float(meta['best_validation']['aa']):.6f} | "
        f"F1={float(meta['best_validation']['macro_f1']):.6f} | "
        f"ValOA={float(meta['best_validation']['oa']):.6f} | "
        f"BestCheckpointUsesDRW={best_epoch >= 80}"
    )


print()

print(
    "Selected C:",
    QINGYUN_LDAM_SELECTED_C
)

print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print("=" * 125)


QINGYUN LDAM-DRW | C=0.50 | seed=123
No previous artifacts detected.
Action: new 100-epoch training run
TRAINING RUN | Qingyun | LDAM_DRW | candidate=0.5 | seed=123
Epoch   1 | TrainLoss 3.5675 | ValLoss 1.3507 | ValOA 0.923031 | ValAA 0.788149 | F1 0.779143 | 12.9s BEST
Epoch   2 | TrainLoss 1.1559 | ValLoss 1.0236 | ValOA 0.944429 | ValAA 0.848578 | F1 0.872740 | 6.2s BEST
Epoch   3 | TrainLoss 0.7801 | ValLoss 0.6828 | ValOA 0.963453 | ValAA 0.901546 | F1 0.925465 | 7.0s BEST
Epoch   4 | TrainLoss 0.6037 | ValLoss 0.4886 | ValOA 0.974030 | ValAA 0.918215 | F1 0.940945 | 6.3s BEST
Epoch   5 | TrainLoss 0.4824 | ValLoss 0.6061 | ValOA 0.967048 | ValAA 0.931792 | F1 0.944858 | 6.8s BEST
Epoch   6 | TrainLoss 0.3905 | ValLoss 0.3191 | ValOA 0.983245 | ValAA 0.945392 | F1 0.962347 | 6.5s BEST
Epoch   7 | TrainLoss 0.3439 | ValLoss 0.3665 | ValOA 0.980383 | ValAA 0.951539 | F1 0.965916 | 7.1s BEST
Epoch   9 | TrainLoss 0.2407 | ValLoss 0.2108 | ValOA 0.989563 | ValAA 0.962063 | F1 0.9730

In [ ]:
# ============================================================
# CELL 65 — QINGYUN LDAM-DRW C=0.50
# THREE-SEED ARTIFACT AUDIT
# ============================================================

assert QINGYUN_LDAM_SELECTED_C == 0.50

QINGYUN_LDAM_3SEED_AUDITS = {}


print("=" * 130)
print(
    "QINGYUN LDAM-DRW C=0.50 — "
    "THREE-SEED ARTIFACT AUDIT"
)
print("=" * 130)


for seed in TRAINING_SEEDS:

    audit = audit_longtail_run(
        dataset_name="Qingyun",
        method="LDAM_DRW",
        candidate=QINGYUN_LDAM_SELECTED_C,
        seed=seed
    )

    QINGYUN_LDAM_3SEED_AUDITS[
        seed
    ] = audit

    best = audit[
        "recomputed_best"
    ]

    best_epoch = int(
        best["epoch"]
    )

    print(
        f"Seed {seed:4d} | "
        f"Epoch={best_epoch:3d} | "
        f"ValAA={best['aa']:.6f} | "
        f"F1={best['macro_f1']:.6f} | "
        f"ValOA={best['oa']:.6f} | "
        f"UsesDRW={'YES' if best_epoch >= 80 else 'NO '} | "
        f"Artifact={'PASS' if audit['pass'] else 'FAIL'}"
    )


ALL_QINGYUN_LDAM_SEEDS_PASS = all(
    audit["pass"]
    for audit in
    QINGYUN_LDAM_3SEED_AUDITS.values()
)


print("-" * 130)

print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print()

print(
    "QINGYUN LDAM-DRW 3-SEED GATE:",
    "PASS"
    if ALL_QINGYUN_LDAM_SEEDS_PASS
    else "FAIL"
)

print("=" * 130)


if not ALL_QINGYUN_LDAM_SEEDS_PASS:

    raise RuntimeError(
        "Qingyun LDAM-DRW "
        "three-seed audit failed."
    )

QINGYUN LDAM-DRW C=0.50 — THREE-SEED ARTIFACT AUDIT
Seed   42 | Epoch= 85 | ValAA=0.999365 | F1=0.997877 | ValOA=0.999058 | UsesDRW=YES | Artifact=PASS
Seed  123 | Epoch= 96 | ValAA=0.999372 | F1=0.999387 | ValOA=0.999651 | UsesDRW=YES | Artifact=PASS
Seed 3407 | Epoch= 86 | ValAA=0.998852 | F1=0.998851 | ValOA=0.999407 | UsesDRW=YES | Artifact=PASS
----------------------------------------------------------------------------------------------------------------------------------
Official-test dataset created    : False
Official-test DataLoader created : False

QINGYUN LDAM-DRW 3-SEED GATE: PASS


In [ ]:
# ============================================================
# CELL 66 — FREEZE QINGYUN LDAM-DRW
# THREE-SEED VALIDATION RESULT
# ============================================================

assert ALL_QINGYUN_LDAM_SEEDS_PASS
assert QINGYUN_LDAM_SELECTED_C == 0.50

assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


# ============================================================
# RECOVER SCHEDULE-NOTE SHA
# ============================================================

QINGYUN_LDAM_SCHEDULE_NOTE_PATH = (
    NOTEBOOK04_AUDIT_DIR /
    "qingyun_ldam_drw_checkpoint_schedule_note.json"
)

assert QINGYUN_LDAM_SCHEDULE_NOTE_PATH.exists()


QINGYUN_LDAM_SCHEDULE_NOTE_SHA = sha256_file(
    QINGYUN_LDAM_SCHEDULE_NOTE_PATH
)


# ============================================================
# BUILD THREE-SEED RECORDS
# ============================================================

records = []


for seed in TRAINING_SEEDS:

    audit = (
        QINGYUN_LDAM_3SEED_AUDITS[
            seed
        ]
    )

    best = audit[
        "recomputed_best"
    ]

    best_epoch = int(
        best["epoch"]
    )

    records.append({

        "dataset":
            "Qingyun",

        "method":
            "LDAM_DRW",

        "C":
            float(
                QINGYUN_LDAM_SELECTED_C
            ),

        "scale":
            30.0,

        "drw_start_epoch":
            80,

        "drw_beta":
            0.9999,

        "seed":
            int(seed),

        "best_epoch":
            best_epoch,

        "best_checkpoint_uses_drw":
            bool(
                best_epoch >= 80
            ),

        "validation_aa":
            float(
                best["aa"]
            ),

        "validation_macro_f1":
            float(
                best["macro_f1"]
            ),

        "validation_oa":
            float(
                best["oa"]
            ),

        "validation_kappa":
            float(
                best["kappa"]
            ),

        "validation_loss":
            float(
                best["loss"]
            ),

        "checkpoint_sha256":
            audit[
                "checkpoint_sha256"
            ],
    })


QINGYUN_LDAM_3SEED_DF = pd.DataFrame(
    records
)


# ============================================================
# SUMMARY
# ============================================================

ldam_summary = {

    "validation_aa_mean":
        float(
            QINGYUN_LDAM_3SEED_DF[
                "validation_aa"
            ].mean()
        ),

    "validation_aa_std":
        float(
            QINGYUN_LDAM_3SEED_DF[
                "validation_aa"
            ].std(ddof=1)
        ),

    "validation_macro_f1_mean":
        float(
            QINGYUN_LDAM_3SEED_DF[
                "validation_macro_f1"
            ].mean()
        ),

    "validation_macro_f1_std":
        float(
            QINGYUN_LDAM_3SEED_DF[
                "validation_macro_f1"
            ].std(ddof=1)
        ),

    "validation_oa_mean":
        float(
            QINGYUN_LDAM_3SEED_DF[
                "validation_oa"
            ].mean()
        ),

    "validation_oa_std":
        float(
            QINGYUN_LDAM_3SEED_DF[
                "validation_oa"
            ].std(ddof=1)
        ),

    "selected_checkpoints_after_drw":
        int(
            QINGYUN_LDAM_3SEED_DF[
                "best_checkpoint_uses_drw"
            ].sum()
        ),
}


# ============================================================
# MANIFEST
# ============================================================

QINGYUN_LDAM_MANIFEST = {

    "dataset":
        "Qingyun",

    "method":
        "LDAM_DRW",

    "selected_C":
        float(
            QINGYUN_LDAM_SELECTED_C
        ),

    "candidate_C":
        [0.25, 0.50],

    "scale":
        30.0,

    "drw_start_epoch":
        80,

    "drw_beta":
        0.9999,

    "selection_metric":
        "validation_AA",

    "selection_sha256":
        QINGYUN_LDAM_SELECTION_SHA,

    "schedule_note_sha256":
        QINGYUN_LDAM_SCHEDULE_NOTE_SHA,

    "training_seeds":
        TRAINING_SEEDS,

    "runs":
        records,

    "summary":
        ldam_summary,

    "protocol_sha256":
        protocol_sha256,

    "engine_sha256":
        ENGINE_SPEC_SHA256,

    "official_test_used":
        False,
}


QINGYUN_LDAM_MANIFEST_PATH = (
    NOTEBOOK04_AUDIT_DIR /
    "qingyun_ldam_drw_three_seed_manifest.json"
)


if QINGYUN_LDAM_MANIFEST_PATH.exists():

    with open(
        QINGYUN_LDAM_MANIFEST_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        existing = json.load(f)

    if existing != QINGYUN_LDAM_MANIFEST:

        raise RuntimeError(
            "Existing Qingyun LDAM-DRW manifest "
            "does not match the current frozen result."
        )

else:

    with open(
        QINGYUN_LDAM_MANIFEST_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            QINGYUN_LDAM_MANIFEST,
            f,
            indent=2,
            sort_keys=True
        )


QINGYUN_LDAM_MANIFEST_SHA = sha256_file(
    QINGYUN_LDAM_MANIFEST_PATH
)


# ============================================================
# FINAL OUTPUT
# ============================================================

print("=" * 125)
print(
    "QINGYUN LDAM-DRW — "
    "FROZEN THREE-SEED VALIDATION RESULT"
)
print("=" * 125)


display(
    QINGYUN_LDAM_3SEED_DF[
        [
            "seed",
            "best_epoch",
            "best_checkpoint_uses_drw",
            "validation_aa",
            "validation_macro_f1",
            "validation_oa",
        ]
    ]
)


print()

print(
    "Val AA       : "
    f"{ldam_summary['validation_aa_mean']:.6f} ± "
    f"{ldam_summary['validation_aa_std']:.6f}"
)

print(
    "Val Macro-F1 : "
    f"{ldam_summary['validation_macro_f1_mean']:.6f} ± "
    f"{ldam_summary['validation_macro_f1_std']:.6f}"
)

print(
    "Val OA       : "
    f"{ldam_summary['validation_oa_mean']:.6f} ± "
    f"{ldam_summary['validation_oa_std']:.6f}"
)


print()

print(
    "Selected C:",
    QINGYUN_LDAM_SELECTED_C
)

print(
    "Checkpoints after DRW activation:",
    f"{ldam_summary['selected_checkpoints_after_drw']}/3"
)

print(
    "Selection SHA:",
    QINGYUN_LDAM_SELECTION_SHA
)

print(
    "Schedule-note SHA:",
    QINGYUN_LDAM_SCHEDULE_NOTE_SHA
)

print(
    "Manifest SHA:",
    QINGYUN_LDAM_MANIFEST_SHA
)

print(
    "Official test used:",
    False
)

print("=" * 125)
print("QINGYUN LDAM-DRW FREEZE GATE: PASS")
print("=" * 125)

QINGYUN LDAM-DRW — FROZEN THREE-SEED VALIDATION RESULT


,seed,best_epoch,best_checkpoint_uses_drw,validation_aa,validation_macro_f1,validation_oa
0,42,85,True,0.999365,0.997877,0.999058
1,123,96,True,0.999372,0.999387,0.999651
2,3407,86,True,0.998852,0.998851,0.999407



Val AA       : 0.999196 ± 0.000298
Val Macro-F1 : 0.998705 ± 0.000765
Val OA       : 0.999372 ± 0.000298

Selected C: 0.5
Checkpoints after DRW activation: 3/3
Selection SHA: 3c84b2887c2c2ea6a7e35456858abe6aca49b966c1828bcc7c46dcbf4d50814b
Schedule-note SHA: 7852786455bce45bd492033da7067939a07894649443c68a21685b4e9474770e
Manifest SHA: e037a7b3753ebc88ec6416af471f0673dfcddf32c421e9422ce26f38a8c98a5f
Official test used: False
QINGYUN LDAM-DRW FREEZE GATE: PASS


In [ ]:
# ============================================================
# CELL 67 — QINGYUN BALANCED SOFTMAX
# THREE-SEED TRAINING / RECOVERY
# ============================================================

assert NOTEBOOK04_REAL_TRAINING_GATE_PASS

assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


# ------------------------------------------------------------
# Previous baseline must be frozen
# ------------------------------------------------------------

QINGYUN_LDAM_MANIFEST_PATH = (
    NOTEBOOK04_AUDIT_DIR /
    "qingyun_ldam_drw_three_seed_manifest.json"
)

assert QINGYUN_LDAM_MANIFEST_PATH.exists(), (
    "Frozen Qingyun LDAM-DRW manifest is missing."
)


QINGYUN_BS_METADATA = {}


print("=" * 125)
print("QINGYUN BALANCED SOFTMAX — THREE-SEED TRAINING")
print("=" * 125)


for seed in TRAINING_SEEDS:

    meta = run_or_recover_longtail(
        dataset_name="Qingyun",
        method="BalancedSoftmax",
        candidate="training_counts",
        seed=seed,
        verbose=True
    )

    QINGYUN_BS_METADATA[
        seed
    ] = meta


print()
print("=" * 125)
print("QINGYUN BALANCED SOFTMAX — TRAINING COMPLETE")
print("=" * 125)


for seed in TRAINING_SEEDS:

    meta = QINGYUN_BS_METADATA[
        seed
    ]

    print(
        f"Seed {seed:4d} | "
        f"BestEpoch={int(meta['best_epoch']):3d} | "
        f"ValAA={float(meta['best_validation']['aa']):.6f} | "
        f"F1={float(meta['best_validation']['macro_f1']):.6f} | "
        f"ValOA={float(meta['best_validation']['oa']):.6f}"
    )


print()

print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print("=" * 125)

QINGYUN BALANCED SOFTMAX — THREE-SEED TRAINING

Qingyun | BalancedSoftmax | candidate=training_counts | seed=42
No previous artifacts detected.
Action: new 100-epoch training run
TRAINING RUN | Qingyun | BalancedSoftmax | candidate=training_counts | seed=42
Epoch   1 | TrainLoss 0.6188 | ValLoss 0.2198 | ValOA 0.890882 | ValAA 0.854621 | F1 0.793070 | 13.1s BEST
Epoch   2 | TrainLoss 0.2045 | ValLoss 0.1426 | ValOA 0.923415 | ValAA 0.886505 | F1 0.822902 | 6.3s BEST
Epoch   3 | TrainLoss 0.1353 | ValLoss 0.1183 | ValOA 0.925614 | ValAA 0.915618 | F1 0.834218 | 7.3s BEST
Epoch   4 | TrainLoss 0.0931 | ValLoss 0.0905 | ValOA 0.955355 | ValAA 0.944502 | F1 0.886458 | 6.4s BEST
Epoch   5 | TrainLoss 0.0728 | ValLoss 0.0637 | ValOA 0.974623 | ValAA 0.959339 | F1 0.918694 | 7.3s BEST
Epoch   6 | TrainLoss 0.0538 | ValLoss 0.0414 | ValOA 0.972529 | ValAA 0.967913 | F1 0.910924 | 6.3s BEST
Epoch   7 | TrainLoss 0.0446 | ValLoss 0.0280 | ValOA 0.981185 | ValAA 0.975342 | F1 0.932013 | 6.9s BEST

In [ ]:
# ============================================================
# CELL 68 — QINGYUN BALANCED SOFTMAX
# THREE-SEED ARTIFACT AUDIT
# ============================================================

QINGYUN_BS_AUDITS = {}


print("=" * 130)
print(
    "QINGYUN BALANCED SOFTMAX — "
    "THREE-SEED ARTIFACT AUDIT"
)
print("=" * 130)


for seed in TRAINING_SEEDS:

    audit = audit_longtail_run(
        dataset_name="Qingyun",
        method="BalancedSoftmax",
        candidate="training_counts",
        seed=seed
    )

    QINGYUN_BS_AUDITS[
        seed
    ] = audit


    best = audit[
        "recomputed_best"
    ]


    print(
        f"Seed {seed:4d} | "
        f"Epoch={int(best['epoch']):3d} | "
        f"ValAA={best['aa']:.6f} | "
        f"F1={best['macro_f1']:.6f} | "
        f"ValOA={best['oa']:.6f} | "
        f"Artifact={'PASS' if audit['pass'] else 'FAIL'}"
    )


ALL_QINGYUN_BS_SEEDS_PASS = all(
    audit["pass"]
    for audit in
    QINGYUN_BS_AUDITS.values()
)


print("-" * 130)

print(
    "Official-test dataset created    :",
    OFFICIAL_TEST_DATASET_CREATED
)

print(
    "Official-test DataLoader created :",
    OFFICIAL_TEST_DATALOADER_CREATED
)

print()

print(
    "QINGYUN BALANCED SOFTMAX 3-SEED GATE:",
    "PASS"
    if ALL_QINGYUN_BS_SEEDS_PASS
    else "FAIL"
)

print("=" * 130)


if not ALL_QINGYUN_BS_SEEDS_PASS:

    raise RuntimeError(
        "Qingyun Balanced Softmax "
        "three-seed audit failed."
    )

QINGYUN BALANCED SOFTMAX — THREE-SEED ARTIFACT AUDIT
Seed   42 | Epoch= 97 | ValAA=0.999916 | F1=0.999546 | ValOA=0.999895 | Artifact=PASS
Seed  123 | Epoch= 53 | ValAA=0.999525 | F1=0.999340 | ValOA=0.999860 | Artifact=PASS
Seed 3407 | Epoch= 88 | ValAA=0.999837 | F1=0.999274 | ValOA=0.999756 | Artifact=PASS
----------------------------------------------------------------------------------------------------------------------------------
Official-test dataset created    : False
Official-test DataLoader created : False

QINGYUN BALANCED SOFTMAX 3-SEED GATE: PASS


In [ ]:
# ============================================================
# CELL 69 — FREEZE QINGYUN BALANCED SOFTMAX
# THREE-SEED VALIDATION RESULT
# ============================================================

assert ALL_QINGYUN_BS_SEEDS_PASS

assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


records = []


for seed in TRAINING_SEEDS:

    audit = QINGYUN_BS_AUDITS[
        seed
    ]

    best = audit[
        "recomputed_best"
    ]


    records.append({

        "dataset":
            "Qingyun",

        "method":
            "BalancedSoftmax",

        "class_statistics":
            "training_counts",

        "seed":
            int(seed),

        "best_epoch":
            int(best["epoch"]),

        "validation_aa":
            float(best["aa"]),

        "validation_macro_f1":
            float(best["macro_f1"]),

        "validation_oa":
            float(best["oa"]),

        "validation_kappa":
            float(best["kappa"]),

        "validation_loss":
            float(best["loss"]),

        "checkpoint_sha256":
            audit["checkpoint_sha256"],
    })


QINGYUN_BS_3SEED_DF = pd.DataFrame(
    records
)


bs_summary = {

    "validation_aa_mean":
        float(
            QINGYUN_BS_3SEED_DF[
                "validation_aa"
            ].mean()
        ),

    "validation_aa_std":
        float(
            QINGYUN_BS_3SEED_DF[
                "validation_aa"
            ].std(ddof=1)
        ),

    "validation_macro_f1_mean":
        float(
            QINGYUN_BS_3SEED_DF[
                "validation_macro_f1"
            ].mean()
        ),

    "validation_macro_f1_std":
        float(
            QINGYUN_BS_3SEED_DF[
                "validation_macro_f1"
            ].std(ddof=1)
        ),

    "validation_oa_mean":
        float(
            QINGYUN_BS_3SEED_DF[
                "validation_oa"
            ].mean()
        ),

    "validation_oa_std":
        float(
            QINGYUN_BS_3SEED_DF[
                "validation_oa"
            ].std(ddof=1)
        ),
}


QINGYUN_BS_MANIFEST = {

    "dataset":
        "Qingyun",

    "method":
        "BalancedSoftmax",

    "class_statistics":
        "training_counts",

    "hyperparameter_tuning_required":
        False,

    "training_seeds":
        TRAINING_SEEDS,

    "runs":
        records,

    "summary":
        bs_summary,

    "protocol_sha256":
        protocol_sha256,

    "engine_sha256":
        ENGINE_SPEC_SHA256,

    "official_test_used":
        False,
}


QINGYUN_BS_MANIFEST_PATH = (
    NOTEBOOK04_AUDIT_DIR /
    "qingyun_balanced_softmax_three_seed_manifest.json"
)


if QINGYUN_BS_MANIFEST_PATH.exists():

    with open(
        QINGYUN_BS_MANIFEST_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        existing = json.load(f)

    if existing != QINGYUN_BS_MANIFEST:

        raise RuntimeError(
            "Existing Qingyun Balanced Softmax manifest "
            "does not match the current frozen result."
        )

else:

    with open(
        QINGYUN_BS_MANIFEST_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            QINGYUN_BS_MANIFEST,
            f,
            indent=2,
            sort_keys=True
        )


QINGYUN_BS_MANIFEST_SHA = sha256_file(
    QINGYUN_BS_MANIFEST_PATH
)


print("=" * 125)
print(
    "QINGYUN BALANCED SOFTMAX — "
    "FROZEN THREE-SEED VALIDATION RESULT"
)
print("=" * 125)


display(
    QINGYUN_BS_3SEED_DF[
        [
            "seed",
            "best_epoch",
            "validation_aa",
            "validation_macro_f1",
            "validation_oa",
        ]
    ]
)


print()

print(
    "Val AA       : "
    f"{bs_summary['validation_aa_mean']:.6f} ± "
    f"{bs_summary['validation_aa_std']:.6f}"
)

print(
    "Val Macro-F1 : "
    f"{bs_summary['validation_macro_f1_mean']:.6f} ± "
    f"{bs_summary['validation_macro_f1_std']:.6f}"
)

print(
    "Val OA       : "
    f"{bs_summary['validation_oa_mean']:.6f} ± "
    f"{bs_summary['validation_oa_std']:.6f}"
)


print()

print(
    "Class statistics:",
    "training counts only"
)

print(
    "Hyperparameter tuning:",
    "NONE"
)

print(
    "Manifest SHA:",
    QINGYUN_BS_MANIFEST_SHA
)

print(
    "Official test used:",
    False
)

print("=" * 125)
print("QINGYUN BALANCED SOFTMAX FREEZE GATE: PASS")
print("=" * 125)

QINGYUN BALANCED SOFTMAX — FROZEN THREE-SEED VALIDATION RESULT


,seed,best_epoch,validation_aa,validation_macro_f1,validation_oa
0,42,97,0.999916,0.999546,0.999895
1,123,53,0.999525,0.999340,0.999860
2,3407,88,0.999837,0.999274,0.999756



Val AA       : 0.999759 ± 0.000207
Val Macro-F1 : 0.999386 ± 0.000142
Val OA       : 0.999837 ± 0.000073

Class statistics: training counts only
Hyperparameter tuning: NONE
Manifest SHA: 7500a19a77a8dfcc1f21326781c77ce309c83b034c69dd466f119cb20bcfdcfb
Official test used: False
QINGYUN BALANCED SOFTMAX FREEZE GATE: PASS


In [ ]:
# ============================================================
# CELL 70 — QINGYUN LA-LOSS
# TUNING + AUDIT + VALIDATION-ONLY SELECTION
# tau = 0.5, 1.0 | tuning seed = 42
# RESTART-SAFE
# ============================================================

assert NOTEBOOK04_REAL_TRAINING_GATE_PASS
assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


# ============================================================
# Previous Qingyun baseline must be frozen
# ============================================================

QINGYUN_BS_MANIFEST_PATH = (
    NOTEBOOK04_AUDIT_DIR /
    "qingyun_balanced_softmax_three_seed_manifest.json"
)

assert QINGYUN_BS_MANIFEST_PATH.exists(), (
    "Frozen Qingyun Balanced Softmax manifest is missing."
)


# ============================================================
# 1. TRAIN / RECOVER BOTH TUNING CANDIDATES
# ============================================================

print("=" * 125)
print("QINGYUN LA-LOSS — VALIDATION-ONLY TUNING")
print("=" * 125)

print("Candidate tau values :", [0.5, 1.0])
print("Tuning seed          :", 42)
print("Selection primary    : Validation AA")
print("Official test used   :", False)

print("=" * 125)


QINGYUN_LA_TUNING_META = {}


for tau in [0.5, 1.0]:

    meta = run_or_recover_longtail(
        dataset_name="Qingyun",
        method="LA_Loss",
        candidate=tau,
        seed=42,
        verbose=True
    )

    QINGYUN_LA_TUNING_META[tau] = meta


# ============================================================
# 2. INDEPENDENT ARTIFACT AUDIT
# ============================================================

QINGYUN_LA_TUNING_AUDITS = {}


print()
print("=" * 125)
print("QINGYUN LA-LOSS — TUNING ARTIFACT AUDIT")
print("=" * 125)


for tau in [0.5, 1.0]:

    audit = audit_longtail_run(
        dataset_name="Qingyun",
        method="LA_Loss",
        candidate=tau,
        seed=42
    )

    QINGYUN_LA_TUNING_AUDITS[tau] = audit

    best = audit["recomputed_best"]

    print(
        f"tau={tau:.1f} | "
        f"Epoch={int(best['epoch']):3d} | "
        f"ValAA={best['aa']:.6f} | "
        f"F1={best['macro_f1']:.6f} | "
        f"ValOA={best['oa']:.6f} | "
        f"Artifact={'PASS' if audit['pass'] else 'FAIL'}"
    )


ALL_QINGYUN_LA_TUNING_PASS = all(
    audit["pass"]
    for audit in QINGYUN_LA_TUNING_AUDITS.values()
)

assert ALL_QINGYUN_LA_TUNING_PASS, (
    "Qingyun LA-loss tuning artifact audit failed."
)


# ============================================================
# 3. BUILD CANDIDATE RECORDS
# ============================================================

candidate_records = []


for tau in [0.5, 1.0]:

    audit = QINGYUN_LA_TUNING_AUDITS[tau]
    best = audit["recomputed_best"]

    candidate_records.append({

        "tau":
            float(tau),

        "best_epoch":
            int(best["epoch"]),

        "validation_aa":
            float(best["aa"]),

        "validation_macro_f1":
            float(best["macro_f1"]),

        "validation_oa":
            float(best["oa"]),

        "validation_loss":
            float(best["loss"]),

        "checkpoint_sha256":
            audit["checkpoint_sha256"],
    })


QINGYUN_LA_TUNING_DF = pd.DataFrame(
    candidate_records
)


# ============================================================
# 4. FROZEN SELECTION RULE
#    AA -> Macro-F1 -> OA -> smaller tau
# ============================================================

def qingyun_la_better(candidate, current_best):

    if current_best is None:
        return True

    # 1 — Validation AA
    if (
        candidate["validation_aa"]
        >
        current_best["validation_aa"] + CHECKPOINT_TOL
    ):
        return True

    if not math.isclose(
        candidate["validation_aa"],
        current_best["validation_aa"],
        abs_tol=CHECKPOINT_TOL
    ):
        return False

    # 2 — Macro-F1
    if (
        candidate["validation_macro_f1"]
        >
        current_best["validation_macro_f1"] + CHECKPOINT_TOL
    ):
        return True

    if not math.isclose(
        candidate["validation_macro_f1"],
        current_best["validation_macro_f1"],
        abs_tol=CHECKPOINT_TOL
    ):
        return False

    # 3 — OA
    if (
        candidate["validation_oa"]
        >
        current_best["validation_oa"] + CHECKPOINT_TOL
    ):
        return True

    if not math.isclose(
        candidate["validation_oa"],
        current_best["validation_oa"],
        abs_tol=CHECKPOINT_TOL
    ):
        return False

    # 4 — simpler/smaller tau
    return (
        candidate["tau"]
        <
        current_best["tau"]
    )


selected = None


for record in candidate_records:

    if qingyun_la_better(
        record,
        selected
    ):
        selected = record.copy()


QINGYUN_LA_SELECTED_TAU = float(
    selected["tau"]
)


# ============================================================
# 5. FREEZE SELECTION
# ============================================================

QINGYUN_LA_SELECTION = {

    "dataset":
        "Qingyun",

    "method":
        "LA_Loss",

    "tuning_seed":
        42,

    "candidate_tau":
        [0.5, 1.0],

    "selection_primary":
        "validation_AA",

    "tie_1":
        "validation_Macro_F1",

    "tie_2":
        "validation_OA",

    "final_tie":
        "smaller_tau",

    "selected_tau":
        QINGYUN_LA_SELECTED_TAU,

    "selected_best_epoch":
        int(selected["best_epoch"]),

    "selected_validation_aa":
        float(selected["validation_aa"]),

    "selected_validation_macro_f1":
        float(selected["validation_macro_f1"]),

    "selected_validation_oa":
        float(selected["validation_oa"]),

    "candidate_results":
        candidate_records,

    "protocol_sha256":
        protocol_sha256,

    "engine_sha256":
        ENGINE_SPEC_SHA256,

    "official_test_used":
        False,
}


QINGYUN_LA_SELECTION_PATH = (
    NOTEBOOK04_AUDIT_DIR /
    "qingyun_la_loss_hyperparameter_selection.json"
)


if QINGYUN_LA_SELECTION_PATH.exists():

    with open(
        QINGYUN_LA_SELECTION_PATH,
        "r",
        encoding="utf-8"
    ) as f:
        existing = json.load(f)

    if existing != QINGYUN_LA_SELECTION:
        raise RuntimeError(
            "Existing Qingyun LA-loss selection "
            "does not match current validation result."
        )

else:

    with open(
        QINGYUN_LA_SELECTION_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            QINGYUN_LA_SELECTION,
            f,
            indent=2,
            sort_keys=True
        )


QINGYUN_LA_SELECTION_SHA = sha256_file(
    QINGYUN_LA_SELECTION_PATH
)


# ============================================================
# FINAL CELL 70 OUTPUT
# ============================================================

print()
print("=" * 125)
print("QINGYUN LA-LOSS — FROZEN TUNING RESULT")
print("=" * 125)


display(
    QINGYUN_LA_TUNING_DF[
        [
            "tau",
            "best_epoch",
            "validation_aa",
            "validation_macro_f1",
            "validation_oa",
        ]
    ]
)


print()

print(
    "SELECTED TAU:",
    QINGYUN_LA_SELECTED_TAU
)

print(
    "Selected Val AA:",
    f"{selected['validation_aa']:.6f}"
)

print(
    "Selection SHA:",
    QINGYUN_LA_SELECTION_SHA
)

print(
    "Official test used:",
    False
)

print("=" * 125)
print("QINGYUN LA-LOSS TUNING GATE: PASS")
print("=" * 125)

QINGYUN LA-LOSS — VALIDATION-ONLY TUNING
Candidate tau values : [0.5, 1.0]
Tuning seed          : 42
Selection primary    : Validation AA
Official test used   : False

Qingyun | LA_Loss | candidate=0.5 | seed=42
No previous artifacts detected.
Action: new 100-epoch training run
TRAINING RUN | Qingyun | LA_Loss | candidate=0.5 | seed=42
Epoch   1 | TrainLoss 0.6319 | ValLoss 0.2523 | ValOA 0.916643 | ValAA 0.787565 | F1 0.768954 | 9.9s BEST
Epoch   2 | TrainLoss 0.2315 | ValLoss 0.1689 | ValOA 0.944289 | ValAA 0.828653 | F1 0.833636 | 6.7s BEST
Epoch   3 | TrainLoss 0.1511 | ValLoss 0.1285 | ValOA 0.954028 | ValAA 0.907417 | F1 0.899677 | 6.2s BEST
Epoch   4 | TrainLoss 0.1136 | ValLoss 0.1018 | ValOA 0.962615 | ValAA 0.921073 | F1 0.910851 | 7.2s BEST
Epoch   5 | TrainLoss 0.0876 | ValLoss 0.0574 | ValOA 0.984327 | ValAA 0.951358 | F1 0.954483 | 6.1s BEST
Epoch   6 | TrainLoss 0.0715 | ValLoss 0.0453 | ValOA 0.983978 | ValAA 0.964832 | F1 0.961152 | 7.1s BEST
Epoch   9 | TrainLoss 0.04

,tau,best_epoch,validation_aa,validation_macro_f1,validation_oa
0,0.5,91,0.999368,0.999196,0.999651
1,1.0,100,0.999867,0.999478,0.999791



SELECTED TAU: 1.0
Selected Val AA: 0.999867
Selection SHA: 0b6680c58fb56d7e7acb4e44f2f0b09c282c4d0de77dfb22848af1e61da6d61a
Official test used: False
QINGYUN LA-LOSS TUNING GATE: PASS


In [ ]:
# ============================================================
# CELL 71 — QINGYUN LA-LOSS
# REMAINING SEEDS + 3-SEED AUDIT + FINAL FREEZE
# ============================================================

assert OFFICIAL_TEST_DATASET_CREATED is False
assert OFFICIAL_TEST_DATALOADER_CREATED is False


# ============================================================
# 1. RECOVER CELL 70 SELECTION FROM DRIVE
# ============================================================

QINGYUN_LA_SELECTION_PATH = (
    NOTEBOOK04_AUDIT_DIR /
    "qingyun_la_loss_hyperparameter_selection.json"
)

assert QINGYUN_LA_SELECTION_PATH.exists()


with open(
    QINGYUN_LA_SELECTION_PATH,
    "r",
    encoding="utf-8"
) as f:

    QINGYUN_LA_SELECTION = json.load(f)


QINGYUN_LA_SELECTED_TAU = float(
    QINGYUN_LA_SELECTION["selected_tau"]
)

QINGYUN_LA_SELECTION_SHA = sha256_file(
    QINGYUN_LA_SELECTION_PATH
)


assert QINGYUN_LA_SELECTED_TAU in [0.5, 1.0]
assert QINGYUN_LA_SELECTION["official_test_used"] is False


print("=" * 125)
print("QINGYUN LA-LOSS — FROZEN SELECTION RECOVERED")
print("=" * 125)

print(
    "Selected tau:",
    QINGYUN_LA_SELECTED_TAU
)

print(
    "Selection SHA:",
    QINGYUN_LA_SELECTION_SHA
)

print("=" * 125)


# ============================================================
# 2. TRAIN / RECOVER REMAINING SEEDS
# ============================================================

QINGYUN_LA_REMAINING_META = {}


for seed in [123, 3407]:

    meta = run_or_recover_longtail(
        dataset_name="Qingyun",
        method="LA_Loss",
        candidate=QINGYUN_LA_SELECTED_TAU,
        seed=seed,
        verbose=True
    )

    QINGYUN_LA_REMAINING_META[
        seed
    ] = meta


# ============================================================
# 3. THREE-SEED ARTIFACT AUDIT
# ============================================================

QINGYUN_LA_3SEED_AUDITS = {}


print()
print("=" * 130)
print(
    f"QINGYUN LA-LOSS tau={QINGYUN_LA_SELECTED_TAU:.1f} "
    "— THREE-SEED ARTIFACT AUDIT"
)
print("=" * 130)


for seed in TRAINING_SEEDS:

    audit = audit_longtail_run(
        dataset_name="Qingyun",
        method="LA_Loss",
        candidate=QINGYUN_LA_SELECTED_TAU,
        seed=seed
    )

    QINGYUN_LA_3SEED_AUDITS[
        seed
    ] = audit

    best = audit[
        "recomputed_best"
    ]

    print(
        f"Seed {seed:4d} | "
        f"Epoch={int(best['epoch']):3d} | "
        f"ValAA={best['aa']:.6f} | "
        f"F1={best['macro_f1']:.6f} | "
        f"ValOA={best['oa']:.6f} | "
        f"Artifact={'PASS' if audit['pass'] else 'FAIL'}"
    )


ALL_QINGYUN_LA_SEEDS_PASS = all(
    audit["pass"]
    for audit in
    QINGYUN_LA_3SEED_AUDITS.values()
)


if not ALL_QINGYUN_LA_SEEDS_PASS:

    raise RuntimeError(
        "Qingyun LA-loss three-seed audit failed."
    )


print(
    "QINGYUN LA-LOSS 3-SEED AUDIT GATE: PASS"
)


# ============================================================
# 4. BUILD THREE-SEED RECORDS
# ============================================================

records = []


for seed in TRAINING_SEEDS:

    audit = QINGYUN_LA_3SEED_AUDITS[
        seed
    ]

    best = audit[
        "recomputed_best"
    ]


    records.append({

        "dataset":
            "Qingyun",

        "method":
            "LA_Loss",

        "tau":
            float(
                QINGYUN_LA_SELECTED_TAU
            ),

        "seed":
            int(seed),

        "best_epoch":
            int(best["epoch"]),

        "validation_aa":
            float(best["aa"]),

        "validation_macro_f1":
            float(best["macro_f1"]),

        "validation_oa":
            float(best["oa"]),

        "validation_kappa":
            float(best["kappa"]),

        "validation_loss":
            float(best["loss"]),

        "checkpoint_sha256":
            audit["checkpoint_sha256"],
    })


QINGYUN_LA_3SEED_DF = pd.DataFrame(
    records
)


# ============================================================
# 5. SUMMARY
# ============================================================

la_summary = {

    "validation_aa_mean":
        float(
            QINGYUN_LA_3SEED_DF[
                "validation_aa"
            ].mean()
        ),

    "validation_aa_std":
        float(
            QINGYUN_LA_3SEED_DF[
                "validation_aa"
            ].std(ddof=1)
        ),

    "validation_macro_f1_mean":
        float(
            QINGYUN_LA_3SEED_DF[
                "validation_macro_f1"
            ].mean()
        ),

    "validation_macro_f1_std":
        float(
            QINGYUN_LA_3SEED_DF[
                "validation_macro_f1"
            ].std(ddof=1)
        ),

    "validation_oa_mean":
        float(
            QINGYUN_LA_3SEED_DF[
                "validation_oa"
            ].mean()
        ),

    "validation_oa_std":
        float(
            QINGYUN_LA_3SEED_DF[
                "validation_oa"
            ].std(ddof=1)
        ),
}


# ============================================================
# 6. FINAL MANIFEST
# ============================================================

equivalence_note = None

if QINGYUN_LA_SELECTED_TAU == 1.0:

    equivalence_note = (
        "Under the frozen loss definitions, LA-loss with "
        "tau=1.0 and Balanced Softmax use equivalent "
        "class-prior logit adjustments up to a "
        "class-independent additive constant. Both are "
        "retained because both baselines were predeclared."
    )


QINGYUN_LA_MANIFEST = {

    "dataset":
        "Qingyun",

    "method":
        "LA_Loss",

    "selected_tau":
        float(
            QINGYUN_LA_SELECTED_TAU
        ),

    "candidate_tau":
        [0.5, 1.0],

    "selection_metric":
        "validation_AA",

    "selection_sha256":
        QINGYUN_LA_SELECTION_SHA,

    "training_seeds":
        TRAINING_SEEDS,

    "runs":
        records,

    "summary":
        la_summary,

    "balanced_softmax_equivalence_note":
        equivalence_note,

    "protocol_sha256":
        protocol_sha256,

    "engine_sha256":
        ENGINE_SPEC_SHA256,

    "official_test_used":
        False,
}


QINGYUN_LA_MANIFEST_PATH = (
    NOTEBOOK04_AUDIT_DIR /
    "qingyun_la_loss_three_seed_manifest.json"
)


if QINGYUN_LA_MANIFEST_PATH.exists():

    with open(
        QINGYUN_LA_MANIFEST_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        existing = json.load(f)

    if existing != QINGYUN_LA_MANIFEST:

        raise RuntimeError(
            "Existing Qingyun LA-loss manifest does not "
            "match the current frozen result."
        )

else:

    with open(
        QINGYUN_LA_MANIFEST_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            QINGYUN_LA_MANIFEST,
            f,
            indent=2,
            sort_keys=True
        )


QINGYUN_LA_MANIFEST_SHA = sha256_file(
    QINGYUN_LA_MANIFEST_PATH
)


# ============================================================
# 7. FINAL QINGYUN LA OUTPUT
# ============================================================

print()
print("=" * 125)
print(
    "QINGYUN LA-LOSS — "
    "FROZEN THREE-SEED VALIDATION RESULT"
)
print("=" * 125)


display(
    QINGYUN_LA_3SEED_DF[
        [
            "seed",
            "best_epoch",
            "validation_aa",
            "validation_macro_f1",
            "validation_oa",
        ]
    ]
)


print()

print(
    "Val AA       : "
    f"{la_summary['validation_aa_mean']:.6f} ± "
    f"{la_summary['validation_aa_std']:.6f}"
)

print(
    "Val Macro-F1 : "
    f"{la_summary['validation_macro_f1_mean']:.6f} ± "
    f"{la_summary['validation_macro_f1_std']:.6f}"
)

print(
    "Val OA       : "
    f"{la_summary['validation_oa_mean']:.6f} ± "
    f"{la_summary['validation_oa_std']:.6f}"
)


print()

print(
    "Selected tau:",
    QINGYUN_LA_SELECTED_TAU
)

print(
    "Selection SHA:",
    QINGYUN_LA_SELECTION_SHA
)

print(
    "Manifest SHA:",
    QINGYUN_LA_MANIFEST_SHA
)

print(
    "Official test used:",
    False
)

print("=" * 125)
print("QINGYUN LA-LOSS FREEZE GATE: PASS")
print("=" * 125)


# ============================================================
# 8. QINGYUN DATASET COMPLETION GATE
# ============================================================

required_qingyun_manifests = [

    NOTEBOOK04_AUDIT_DIR /
    "qingyun_focal_three_seed_manifest.json",

    NOTEBOOK04_AUDIT_DIR /
    "qingyun_ldam_drw_three_seed_manifest.json",

    NOTEBOOK04_AUDIT_DIR /
    "qingyun_balanced_softmax_three_seed_manifest.json",

    NOTEBOOK04_AUDIT_DIR /
    "qingyun_la_loss_three_seed_manifest.json",
]


assert all(
    p.exists()
    for p in required_qingyun_manifests
)


print()
print("=" * 125)
print("QINGYUN — ALL FOUR TRAINING BASELINES FROZEN")
print("=" * 125)

for p in required_qingyun_manifests:
    print(
        "PASS |",
        p.name
    )

print("=" * 125)
print("QINGYUN DATASET TRAINING GATE: PASS")
print("=" * 125)

QINGYUN LA-LOSS — FROZEN SELECTION RECOVERED
Selected tau: 1.0
Selection SHA: 0b6680c58fb56d7e7acb4e44f2f0b09c282c4d0de77dfb22848af1e61da6d61a

Qingyun | LA_Loss | candidate=1.0 | seed=123
No previous artifacts detected.
Action: new 100-epoch training run
TRAINING RUN | Qingyun | LA_Loss | candidate=1.0 | seed=123
Epoch   1 | TrainLoss 0.7149 | ValLoss 0.2996 | ValOA 0.880690 | ValAA 0.830058 | F1 0.744897 | 13.3s BEST
Epoch   2 | TrainLoss 0.2523 | ValLoss 0.1924 | ValOA 0.902820 | ValAA 0.870694 | F1 0.818329 | 6.9s BEST
Epoch   3 | TrainLoss 0.1768 | ValLoss 0.1347 | ValOA 0.924148 | ValAA 0.904122 | F1 0.839871 | 7.4s BEST
Epoch   4 | TrainLoss 0.1288 | ValLoss 0.0958 | ValOA 0.929175 | ValAA 0.928023 | F1 0.848809 | 6.5s BEST
Epoch   6 | TrainLoss 0.0832 | ValLoss 0.0579 | ValOA 0.978917 | ValAA 0.955042 | F1 0.937086 | 6.3s BEST
Epoch   8 | TrainLoss 0.0618 | ValLoss 0.0449 | ValOA 0.982756 | ValAA 0.964040 | F1 0.949416 | 6.5s BEST
Epoch   9 | TrainLoss 0.0460 | ValLoss 0.0280 |

In [1]:
# ============================================================
# GPU-FREE RECOVERY CHECK
# QINGYUN LA-LOSS — INTERRUPTED CELL 71
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/CSE498/LTLC"
)

AUDIT_DIR = (
    ROOT /
    "audit/notebook04_longtail"
)

RUN_ROOT = (
    ROOT /
    "runs/longtail_training_baselines"
)

CHECKPOINT_DIR = (
    RUN_ROOT /
    "checkpoints"
)

HISTORY_DIR = (
    RUN_ROOT /
    "histories"
)

METADATA_DIR = (
    RUN_ROOT /
    "metadata"
)


# ------------------------------------------------------------
# Recover selected tau from Cell 70
# ------------------------------------------------------------

selection_path = (
    AUDIT_DIR /
    "qingyun_la_loss_hyperparameter_selection.json"
)

assert selection_path.exists(), (
    "Qingyun LA-loss selection file is missing."
)

with open(
    selection_path,
    "r",
    encoding="utf-8"
) as f:
    selection = json.load(f)


tau = float(
    selection["selected_tau"]
)


candidate_string = (
    f"{tau:.4f}"
    .replace(".", "p")
)


print("=" * 100)
print("QINGYUN LA-LOSS — GPU-FREE RECOVERY CHECK")
print("=" * 100)

print("Selected tau:", tau)
print()


for seed in [42, 123, 3407]:

    run_id = (
        f"qingyun__la_loss__"
        f"{candidate_string}__seed{seed}"
    )

    checkpoint_path = (
        CHECKPOINT_DIR /
        f"{run_id}.pt"
    )

    history_path = (
        HISTORY_DIR /
        f"{run_id}_history.csv"
    )

    metadata_path = (
        METADATA_DIR /
        f"{run_id}_metadata.json"
    )


    ckpt = checkpoint_path.exists()
    hist = history_path.exists()
    meta = metadata_path.exists()


    epochs = None

    if hist:

        df = pd.read_csv(
            history_path
        )

        epochs = len(df)


    status = (
        "COMPLETE"
        if ckpt and hist and meta and epochs == 100
        else
        "PARTIAL"
        if ckpt or hist or meta
        else
        "MISSING"
    )


    print(
        f"Seed {seed:4d} | "
        f"Checkpoint={ckpt} | "
        f"History={hist} | "
        f"Metadata={meta} | "
        f"Epochs={epochs} | "
        f"STATUS={status}"
    )


print()
print("=" * 100)
print("GPU-FREE RECOVERY CHECK COMPLETE")
print("=" * 100)

Mounted at /content/drive
QINGYUN LA-LOSS — GPU-FREE RECOVERY CHECK
Selected tau: 1.0

Seed   42 | Checkpoint=True | History=True | Metadata=True | Epochs=100 | STATUS=COMPLETE
Seed  123 | Checkpoint=False | History=False | Metadata=False | Epochs=None | STATUS=MISSING
Seed 3407 | Checkpoint=False | History=False | Metadata=False | Epochs=None | STATUS=MISSING

GPU-FREE RECOVERY CHECK COMPLETE
